In [1]:
# ================================================================
# ICH SLICE-LEVEL EVALUATION PIPELINE — FINAL PUBLICATION VERSION
# ================================================================
#
# Models:
#   ResNet-50
#   MobileNetV2
#   EfficientNet-B0
#
# Experimental protocol:
#   70% slice-level TRAIN
#   15% slice-level VALIDATION
#   15% slice-level TEST
#
# Cross-validation:
#   4-fold slice-level StratifiedKFold within the 70% training set
#
# Seed:
#   42
#
# Primary endpoints:
#   Slice-level AUROC
#   Slice-level AUPRC
#
# Secondary:
#   Accuracy
#   Precision
#   Sensitivity
#   Specificity
#   F1
#   Model parameters
#   CPU/GPU latency
#
# Additional analysis:
#   Patient overlap audit
#   Descriptive patient aggregation
#   Development-only high-specificity threshold
#   Patient-cluster bootstrap CIs
#   Grad-CAM
#   Grad-CAM deletion/insertion faithfulness
#   Publication-quality figures
#   Complete ZIP export
#
# IMPORTANT:
#   This is intentionally a SLICE-LEVEL experiment.
#   Patient overlap across train/validation/test is allowed and audited.
#   Patient-aggregated test performance is descriptive only.
#
#   Grad-CAM is explanation/saliency, not lesion segmentation.
#   No Dice/IoU is calculated because validated lesion masks are absent.
# ================================================================


# ================================================================
# 0. IMPORTS
# ================================================================

import os
import gc
import random
import time
import warnings
import re
import json
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models

from sklearn.model_selection import train_test_split, StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings("ignore")


# ================================================================
# 1. CONFIGURATION
# ================================================================

SEED = 42

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0

DROPOUT_P = 0.20

OPTIMIZER_NAME = "Adam"
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10

SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 3
SCHEDULER_MIN_LR = 1e-7

N_FOLDS = 4

TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15

PRIMARY_THRESHOLD = 0.50

HIGH_SPEC_TARGET = 0.90

N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 4242

LATENCY_WARMUP = 50
LATENCY_REPEATS_GPU = 200
LATENCY_REPEATS_CPU = 100

NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]

MODEL_NAMES = [
    "resnet50",
    "mobilenet_v2",
    "efficientnet_b0",
]

# ------------------------------------------------
# Input dataset
# ------------------------------------------------

HEMORRHAGIC_DIR = (
    "/kaggle/input/datasets/abdulkader90/"
    "brain-ct-hemorrhage-dataset/Data/Hemorrhagic/KANAMA"
)

NORMAL_DIR = (
    "/kaggle/input/datasets/abdulkader90/"
    "brain-ct-hemorrhage-dataset/Data/NORMAL"
)

# ------------------------------------------------
# Main output directory
# ------------------------------------------------

OUT_DIR = Path(
    "/kaggle/working/ich_slice_level_clean"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------
# FINAL ZIP
# ------------------------------------------------

ZIP_PATH = Path(
    "/kaggle/working/"
    "ICH_slice_level_complete_results.zip"
)

# ------------------------------------------------
# Include original CT images in ZIP
# ------------------------------------------------
#
# TRUE:
#   Original source images from /kaggle/input are added directly
#   to the ZIP.
#
# FALSE:
#   Only generated results are included.
#
# You requested all data/images in the final ZIP, therefore TRUE.
# ------------------------------------------------

INCLUDE_SOURCE_IMAGES_IN_ZIP = True

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

USE_AMP = (
    DEVICE.type == "cuda"
)

VALID_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
}

print("=" * 120)
print(
    "ICH SLICE-LEVEL EVALUATION PIPELINE — "
    "FINAL PUBLICATION VERSION"
)
print("=" * 120)

print(f"Device: {DEVICE}")

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print(
    "Models:",
    MODEL_NAMES
)

print(
    "Seed:",
    SEED
)

print(
    f"Split: "
    f"{TRAIN_FRACTION:.0%} train / "
    f"{VAL_FRACTION:.0%} validation / "
    f"{TEST_FRACTION:.0%} test"
)

print(
    f"CV: {N_FOLDS}-fold slice-level CV"
)

print(
    "Primary endpoints: AUROC + AUPRC"
)

print(
    "Grad-CAM: enabled"
)

print(
    "Source images in final ZIP:",
    INCLUDE_SOURCE_IMAGES_IN_ZIP
)

print()


# ================================================================
# 2. REPRODUCIBILITY
# ================================================================

def seed_everything(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(seed)

        torch.cuda.manual_seed_all(
            seed
        )

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


seed_everything(SEED)


# ================================================================
# 3. DATA MANIFEST
# ================================================================

def natural_key(text):

    return [
        int(x)
        if x.isdigit()
        else x.lower()
        for x in re.split(
            r"(\d+)",
            str(text)
        )
    ]


def build_records(
    class_dir,
    label,
    class_name,
):

    records = []

    if not os.path.isdir(class_dir):

        raise FileNotFoundError(
            f"Dataset directory not found:\n{class_dir}"
        )

    folders = sorted(
        [
            f
            for f in os.listdir(
                class_dir
            )
            if os.path.isdir(
                os.path.join(
                    class_dir,
                    f
                )
            )
        ],
        key=natural_key,
    )

    for folder in folders:

        folder_path = os.path.join(
            class_dir,
            folder
        )

        files = sorted(
            [
                f
                for f in os.listdir(
                    folder_path
                )
                if os.path.splitext(
                    f
                )[1].lower()
                in VALID_EXTENSIONS
            ],
            key=natural_key,
        )

        patient_id = (
            f"{class_name}_{folder}"
        )

        for fn in files:

            records.append(
                {
                    "patient_id": patient_id,
                    "folder_name": folder,
                    "class_name": class_name,
                    "label": int(label),
                    "image_path": os.path.join(
                        folder_path,
                        fn,
                    ),
                }
            )

    return records


def build_manifest():

    records = build_records(
        HEMORRHAGIC_DIR,
        1,
        "HEMORRHAGIC",
    )

    records += build_records(
        NORMAL_DIR,
        0,
        "NORMAL",
    )

    df = pd.DataFrame(
        records
    )

    if df.empty:

        raise RuntimeError(
            "Dataset manifest is empty."
        )

    labels_per_patient = (
        df.groupby(
            "patient_id"
        )["label"]
        .nunique()
    )

    if not (
        labels_per_patient == 1
    ).all():

        raise RuntimeError(
            "At least one patient has mixed labels."
        )

    return df


df = build_manifest()

print("=" * 120)
print("DATASET AUDIT")
print("=" * 120)

print(
    f"Total slices: {len(df)}"
)

print(
    f"Hemorrhagic slices: "
    f"{int((df.label == 1).sum())}"
)

print(
    f"Normal slices: "
    f"{int((df.label == 0).sum())}"
)

print(
    f"Total patients: "
    f"{df.patient_id.nunique()}"
)

print(
    f"Hemorrhagic patients: "
    f"{df.loc[df.label == 1, 'patient_id'].nunique()}"
)

print(
    f"Normal patients: "
    f"{df.loc[df.label == 0, 'patient_id'].nunique()}"
)

print(
    "\nSlices per patient:"
)

print(
    df.groupby(
        "patient_id"
    ).size().describe()
)

print()


# ================================================================
# 4. SLICE-LEVEL 70/15/15 SPLIT
# ================================================================

indices = np.arange(
    len(df)
)

labels = df[
    "label"
].to_numpy()

train_val_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_FRACTION,
    stratify=labels,
    random_state=SEED,
)

train_val_labels = labels[
    train_val_idx
]

remaining_fraction = (
    VAL_FRACTION
    / (
        TRAIN_FRACTION
        + VAL_FRACTION
    )
)

train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=remaining_fraction,
    stratify=train_val_labels,
    random_state=SEED,
)

train_idx = np.asarray(
    train_idx,
    dtype=int,
)

val_idx = np.asarray(
    val_idx,
    dtype=int,
)

test_idx = np.asarray(
    test_idx,
    dtype=int,
)

assert (
    len(train_idx)
    + len(val_idx)
    + len(test_idx)
    == len(df)
)

assert len(
    set(train_idx)
    & set(val_idx)
) == 0

assert len(
    set(train_idx)
    & set(test_idx)
) == 0

assert len(
    set(val_idx)
    & set(test_idx)
) == 0

train_df = (
    df.iloc[
        train_idx
    ]
    .reset_index(drop=True)
)

val_df = (
    df.iloc[
        val_idx
    ]
    .reset_index(drop=True)
)

test_df = (
    df.iloc[
        test_idx
    ]
    .reset_index(drop=True)
)


def patient_set(frame):

    return set(
        frame.patient_id.unique()
    )


train_patients = patient_set(
    train_df
)

val_patients = patient_set(
    val_df
)

test_patients = patient_set(
    test_df
)


patient_overlap_audit = pd.DataFrame(
    [
        {
            "split_a": "train",
            "split_b": "validation",
            "overlap_patients": len(
                train_patients
                & val_patients
            ),
        },
        {
            "split_a": "train",
            "split_b": "test",
            "overlap_patients": len(
                train_patients
                & test_patients
            ),
        },
        {
            "split_a": "validation",
            "split_b": "test",
            "overlap_patients": len(
                val_patients
                & test_patients
            ),
        },
    ]
)

print("=" * 120)
print("SLICE-LEVEL 70/15/15 SPLIT")
print("=" * 120)

for name, frame in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:

    print(
        f"{name:12s}: "
        f"{len(frame):5d} slices | "
        f"positive={int(frame.label.sum()):5d} | "
        f"negative={int((frame.label == 0).sum()):5d} | "
        f"patients={frame.patient_id.nunique():3d}"
    )

print()

print(
    "Patient overlap audit:"
)

print(
    patient_overlap_audit.to_string(
        index=False
    )
)

print()

split_manifest = df.copy()

split_manifest[
    "split"
] = ""

split_manifest.loc[
    train_idx,
    "split"
] = "train_70"

split_manifest.loc[
    val_idx,
    "split"
] = "validation_15"

split_manifest.loc[
    test_idx,
    "split"
] = "test_15"

split_manifest.to_csv(
    OUT_DIR
    / "slice_level_70_15_15_manifest.csv",
    index=False,
)

patient_overlap_audit.to_csv(
    OUT_DIR
    / "patient_overlap_audit.csv",
    index=False,
)


# ================================================================
# 5. TRANSFORMS
# ================================================================

train_transform = transforms.Compose(
    [
        transforms.Resize(
            (IMG_SIZE, IMG_SIZE)
        ),
        transforms.RandomHorizontalFlip(
            p=0.5
        ),
        transforms.RandomRotation(
            degrees=10
        ),
        transforms.ColorJitter(
            brightness=0.15,
            contrast=0.15,
        ),
        transforms.RandomAffine(
            degrees=0,
            translate=(
                0.03,
                0.03,
            ),
            scale=(
                0.95,
                1.05,
            ),
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=NORMALIZE_MEAN,
            std=NORMALIZE_STD,
        ),
    ]
)

val_transform = transforms.Compose(
    [
        transforms.Resize(
            (IMG_SIZE, IMG_SIZE)
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=NORMALIZE_MEAN,
            std=NORMALIZE_STD,
        ),
    ]
)


# ================================================================
# 6. PATIENT-BALANCED WEIGHTS
# ================================================================

def add_patient_balanced_weights(
    dataframe
):

    out = dataframe.copy()

    patient_slice_counts = (
        out.groupby(
            "patient_id"
        ).size().to_dict()
    )

    class_patient_counts = (
        out.groupby(
            "label"
        )["patient_id"]
        .nunique()
        .to_dict()
    )

    n_total = len(out)

    weights = []

    for _, row in out.iterrows():

        n_slices = (
            patient_slice_counts[
                row.patient_id
            ]
        )

        n_class_patients = (
            class_patient_counts[
                int(row.label)
            ]
        )

        weight = (
            n_total
            * 0.5
            / (
                n_class_patients
                * n_slices
            )
        )

        weights.append(
            weight
        )

    out[
        "sample_weight"
    ] = np.asarray(
        weights,
        dtype=np.float32,
    )

    class_weight_sums = (
        out.groupby(
            "label"
        )["sample_weight"]
        .sum()
    )

    assert np.isclose(
        class_weight_sums.iloc[0],
        class_weight_sums.iloc[1],
        rtol=1e-5,
        atol=1e-5,
    )

    assert np.isclose(
        out.sample_weight.mean(),
        1.0,
        rtol=1e-5,
        atol=1e-5,
    )

    return out


# ================================================================
# 7. DATASET / DATALOADER
# ================================================================

class BrainCTDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform=None,
        use_weights=False,
    ):

        self.df = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.transform = transform
        self.use_weights = use_weights

    def __len__(self):

        return len(self.df)

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[
            idx
        ]

        image = Image.open(
            row.image_path
        ).convert("RGB")

        if self.transform:
            image = self.transform(
                image
            )

        label = torch.tensor(
            float(row.label),
            dtype=torch.float32,
        )

        weight_value = (
            row.sample_weight
            if self.use_weights
            else 1.0
        )

        weight = torch.tensor(
            float(weight_value),
            dtype=torch.float32,
        )

        return (
            image,
            label,
            weight,
            row.patient_id,
            row.image_path,
        )


def create_loader(
    dataframe,
    transform,
    shuffle,
    use_weights=False,
):

    dataset = BrainCTDataset(
        dataframe,
        transform=transform,
        use_weights=use_weights,
    )

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(
            DEVICE.type == "cuda"
        ),
        persistent_workers=False,
    )


# ================================================================
# 8. MODELS
# ================================================================

def create_model(
    model_name
):

    if model_name == "resnet50":

        model = models.resnet50(
            weights=(
                models.ResNet50_Weights.IMAGENET1K_V2
            )
        )

        in_features = (
            model.fc.in_features
        )

        model.fc = nn.Sequential(
            nn.Dropout(
                DROPOUT_P
            ),
            nn.Linear(
                in_features,
                1,
            ),
        )

    elif model_name == "mobilenet_v2":

        model = models.mobilenet_v2(
            weights=(
                models.MobileNet_V2_Weights.IMAGENET1K_V1
            )
        )

        in_features = (
            model.classifier[-1]
            .in_features
        )

        model.classifier = nn.Sequential(
            nn.Dropout(
                DROPOUT_P
            ),
            nn.Linear(
                in_features,
                1,
            ),
        )

    elif model_name == "efficientnet_b0":

        model = models.efficientnet_b0(
            weights=(
                models.EfficientNet_B0_Weights.IMAGENET1K_V1
            )
        )

        in_features = (
            model.classifier[-1]
            .in_features
        )

        model.classifier = nn.Sequential(
            nn.Dropout(
                DROPOUT_P
            ),
            nn.Linear(
                in_features,
                1,
            ),
        )

    else:

        raise ValueError(
            f"Unknown model: {model_name}"
        )

    return model.to(
        DEVICE
    )


def count_parameters(
    model
):

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return (
        total,
        trainable,
    )


# ================================================================
# 9. METRICS
# ================================================================

def calculate_metrics(
    labels,
    probabilities,
    threshold=0.5,
):

    labels = np.asarray(
        labels
    ).astype(int)

    probabilities = np.asarray(
        probabilities
    ).astype(float)

    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = (
        confusion_matrix(
            labels,
            predictions,
            labels=[0, 1],
        ).ravel()
    )

    if len(
        np.unique(labels)
    ) == 2:

        auc = roc_auc_score(
            labels,
            probabilities,
        )

        auprc = average_precision_score(
            labels,
            probabilities,
        )

    else:

        auc = np.nan
        auprc = np.nan

    return {
        "accuracy": accuracy_score(
            labels,
            predictions,
        ),
        "precision": precision_score(
            labels,
            predictions,
            zero_division=0,
        ),
        "sensitivity": recall_score(
            labels,
            predictions,
            zero_division=0,
        ),
        "specificity": (
            tn / (tn + fp)
            if (tn + fp)
            else np.nan
        ),
        "f1": f1_score(
            labels,
            predictions,
            zero_division=0,
        ),
        "auc": auc,
        "auprc": auprc,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def make_slice_prediction_df(
    labels,
    probabilities,
    patient_ids,
    paths,
    split,
    model,
    seed,
):

    labels = np.asarray(
        labels
    ).astype(int)

    probabilities = np.asarray(
        probabilities
    ).astype(float)

    return pd.DataFrame(
        {
            "model": model,
            "seed": seed,
            "split": split,
            "patient_id": list(
                patient_ids
            ),
            "label": labels,
            "probability": probabilities,
            "prediction_0_50": (
                probabilities
                >= PRIMARY_THRESHOLD
            ).astype(int),
            "image_path": list(
                paths
            ),
        }
    )


# ================================================================
# 10. PATIENT AGGREGATION
# ================================================================

def aggregate_patient_predictions(
    slice_prediction_df
):

    rows = []

    for pid, g in (
        slice_prediction_df
        .groupby(
            "patient_id"
        )
    ):

        probabilities = (
            g.probability
            .to_numpy(
                dtype=float
            )
        )

        k = max(
            1,
            int(
                np.ceil(
                    len(probabilities)
                    * 0.10
                )
            ),
        )

        rows.append(
            {
                "patient_id": pid,
                "label": int(
                    g.label.iloc[0]
                ),
                "num_slices_in_split": len(
                    g
                ),
                "mean_probability": float(
                    probabilities.mean()
                ),
                "top10_mean_probability": float(
                    np.mean(
                        np.sort(
                            probabilities
                        )[-k:]
                    )
                ),
                "max_probability": float(
                    probabilities.max()
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


# ================================================================
# 11. DEVELOPMENT-ONLY THRESHOLD
# ================================================================

def select_high_specificity_threshold(
    labels,
    probabilities,
    target_specificity=0.90,
):

    labels = np.asarray(
        labels
    ).astype(int)

    probabilities = np.asarray(
        probabilities
    ).astype(float)

    candidates = np.unique(
        np.concatenate(
            [
                np.array(
                    [0.0, 1.0]
                ),
                probabilities,
            ]
        )
    )

    rows = []

    for threshold in candidates:

        metrics = calculate_metrics(
            labels,
            probabilities,
            float(threshold),
        )

        rows.append(
            {
                "threshold": float(
                    threshold
                ),
                "sensitivity": metrics[
                    "sensitivity"
                ],
                "specificity": metrics[
                    "specificity"
                ],
                "accuracy": metrics[
                    "accuracy"
                ],
                "precision": metrics[
                    "precision"
                ],
                "f1": metrics[
                    "f1"
                ],
            }
        )

    table = pd.DataFrame(
        rows
    )

    eligible = table[
        table.specificity
        >= float(
            target_specificity
        )
    ]

    if len(eligible):

        chosen = (
            eligible
            .sort_values(
                [
                    "sensitivity",
                    "specificity",
                    "threshold",
                ],
                ascending=[
                    False,
                    False,
                    False,
                ],
            )
            .iloc[0]
        )

        fallback = False

    else:

        chosen = (
            table
            .sort_values(
                [
                    "specificity",
                    "sensitivity",
                    "threshold",
                ],
                ascending=[
                    False,
                    False,
                    False,
                ],
            )
            .iloc[0]
        )

        fallback = True

    return (
        float(
            chosen.threshold
        ),
        fallback,
        table,
    )


# ================================================================
# 12. PATIENT-CLUSTER BOOTSTRAP
# ================================================================

def bootstrap_ci_by_patient(
    labels,
    probabilities,
    patient_ids,
    threshold=0.5,
    n_bootstrap=5000,
    seed=4242,
):

    labels = np.asarray(
        labels
    ).astype(int)

    probabilities = np.asarray(
        probabilities
    ).astype(float)

    patient_ids = np.asarray(
        patient_ids
    )

    unique_patients = np.unique(
        patient_ids
    )

    patient_to_indices = {
        pid: np.flatnonzero(
            patient_ids == pid
        )
        for pid in unique_patients
    }

    rng = np.random.default_rng(
        seed
    )

    metric_names = [
        "accuracy",
        "precision",
        "sensitivity",
        "specificity",
        "f1",
        "auc",
        "auprc",
    ]

    samples = {
        name: []
        for name in metric_names
    }

    for _ in range(
        n_bootstrap
    ):

        sampled_patients = rng.choice(
            unique_patients,
            size=len(
                unique_patients
            ),
            replace=True,
        )

        sampled_indices = np.concatenate(
            [
                patient_to_indices[
                    pid
                ]
                for pid
                in sampled_patients
            ]
        )

        y = labels[
            sampled_indices
        ]

        p = probabilities[
            sampled_indices
        ]

        if len(
            np.unique(y)
        ) < 2:
            continue

        metrics = calculate_metrics(
            y,
            p,
            threshold,
        )

        for key in metric_names:

            if np.isfinite(
                metrics[key]
            ):

                samples[
                    key
                ].append(
                    metrics[key]
                )

    ci = {}

    for key in metric_names:

        arr = np.asarray(
            samples[key],
            dtype=float,
        )

        if len(arr) < 100:

            ci[key] = (
                np.nan,
                np.nan,
            )

        else:

            ci[key] = (
                float(
                    np.percentile(
                        arr,
                        2.5
                    )
                ),
                float(
                    np.percentile(
                        arr,
                        97.5
                    )
                ),
            )

    return ci


# ================================================================
# 13. AMP / TRAINING
# ================================================================

def amp_scaler():

    if not USE_AMP:
        return None

    try:

        return torch.amp.GradScaler(
            "cuda",
            enabled=True,
        )

    except Exception:

        return torch.cuda.amp.GradScaler(
            enabled=True
        )


def autocast_context():

    if USE_AMP:

        return torch.amp.autocast(
            device_type="cuda",
            enabled=True,
        )

    return torch.autocast(
        device_type="cpu",
        enabled=False,
    )


def patient_balanced_bce_loss(
    logits,
    labels,
    sample_weights,
):

    per_sample_loss = (
        F.binary_cross_entropy_with_logits(
            logits,
            labels,
            reduction="none",
        )
    )

    weighted_loss = (
        per_sample_loss
        * sample_weights
    )

    return (
        weighted_loss.sum()
        / sample_weights.sum()
        .clamp_min(1e-8)
    )


def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
):

    model.train()

    total_loss = 0.0

    ys = []
    ps = []

    for (
        images,
        labels_batch,
        sample_weights,
        _,
        _,
    ) in loader:

        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        labels_batch = (
            labels_batch.to(
                DEVICE,
                non_blocking=True,
            )
        )

        sample_weights = (
            sample_weights.to(
                DEVICE,
                non_blocking=True,
            )
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with autocast_context():

            logits = model(
                images
            ).squeeze(1)

            loss = patient_balanced_bce_loss(
                logits,
                labels_batch,
                sample_weights,
            )

        if USE_AMP:

            scaler.scale(
                loss
            ).backward()

            scaler.step(
                optimizer
            )

            scaler.update()

        else:

            loss.backward()

            optimizer.step()

        total_loss += (
            float(
                loss.item()
            )
            * images.size(0)
        )

        ys.extend(
            labels_batch.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        ps.extend(
            torch.sigmoid(
                logits
            )
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

    metrics = calculate_metrics(
        ys,
        ps,
        PRIMARY_THRESHOLD,
    )

    return (
        total_loss
        / len(
            loader.dataset
        ),
        metrics,
    )


@torch.no_grad()
def evaluate(
    model,
    loader,
):

    model.eval()

    ys = []
    ps = []
    pids = []
    paths = []

    total_loss = 0.0

    for (
        images,
        labels_batch,
        _,
        patient_ids_batch,
        image_paths_batch,
    ) in loader:

        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        labels_batch = (
            labels_batch.to(
                DEVICE,
                non_blocking=True,
            )
        )

        with autocast_context():

            logits = model(
                images
            ).squeeze(1)

            loss = (
                F.binary_cross_entropy_with_logits(
                    logits,
                    labels_batch,
                    reduction="mean",
                )
            )

        probabilities = torch.sigmoid(
            logits
        )

        total_loss += (
            float(
                loss.item()
            )
            * images.size(0)
        )

        ys.extend(
            labels_batch.cpu()
            .numpy()
            .tolist()
        )

        ps.extend(
            probabilities.cpu()
            .numpy()
            .tolist()
        )

        pids.extend(
            list(
                patient_ids_batch
            )
        )

        paths.extend(
            list(
                image_paths_batch
            )
        )

    metrics = calculate_metrics(
        ys,
        ps,
        PRIMARY_THRESHOLD,
    )

    slice_df = make_slice_prediction_df(
        ys,
        ps,
        pids,
        paths,
        split="unknown",
        model="unknown",
        seed=SEED,
    )

    return {
        "loss": (
            total_loss
            / len(
                loader.dataset
            )
        ),
        "labels": np.asarray(
            ys
        ),
        "probabilities": np.asarray(
            ps
        ),
        "patient_ids": np.asarray(
            pids
        ),
        "paths": np.asarray(
            paths
        ),
        "slice_metrics": metrics,
        "slice_df": slice_df,
    }


# ================================================================
# 14. OPTIMIZER
# ================================================================

def build_optimizer(
    model
):

    if (
        OPTIMIZER_NAME.lower()
        == "adam"
    ):

        return torch.optim.Adam(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )

    if (
        OPTIMIZER_NAME.lower()
        == "adamw"
    ):

        return torch.optim.AdamW(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )

    raise ValueError(
        "OPTIMIZER_NAME must be Adam or AdamW."
    )


# ================================================================
# 15. SLICE-LEVEL CV
# ================================================================

def train_one_cv_fold(
    model_name,
    fold_number,
    fold_train_df,
    fold_val_df,
):

    print()
    print("#" * 120)
    print(
        f"MODEL={model_name} | "
        f"SLICE-CV FOLD={fold_number}/{N_FOLDS}"
    )
    print("#" * 120)

    weighted_train_df = (
        add_patient_balanced_weights(
            fold_train_df
        )
    )

    train_loader = create_loader(
        weighted_train_df,
        train_transform,
        shuffle=True,
        use_weights=True,
    )

    val_loader = create_loader(
        fold_val_df,
        val_transform,
        shuffle=False,
        use_weights=False,
    )

    seed_everything(
        SEED
    )

    model = create_model(
        model_name
    )

    total_params, trainable_params = (
        count_parameters(
            model
        )
    )

    optimizer = build_optimizer(
        model
    )

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=SCHEDULER_FACTOR,
            patience=SCHEDULER_PATIENCE,
            min_lr=SCHEDULER_MIN_LR,
        )
    )

    scaler = amp_scaler()

    history = []

    best_auc = -np.inf
    best_epoch = 0
    best_state = None
    patience_counter = 0

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):

        start = time.time()

        train_loss, train_metrics = (
            train_one_epoch(
                model,
                train_loader,
                optimizer,
                scaler,
            )
        )

        val_eval = evaluate(
            model,
            val_loader,
        )

        val_metrics = (
            val_eval[
                "slice_metrics"
            ]
        )

        val_auc = (
            val_metrics[
                "auc"
            ]
        )

        scheduler.step(
            val_auc
            if np.isfinite(
                val_auc
            )
            else -np.inf
        )

        lr = (
            optimizer
            .param_groups[0]["lr"]
        )

        history.append(
            {
                "model": model_name,
                "seed": SEED,
                "fold": fold_number,
                "epoch": epoch,
                "train_loss": train_loss,
                "train_accuracy": train_metrics[
                    "accuracy"
                ],
                "train_f1": train_metrics[
                    "f1"
                ],
                "train_auc": train_metrics[
                    "auc"
                ],
                "val_loss": val_eval[
                    "loss"
                ],
                "val_accuracy": val_metrics[
                    "accuracy"
                ],
                "val_precision": val_metrics[
                    "precision"
                ],
                "val_sensitivity": val_metrics[
                    "sensitivity"
                ],
                "val_specificity": val_metrics[
                    "specificity"
                ],
                "val_f1": val_metrics[
                    "f1"
                ],
                "val_auc": val_metrics[
                    "auc"
                ],
                "val_auprc": val_metrics[
                    "auprc"
                ],
                "learning_rate": lr,
                "epoch_seconds": (
                    time.time()
                    - start
                ),
            }
        )

        print(
            f"Epoch {epoch:03d} | "
            f"TrainLoss={train_loss:.4f} | "
            f"ValAUC={val_metrics['auc']:.4f} | "
            f"ValAUPRC={val_metrics['auprc']:.4f} | "
            f"LR={lr:.2e}"
        )

        if (
            np.isfinite(
                val_auc
            )
            and val_auc > best_auc
        ):

            best_auc = val_auc

            best_epoch = epoch

            best_state = {
                key: value.detach()
                .cpu()
                .clone()
                for key, value
                in model.state_dict()
                .items()
            }

            patience_counter = 0

        else:

            patience_counter += 1

        if (
            patience_counter
            >= EARLY_STOPPING_PATIENCE
        ):

            print(
                f"Early stopping at epoch {epoch}."
            )

            break

    if best_state is None:

        raise RuntimeError(
            f"No valid best state for "
            f"{model_name}, fold {fold_number}."
        )

    model.load_state_dict(
        best_state
    )

    final_val = evaluate(
        model,
        val_loader,
    )

    val_slice_df = (
        final_val[
            "slice_df"
        ]
        .copy()
    )

    val_slice_df[
        "split"
    ] = "cv_validation"

    val_slice_df[
        "model"
    ] = model_name

    val_slice_df[
        "seed"
    ] = SEED

    val_slice_df[
        "fold"
    ] = fold_number

    result = {
        "model": model_name,
        "fold": fold_number,
        "best_epoch": best_epoch,
        "best_val_auc": best_auc,
        "history_df": pd.DataFrame(
            history
        ),
        "val_eval": final_val,
        "val_slice_df": val_slice_df,
        "total_params": total_params,
        "trainable_params": trainable_params,
    }

    del (
        model,
        optimizer,
        scheduler,
        scaler,
        train_loader,
        val_loader,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# ================================================================
# 16. FINAL MODEL
# ================================================================

def train_final_model(
    model_name,
    num_epochs,
):

    weighted_train_df = (
        add_patient_balanced_weights(
            train_df
        )
    )

    train_loader = create_loader(
        weighted_train_df,
        train_transform,
        shuffle=True,
        use_weights=True,
    )

    seed_everything(
        SEED
    )

    model = create_model(
        model_name
    )

    optimizer = build_optimizer(
        model
    )

    scaler = amp_scaler()

    history = []

    start_time = time.time()

    for epoch in range(
        1,
        num_epochs + 1,
    ):

        epoch_start = time.time()

        loss, metrics = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scaler,
        )

        history.append(
            {
                "model": model_name,
                "seed": SEED,
                "stage": "final_training_70pct",
                "epoch": epoch,
                "train_loss": loss,
                "train_accuracy": metrics[
                    "accuracy"
                ],
                "train_f1": metrics[
                    "f1"
                ],
                "train_auc": metrics[
                    "auc"
                ],
                "learning_rate": optimizer[
                    "param_groups"
                ][0]["lr"],
                "epoch_seconds": (
                    time.time()
                    - epoch_start
                ),
            }
        )

        print(
            f"FINAL {model_name} | "
            f"Epoch {epoch:03d}/{num_epochs} | "
            f"Loss={loss:.4f} | "
            f"TrainAUC={metrics['auc']:.4f}"
        )

    total_seconds = (
        time.time()
        - start_time
    )

    return (
        model,
        pd.DataFrame(
            history
        ),
        total_seconds,
    )


# ================================================================
# 17. LATENCY
# ================================================================

def measure_latency(
    model,
    device,
    repeats,
    warmup,
):

    model = model.to(
        device
    )

    model.eval()

    dummy = torch.randn(
        1,
        3,
        IMG_SIZE,
        IMG_SIZE,
        device=device,
    )

    with torch.inference_mode():

        for _ in range(
            warmup
        ):

            _ = model(
                dummy
            )

        if device.type == "cuda":

            torch.cuda.synchronize()

        timings = []

        for _ in range(
            repeats
        ):

            if device.type == "cuda":

                torch.cuda.synchronize()

            start = (
                time.perf_counter()
            )

            _ = model(
                dummy
            )

            if device.type == "cuda":

                torch.cuda.synchronize()

            timings.append(
                (
                    time.perf_counter()
                    - start
                )
                * 1000.0
            )

    arr = np.asarray(
        timings,
        dtype=float,
    )

    return {
        "mean_ms": float(
            arr.mean()
        ),
        "median_ms": float(
            np.median(arr)
        ),
        "sd_ms": float(
            arr.std(
                ddof=1
            )
        ),
        "p95_ms": float(
            np.percentile(
                arr,
                95
            )
        ),
    }


def try_compute_flops(
    model
):

    try:

        from thop import profile

        model = model.to(
            DEVICE
        )

        model.eval()

        dummy = torch.randn(
            1,
            3,
            IMG_SIZE,
            IMG_SIZE,
            device=DEVICE,
        )

        macs, params = profile(
            model,
            inputs=(dummy,),
            verbose=False,
        )

        flops = (
            2
            * macs
        )

        return (
            float(flops),
            float(macs),
            float(params),
        )

    except Exception as exc:

        print(
            "THOP unavailable:",
            exc
        )

        return (
            np.nan,
            np.nan,
            np.nan,
        )


# ================================================================
# 18. SLICE-LEVEL EXPERIMENT
# ================================================================

print()
print("=" * 120)
print("STARTING SLICE-LEVEL EXPERIMENT")
print("=" * 120)

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED,
)

cv_splits = list(
    skf.split(
        train_df.index.to_numpy(),
        train_df.label.to_numpy(),
    )
)

all_cv_fold_rows = []
all_cv_predictions = []
all_histories = []
all_final_histories = []
all_validation_predictions = []
all_test_predictions = []
all_efficiency = []
all_ci_rows = []
model_selection_rows = []


for model_name in MODEL_NAMES:

    print()
    print("=" * 120)
    print(
        f"START MODEL: {model_name}"
    )
    print("=" * 120)

    cv_results = []

    for (
        fold_number,
        (
            fold_train_positions,
            fold_val_positions,
        ),
    ) in enumerate(
        cv_splits,
        start=1,
    ):

        fold_train_df = (
            train_df.iloc[
                fold_train_positions
            ]
            .reset_index(
                drop=True
            )
        )

        fold_val_df = (
            train_df.iloc[
                fold_val_positions
            ]
            .reset_index(
                drop=True
            )
        )

        result = train_one_cv_fold(
            model_name,
            fold_number,
            fold_train_df,
            fold_val_df,
        )

        cv_results.append(
            result
        )

        all_histories.append(
            result[
                "history_df"
            ]
        )

        all_cv_predictions.append(
            result[
                "val_slice_df"
            ]
        )

        metrics = (
            result[
                "val_eval"
            ][
                "slice_metrics"
            ]
        )

        all_cv_fold_rows.append(
            {
                "model": model_name,
                "seed": SEED,
                "fold": fold_number,
                "train_slices": len(
                    fold_train_df
                ),
                "validation_slices": len(
                    fold_val_df
                ),
                "train_patients": (
                    fold_train_df
                    .patient_id
                    .nunique()
                ),
                "validation_patients": (
                    fold_val_df
                    .patient_id
                    .nunique()
                ),
                "best_epoch": result[
                    "best_epoch"
                ],
                "best_validation_slice_auc": result[
                    "best_val_auc"
                ],
                "validation_accuracy": metrics[
                    "accuracy"
                ],
                "validation_precision": metrics[
                    "precision"
                ],
                "validation_sensitivity": metrics[
                    "sensitivity"
                ],
                "validation_specificity": metrics[
                    "specificity"
                ],
                "validation_f1": metrics[
                    "f1"
                ],
                "validation_auc": metrics[
                    "auc"
                ],
                "validation_auprc": metrics[
                    "auprc"
                ],
            }
        )

    # ------------------------------------------------------------
    # ONE OOF ROW PER TRAINING SLICE
    # ------------------------------------------------------------

    oof_slice = pd.concat(
        [
            r[
                "val_slice_df"
            ]
            for r in cv_results
        ],
        ignore_index=True,
    )

    if len(oof_slice) != len(
        train_df
    ):

        raise RuntimeError(
            f"OOF count mismatch for "
            f"{model_name}: expected "
            f"{len(train_df)}, got "
            f"{len(oof_slice)}"
        )

    if (
        oof_slice
        .image_path
        .nunique()
        != len(train_df)
    ):

        raise RuntimeError(
            f"OOF duplicate/missing slices "
            f"detected for {model_name}."
        )

    if (
        oof_slice
        .image_path
        .duplicated()
        .any()
    ):

        raise RuntimeError(
            f"Duplicate OOF image paths "
            f"detected for {model_name}."
        )

    oof_metrics = calculate_metrics(
        oof_slice.label,
        oof_slice.probability,
        PRIMARY_THRESHOLD,
    )

    oof_patient = (
        aggregate_patient_predictions(
            oof_slice
        )
    )

    oof_patient_metrics = (
        calculate_metrics(
            oof_patient.label,
            oof_patient.mean_probability,
            PRIMARY_THRESHOLD,
        )
    )

    # ------------------------------------------------------------
    # HIGH SPECIFICITY THRESHOLD
    # ------------------------------------------------------------

    (
        high_spec_threshold,
        threshold_fallback,
        threshold_table,
    ) = select_high_specificity_threshold(
        oof_slice.label.to_numpy(),
        oof_slice.probability.to_numpy(),
        HIGH_SPEC_TARGET,
    )

    threshold_table[
        "model"
    ] = model_name

    threshold_table[
        "seed"
    ] = SEED

    threshold_table.to_csv(
        OUT_DIR
        / f"{model_name}_oof_slice_threshold_candidates.csv",
        index=False,
    )

    high_spec_oof_metrics = (
        calculate_metrics(
            oof_slice.label,
            oof_slice.probability,
            high_spec_threshold,
        )
    )

    print()
    print(
        f"{model_name} OOF AUROC: "
        f"{oof_metrics['auc']:.4f}"
    )

    print(
        f"{model_name} OOF AUPRC: "
        f"{oof_metrics['auprc']:.4f}"
    )

    print(
        f"OOF high-specificity threshold: "
        f"{high_spec_threshold:.4f}"
    )

    # ------------------------------------------------------------
    # FINAL EPOCH COUNT
    # ------------------------------------------------------------

    best_epochs = [
        r[
            "best_epoch"
        ]
        for r in cv_results
    ]

    final_epochs = max(
        1,
        min(
            int(
                np.median(
                    best_epochs
                )
            ),
            MAX_EPOCHS,
        ),
    )

    model_selection_rows.append(
        {
            "model": model_name,
            "seed": SEED,
            "mean_cv_slice_auc": float(
                np.mean(
                    [
                        r[
                            "best_val_auc"
                        ]
                        for r in cv_results
                    ]
                )
            ),
            "sd_cv_slice_auc": float(
                np.std(
                    [
                        r[
                            "best_val_auc"
                        ]
                        for r in cv_results
                    ],
                    ddof=1,
                )
            ),
            "mean_cv_slice_auprc": float(
                np.mean(
                    [
                        r[
                            "val_eval"
                        ][
                            "slice_metrics"
                        ][
                            "auprc"
                        ]
                        for r in cv_results
                    ]
                )
            ),
            "sd_cv_slice_auprc": float(
                np.std(
                    [
                        r[
                            "val_eval"
                        ][
                            "slice_metrics"
                        ][
                            "auprc"
                        ]
                        for r in cv_results
                    ],
                    ddof=1,
                )
            ),
            "oof_slice_auc": float(
                oof_metrics[
                    "auc"
                ]
            ),
            "oof_slice_auprc": float(
                oof_metrics[
                    "auprc"
                ]
            ),
            "mean_best_epoch": float(
                np.mean(
                    best_epochs
                )
            ),
            "median_best_epoch": int(
                final_epochs
            ),
        }
    )

    # ------------------------------------------------------------
    # FINAL TRAINING ON 70%
    # ------------------------------------------------------------

    seed_everything(
        SEED
    )

    (
        final_model,
        final_history,
        final_training_seconds,
    ) = train_final_model(
        model_name,
        final_epochs,
    )

    all_final_histories.append(
        final_history
    )

    model_path = (
        OUT_DIR
        / f"{model_name}_seed{SEED}_slice_level_final.pth"
    )

    torch.save(
        {
            "model_name": model_name,
            "seed": SEED,
            "protocol": (
                "slice_level_70_15_15"
            ),
            "cv_protocol": (
                "4_fold_slice_level_CV_"
                "within_70pct_train"
            ),
            "img_size": IMG_SIZE,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "optimizer": OPTIMIZER_NAME,
            "dropout": DROPOUT_P,
            "normalization_mean": (
                NORMALIZE_MEAN
            ),
            "normalization_std": (
                NORMALIZE_STD
            ),
            "primary_threshold": (
                PRIMARY_THRESHOLD
            ),
            "high_spec_target": (
                HIGH_SPEC_TARGET
            ),
            "high_spec_threshold_from_oof": (
                high_spec_threshold
            ),
            "cv_best_epochs": (
                best_epochs
            ),
            "final_training_epochs": (
                final_epochs
            ),
            "model_state_dict": (
                final_model.state_dict()
            ),
        },
        model_path,
    )

    # ------------------------------------------------------------
    # EFFICIENCY
    # ------------------------------------------------------------

    total_params, trainable_params = (
        count_parameters(
            final_model
        )
    )

    gpu_latency = measure_latency(
        final_model,
        DEVICE,
        LATENCY_REPEATS_GPU,
        LATENCY_WARMUP,
    )

    cpu_latency = {
        "mean_ms": np.nan,
        "median_ms": np.nan,
        "sd_ms": np.nan,
        "p95_ms": np.nan,
    }

    try:

        cpu_latency = measure_latency(
            final_model,
            torch.device("cpu"),
            LATENCY_REPEATS_CPU,
            20,
        )

    except Exception as exc:

        print(
            "CPU latency unavailable:",
            exc
        )

    flops, macs, thop_params = (
        try_compute_flops(
            final_model
        )
    )

    final_model = final_model.to(
        DEVICE
    )

    model_size_mb = (
        model_path.stat().st_size
        / (1024 ** 2)
    )

    all_efficiency.append(
        {
            "model": model_name,
            "seed": SEED,
            "total_parameters": (
                total_params
            ),
            "trainable_parameters": (
                trainable_params
            ),
            "parameters_million": (
                total_params
                / 1e6
            ),
            "model_size_mb": (
                model_size_mb
            ),
            "gpu_latency_mean_ms_per_image": (
                gpu_latency[
                    "mean_ms"
                ]
            ),
            "gpu_latency_median_ms_per_image": (
                gpu_latency[
                    "median_ms"
                ]
            ),
            "gpu_latency_sd_ms_per_image": (
                gpu_latency[
                    "sd_ms"
                ]
            ),
            "gpu_latency_p95_ms_per_image": (
                gpu_latency[
                    "p95_ms"
                ]
            ),
            "cpu_latency_mean_ms_per_image": (
                cpu_latency[
                    "mean_ms"
                ]
            ),
            "cpu_latency_median_ms_per_image": (
                cpu_latency[
                    "median_ms"
                ]
            ),
            "cpu_latency_sd_ms_per_image": (
                cpu_latency[
                    "sd_ms"
                ]
            ),
            "cpu_latency_p95_ms_per_image": (
                cpu_latency[
                    "p95_ms"
                ]
            ),
            "FLOPs": flops,
            "MACs": macs,
            "thop_parameters": thop_params,
            "final_training_seconds": (
                final_training_seconds
            ),
        }
    )

    # ------------------------------------------------------------
    # VALIDATION
    # ------------------------------------------------------------

    val_loader_final = create_loader(
        val_df,
        val_transform,
        shuffle=False,
        use_weights=False,
    )

    val_eval_final = evaluate(
        final_model,
        val_loader_final,
    )

    val_slice_pred = (
        make_slice_prediction_df(
            val_eval_final[
                "labels"
            ],
            val_eval_final[
                "probabilities"
            ],
            val_eval_final[
                "patient_ids"
            ],
            val_eval_final[
                "paths"
            ],
            split="validation_15",
            model=model_name,
            seed=SEED,
        )
    )

    val_slice_pred[
        "prediction_high_spec"
    ] = (
        val_slice_pred.probability
        >= high_spec_threshold
    ).astype(int)

    all_validation_predictions.append(
        val_slice_pred
    )

    # ------------------------------------------------------------
    # FINAL TEST
    # ------------------------------------------------------------

    test_loader_final = create_loader(
        test_df,
        val_transform,
        shuffle=False,
        use_weights=False,
    )

    test_eval_final = evaluate(
        final_model,
        test_loader_final,
    )

    test_slice_pred = (
        make_slice_prediction_df(
            test_eval_final[
                "labels"
            ],
            test_eval_final[
                "probabilities"
            ],
            test_eval_final[
                "patient_ids"
            ],
            test_eval_final[
                "paths"
            ],
            split="test_15",
            model=model_name,
            seed=SEED,
        )
    )

    test_slice_pred[
        "prediction_high_spec"
    ] = (
        test_slice_pred.probability
        >= high_spec_threshold
    ).astype(int)

    all_test_predictions.append(
        test_slice_pred
    )

    test_metrics = calculate_metrics(
        test_slice_pred.label,
        test_slice_pred.probability,
        PRIMARY_THRESHOLD,
    )

    test_high_spec_metrics = (
        calculate_metrics(
            test_slice_pred.label,
            test_slice_pred.probability,
            high_spec_threshold,
        )
    )

    # ------------------------------------------------------------
    # DESCRIPTIVE PATIENT AGGREGATION
    # ------------------------------------------------------------

    test_patient_pred = (
        aggregate_patient_predictions(
            test_slice_pred
        )
    )

    test_patient_metrics = (
        calculate_metrics(
            test_patient_pred.label,
            test_patient_pred.mean_probability,
            PRIMARY_THRESHOLD,
        )
    )

    test_patient_pred.to_csv(
        OUT_DIR
        / f"{model_name}_test_patient_aggregated_descriptive.csv",
        index=False,
    )

    # ------------------------------------------------------------
    # CLUSTER BOOTSTRAP
    # ------------------------------------------------------------

    ci_primary = (
        bootstrap_ci_by_patient(
            test_slice_pred.label.to_numpy(),
            test_slice_pred.probability.to_numpy(),
            test_slice_pred.patient_id.to_numpy(),
            threshold=PRIMARY_THRESHOLD,
            n_bootstrap=N_BOOTSTRAP,
            seed=BOOTSTRAP_SEED,
        )
    )

    ci_high_spec = (
        bootstrap_ci_by_patient(
            test_slice_pred.label.to_numpy(),
            test_slice_pred.probability.to_numpy(),
            test_slice_pred.patient_id.to_numpy(),
            threshold=high_spec_threshold,
            n_bootstrap=N_BOOTSTRAP,
            seed=BOOTSTRAP_SEED + 1,
        )
    )

    ci_row = {
        "model": model_name,
        "seed": SEED,
        "test_slices": len(
            test_slice_pred
        ),
        "test_patients": (
            test_slice_pred.patient_id
            .nunique()
        ),
        "primary_threshold": (
            PRIMARY_THRESHOLD
        ),

        "accuracy": (
            test_metrics[
                "accuracy"
            ]
        ),
        "accuracy_ci_low": (
            ci_primary[
                "accuracy"
            ][0]
        ),
        "accuracy_ci_high": (
            ci_primary[
                "accuracy"
            ][1]
        ),

        "precision": (
            test_metrics[
                "precision"
            ]
        ),
        "precision_ci_low": (
            ci_primary[
                "precision"
            ][0]
        ),
        "precision_ci_high": (
            ci_primary[
                "precision"
            ][1]
        ),

        "sensitivity": (
            test_metrics[
                "sensitivity"
            ]
        ),
        "sensitivity_ci_low": (
            ci_primary[
                "sensitivity"
            ][0]
        ),
        "sensitivity_ci_high": (
            ci_primary[
                "sensitivity"
            ][1]
        ),

        "specificity": (
            test_metrics[
                "specificity"
            ]
        ),
        "specificity_ci_low": (
            ci_primary[
                "specificity"
            ][0]
        ),
        "specificity_ci_high": (
            ci_primary[
                "specificity"
            ][1]
        ),

        "f1": (
            test_metrics[
                "f1"
            ]
        ),
        "f1_ci_low": (
            ci_primary[
                "f1"
            ][0]
        ),
        "f1_ci_high": (
            ci_primary[
                "f1"
            ][1]
        ),

        "auc": (
            test_metrics[
                "auc"
            ]
        ),
        "auc_ci_low": (
            ci_primary[
                "auc"
            ][0]
        ),
        "auc_ci_high": (
            ci_primary[
                "auc"
            ][1]
        ),

        "auprc": (
            test_metrics[
                "auprc"
            ]
        ),
        "auprc_ci_low": (
            ci_primary[
                "auprc"
            ][0]
        ),
        "auprc_ci_high": (
            ci_primary[
                "auprc"
            ][1]
        ),

        "tn": test_metrics["tn"],
        "fp": test_metrics["fp"],
        "fn": test_metrics["fn"],
        "tp": test_metrics["tp"],

        "high_spec_threshold_from_oof": (
            high_spec_threshold
        ),
        "high_spec_sensitivity": (
            test_high_spec_metrics[
                "sensitivity"
            ]
        ),
        "high_spec_sensitivity_ci_low": (
            ci_high_spec[
                "sensitivity"
            ][0]
        ),
        "high_spec_sensitivity_ci_high": (
            ci_high_spec[
                "sensitivity"
            ][1]
        ),
        "high_spec_specificity": (
            test_high_spec_metrics[
                "specificity"
            ]
        ),
        "high_spec_specificity_ci_low": (
            ci_high_spec[
                "specificity"
            ][0]
        ),
        "high_spec_specificity_ci_high": (
            ci_high_spec[
                "specificity"
            ][1]
        ),

        "descriptive_patient_aggregated_auc": (
            test_patient_metrics[
                "auc"
            ]
        ),
        "descriptive_patient_aggregated_auprc": (
            test_patient_metrics[
                "auprc"
            ]
        ),
        "descriptive_patient_aggregated_accuracy": (
            test_patient_metrics[
                "accuracy"
            ]
        ),
        "descriptive_patient_aggregated_sensitivity": (
            test_patient_metrics[
                "sensitivity"
            ]
        ),
        "descriptive_patient_aggregated_specificity": (
            test_patient_metrics[
                "specificity"
            ]
        ),
        "descriptive_patient_aggregated_f1": (
            test_patient_metrics[
                "f1"
            ]
        ),
    }

    all_ci_rows.append(
        ci_row
    )

    # ------------------------------------------------------------
    # SAVE PREDICTIONS
    # ------------------------------------------------------------

    oof_slice.to_csv(
        OUT_DIR
        / f"{model_name}_slice_oof_predictions.csv",
        index=False,
    )

    val_slice_pred.to_csv(
        OUT_DIR
        / f"{model_name}_slice_validation15_predictions.csv",
        index=False,
    )

    test_slice_pred.to_csv(
        OUT_DIR
        / f"{model_name}_slice_test15_predictions.csv",
        index=False,
    )

    print()
    print(
        f"{model_name} TEST:"
    )

    print(
        f"AUROC={test_metrics['auc']:.4f}"
        f" | AUPRC={test_metrics['auprc']:.4f}"
        f" | Sensitivity={test_metrics['sensitivity']:.4f}"
        f" | Specificity={test_metrics['specificity']:.4f}"
        f" | F1={test_metrics['f1']:.4f}"
    )

    del (
        final_model,
        val_loader_final,
        test_loader_final,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ================================================================
# 19. CONSOLIDATED RESULTS
# ================================================================

cv_fold_results = pd.DataFrame(
    all_cv_fold_rows
)

cv_model_summary = (
    cv_fold_results
    .groupby(
        "model"
    )
    .agg(
        mean_cv_auc=(
            "validation_auc",
            "mean"
        ),
        sd_cv_auc=(
            "validation_auc",
            "std"
        ),
        mean_cv_auprc=(
            "validation_auprc",
            "mean"
        ),
        sd_cv_auprc=(
            "validation_auprc",
            "std"
        ),
        mean_cv_f1=(
            "validation_f1",
            "mean"
        ),
        sd_cv_f1=(
            "validation_f1",
            "std"
        ),
        mean_best_epoch=(
            "best_epoch",
            "mean"
        ),
        median_best_epoch=(
            "best_epoch",
            "median"
        ),
    )
    .reset_index()
)

model_selection = (
    pd.DataFrame(
        model_selection_rows
    )
    .sort_values(
        [
            "mean_cv_slice_auc",
            "mean_cv_slice_auprc",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

model_selection[
    "cv_rank"
] = np.arange(
    1,
    len(model_selection) + 1
)

selected_model = (
    model_selection.iloc[0][
        "model"
    ]
)

efficiency_df = pd.DataFrame(
    all_efficiency
)

efficiency_summary = (
    efficiency_df
    .groupby(
        "model"
    )
    .agg(
        parameters_million=(
            "parameters_million",
            "mean"
        ),
        model_size_mb=(
            "model_size_mb",
            "mean"
        ),
        FLOPs=(
            "FLOPs",
            "mean"
        ),
        MACs=(
            "MACs",
            "mean"
        ),
        gpu_latency_mean_ms=(
            "gpu_latency_mean_ms_per_image",
            "mean"
        ),
        gpu_latency_median_ms=(
            "gpu_latency_median_ms_per_image",
            "mean"
        ),
        cpu_latency_mean_ms=(
            "cpu_latency_mean_ms_per_image",
            "mean"
        ),
        cpu_latency_median_ms=(
            "cpu_latency_median_ms_per_image",
            "mean"
        ),
    )
    .reset_index()
)

if (
    "resnet50"
    in efficiency_summary.model.values
):

    resnet_ref = (
        efficiency_summary[
            efficiency_summary.model
            == "resnet50"
        ]
        .iloc[0]
    )

    efficiency_summary[
        "parameter_reduction_vs_resnet50"
    ] = (
        1.0
        - efficiency_summary.parameters_million
        / resnet_ref.parameters_million
    )

    efficiency_summary[
        "model_size_reduction_vs_resnet50"
    ] = (
        1.0
        - efficiency_summary.model_size_mb
        / resnet_ref.model_size_mb
    )


cv_predictions_all = pd.concat(
    all_cv_predictions,
    ignore_index=True,
)

validation_predictions_all = pd.concat(
    all_validation_predictions,
    ignore_index=True,
)

test_predictions_all = pd.concat(
    all_test_predictions,
    ignore_index=True,
)

ci_df = pd.DataFrame(
    all_ci_rows
)

history_df = pd.concat(
    all_histories,
    ignore_index=True,
)

final_history_df = pd.concat(
    all_final_histories,
    ignore_index=True,
)


# ================================================================
# 20. SAVE RESULT TABLES
# ================================================================

cv_fold_results.to_csv(
    OUT_DIR
    / "slice_level_cv_fold_results.csv",
    index=False,
)

cv_model_summary.to_csv(
    OUT_DIR
    / "slice_level_cv_model_summary.csv",
    index=False,
)

model_selection.to_csv(
    OUT_DIR
    / "slice_level_model_selection.csv",
    index=False,
)

efficiency_df.to_csv(
    OUT_DIR
    / "slice_level_efficiency_per_model.csv",
    index=False,
)

efficiency_summary.to_csv(
    OUT_DIR
    / "slice_level_efficiency_summary.csv",
    index=False,
)

cv_predictions_all.to_csv(
    OUT_DIR
    / "slice_level_all_oof_predictions.csv",
    index=False,
)

validation_predictions_all.to_csv(
    OUT_DIR
    / "slice_level_all_validation15_predictions.csv",
    index=False,
)

test_predictions_all.to_csv(
    OUT_DIR
    / "slice_level_all_test15_predictions.csv",
    index=False,
)

ci_df.to_csv(
    OUT_DIR
    / "slice_level_test_95CI.csv",
    index=False,
)

history_df.to_csv(
    OUT_DIR
    / "slice_level_cv_training_history.csv",
    index=False,
)

final_history_df.to_csv(
    OUT_DIR
    / "slice_level_final_training_history.csv",
    index=False,
)


# ================================================================
# 21. PAPER-READY TABLES
# ================================================================

slice_main_table = (
    ci_df[
        [
            "model",
            "test_slices",
            "test_patients",
            "auc",
            "auc_ci_low",
            "auc_ci_high",
            "auprc",
            "auprc_ci_low",
            "auprc_ci_high",
        ]
    ]
    .copy()
    .rename(
        columns={
            "auc": "test_auroc",
            "auc_ci_low": (
                "test_auroc_ci_low"
            ),
            "auc_ci_high": (
                "test_auroc_ci_high"
            ),
            "auprc": "test_auprc",
            "auprc_ci_low": (
                "test_auprc_ci_low"
            ),
            "auprc_ci_high": (
                "test_auprc_ci_high"
            ),
        }
    )
)

slice_main_table.to_csv(
    OUT_DIR
    / "TABLE_slice_level_main_results.csv",
    index=False,
)

slice_cv_table = (
    cv_model_summary[
        [
            "model",
            "mean_cv_auc",
            "sd_cv_auc",
            "mean_cv_auprc",
            "sd_cv_auprc",
            "mean_best_epoch",
            "median_best_epoch",
        ]
    ]
    .copy()
    .rename(
        columns={
            "mean_cv_auc": (
                "cv_auroc_mean"
            ),
            "sd_cv_auc": (
                "cv_auroc_sd"
            ),
            "mean_cv_auprc": (
                "cv_auprc_mean"
            ),
            "sd_cv_auprc": (
                "cv_auprc_sd"
            ),
        }
    )
)

slice_cv_table.to_csv(
    OUT_DIR
    / "TABLE_slice_level_cv_results.csv",
    index=False,
)


# ================================================================
# 22. PUBLICATION FIGURES
# ================================================================

PLOT_DPI = 600

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 12,
        "axes.titlesize": 16,
        "axes.labelsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 11,
        "axes.linewidth": 1.0,
        "figure.dpi": 150,
        "savefig.dpi": PLOT_DPI,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)


def save_publication_figure(
    fig,
    filename_stem,
):

    png_path = (
        OUT_DIR
        / f"{filename_stem}.png"
    )

    pdf_path = (
        OUT_DIR
        / f"{filename_stem}.pdf"
    )

    fig.savefig(
        png_path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
        facecolor="white",
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.close(
        fig
    )


plot_order = MODEL_NAMES

display_names = {
    "resnet50": "ResNet-50",
    "mobilenet_v2": "MobileNetV2",
    "efficientnet_b0": "EfficientNet-B0",
}


# ---------------------------------------------------------------
# CV AUROC
# ---------------------------------------------------------------

plot_df = (
    cv_model_summary
    .set_index("model")
    .reindex(
        plot_order
    )
)

x = np.arange(
    len(plot_order)
)

values = (
    plot_df[
        "mean_cv_auc"
    ]
    .to_numpy(
        dtype=float
    )
)

errors = (
    plot_df[
        "sd_cv_auc"
    ]
    .fillna(0)
    .to_numpy(
        dtype=float
    )
)

fig, ax = plt.subplots(
    figsize=(9.5, 6.2)
)

bars = ax.bar(
    x,
    values,
    yerr=errors,
    capsize=5,
    width=0.62,
)

for bar, value in zip(
    bars,
    values
):

    ax.text(
        bar.get_x()
        + bar.get_width()
        / 2,
        min(
            float(value)
            + 0.015,
            1.03,
        ),
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontweight="bold",
    )

ax.set_xticks(x)

ax.set_xticklabels(
    [
        display_names[
            m
        ]
        for m in plot_order
    ]
)

ax.set_xlabel(
    "Model"
)

ax.set_ylabel(
    "Mean 4-fold CV AUROC"
)

ax.set_title(
    "Slice-Level Cross-Validation AUROC"
)

ax.set_ylim(
    0,
    1.08
)

ax.grid(
    axis="y",
    alpha=0.25,
)

save_publication_figure(
    fig,
    "fig2_slice_level_cv_auroc",
)


# ---------------------------------------------------------------
# CV AUPRC
# ---------------------------------------------------------------

values = (
    plot_df[
        "mean_cv_auprc"
    ]
    .to_numpy(
        dtype=float
    )
)

errors = (
    plot_df[
        "sd_cv_auprc"
    ]
    .fillna(0)
    .to_numpy(
        dtype=float
    )
)

fig, ax = plt.subplots(
    figsize=(9.5, 6.2)
)

bars = ax.bar(
    x,
    values,
    yerr=errors,
    capsize=5,
    width=0.62,
)

for bar, value in zip(
    bars,
    values
):

    ax.text(
        bar.get_x()
        + bar.get_width()
        / 2,
        min(
            float(value)
            + 0.015,
            1.03,
        ),
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontweight="bold",
    )

ax.set_xticks(x)

ax.set_xticklabels(
    [
        display_names[
            m
        ]
        for m in plot_order
    ]
)

ax.set_xlabel(
    "Model"
)

ax.set_ylabel(
    "Mean 4-fold CV AUPRC"
)

ax.set_title(
    "Slice-Level Cross-Validation AUPRC"
)

ax.set_ylim(
    0,
    1.08
)

ax.grid(
    axis="y",
    alpha=0.25,
)

save_publication_figure(
    fig,
    "fig3_slice_level_cv_auprc",
)


# ---------------------------------------------------------------
# TEST AUROC
# ---------------------------------------------------------------

tp = (
    ci_df
    .set_index("model")
    .reindex(
        plot_order
    )
)

y = (
    tp[
        "auc"
    ]
    .to_numpy(
        dtype=float
    )
)

yerr_low = (
    y
    - tp[
        "auc_ci_low"
    ].to_numpy(
        dtype=float
    )
)

yerr_high = (
    tp[
        "auc_ci_high"
    ].to_numpy(
        dtype=float
    )
    - y
)

fig, ax = plt.subplots(
    figsize=(9.5, 6.2)
)

bars = ax.bar(
    x,
    y,
    yerr=np.vstack(
        [
            yerr_low,
            yerr_high,
        ]
    ),
    capsize=6,
    width=0.62,
)

for bar, value in zip(
    bars,
    y
):

    ax.text(
        bar.get_x()
        + bar.get_width()
        / 2,
        min(
            float(value)
            + 0.015,
            1.03,
        ),
        f"{value:.4f}",
        ha="center",
        va="bottom",
        fontweight="bold",
    )

ax.set_xticks(x)

ax.set_xticklabels(
    [
        display_names[
            m
        ]
        for m in plot_order
    ]
)

ax.set_xlabel(
    "Model"
)

ax.set_ylabel(
    "Held-out slice-level AUROC"
)

ax.set_title(
    "Slice-Level Test AUROC "
    "With Patient-Cluster Bootstrap 95% CI"
)

ax.set_ylim(
    0,
    1.08
)

ax.grid(
    axis="y",
    alpha=0.25,
)

save_publication_figure(
    fig,
    "fig4_slice_level_test_auroc_ci",
)


# ---------------------------------------------------------------
# TEST AUPRC
# ---------------------------------------------------------------

y = (
    tp[
        "auprc"
    ]
    .to_numpy(
        dtype=float
    )
)

yerr_low = (
    y
    - tp[
        "auprc_ci_low"
    ].to_numpy(
        dtype=float
    )
)

yerr_high = (
    tp[
        "auprc_ci_high"
    ].to_numpy(
        dtype=float
    )
    - y
)

fig, ax = plt.subplots(
    figsize=(9.5, 6.2)
)

bars = ax.bar(
    x,
    y,
    yerr=np.vstack(
        [
            yerr_low,
            yerr_high,
        ]
    ),
    capsize=6,
    width=0.62,
)

for bar, value in zip(
    bars,
    y
):

    ax.text(
        bar.get_x()
        + bar.get_width()
        / 2,
        min(
            float(value)
            + 0.015,
            1.03,
        ),
        f"{value:.4f}",
        ha="center",
        va="bottom",
        fontweight="bold",
    )

ax.set_xticks(x)

ax.set_xticklabels(
    [
        display_names[
            m
        ]
        for m in plot_order
    ]
)

ax.set_xlabel(
    "Model"
)

ax.set_ylabel(
    "Held-out slice-level AUPRC"
)

ax.set_title(
    "Slice-Level Test AUPRC "
    "With Patient-Cluster Bootstrap 95% CI"
)

ax.set_ylim(
    0,
    1.08
)

ax.grid(
    axis="y",
    alpha=0.25,
)

save_publication_figure(
    fig,
    "fig5_slice_level_test_auprc_ci",
)


# ---------------------------------------------------------------
# TEST ROC
# ---------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(7.8, 7.2)
)

for model_name in MODEL_NAMES:

    g = test_predictions_all[
        test_predictions_all.model
        == model_name
    ]

    if (
        len(g) == 0
        or g.label.nunique()
        < 2
    ):

        continue

    fpr, tpr, _ = roc_curve(
        g.label.to_numpy(),
        g.probability.to_numpy(),
    )

    auc_value = roc_auc_score(
        g.label.to_numpy(),
        g.probability.to_numpy(),
    )

    ax.plot(
        fpr,
        tpr,
        linewidth=2.4,
        label=(
            f"{display_names[model_name]} "
            f"(AUROC={auc_value:.4f})"
        ),
    )

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.4,
    label="Chance",
)

ax.set_xlabel(
    "False Positive Rate"
)

ax.set_ylabel(
    "True Positive Rate"
)

ax.set_title(
    "Slice-Level Held-Out Test ROC Curves"
)

ax.set_xlim(
    0,
    1
)

ax.set_ylim(
    0,
    1.01
)

ax.grid(
    alpha=0.25
)

ax.legend(
    loc="lower right"
)

save_publication_figure(
    fig,
    "fig6_slice_level_test_roc_curves",
)


# ---------------------------------------------------------------
# TEST PR
# ---------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(7.8, 7.2)
)

positive_prevalence = (
    test_df["label"].mean()
)

for model_name in MODEL_NAMES:

    g = test_predictions_all[
        test_predictions_all.model
        == model_name
    ]

    if (
        len(g) == 0
        or g.label.nunique()
        < 2
    ):

        continue

    precision, recall, _ = (
        precision_recall_curve(
            g.label.to_numpy(),
            g.probability.to_numpy(),
        )
    )

    auprc_value = (
        average_precision_score(
            g.label.to_numpy(),
            g.probability.to_numpy(),
        )
    )

    ax.plot(
        recall,
        precision,
        linewidth=2.4,
        label=(
            f"{display_names[model_name]} "
            f"(AUPRC={auprc_value:.4f})"
        ),
    )

ax.axhline(
    positive_prevalence,
    linestyle="--",
    linewidth=1.4,
    label=(
        "No-skill prevalence="
        f"{positive_prevalence:.3f}"
    ),
)

ax.set_xlabel(
    "Recall"
)

ax.set_ylabel(
    "Precision"
)

ax.set_title(
    "Slice-Level Held-Out Test "
    "Precision-Recall Curves"
)

ax.set_xlim(
    0,
    1
)

ax.set_ylim(
    0,
    1.01
)

ax.grid(
    alpha=0.25
)

ax.legend(
    loc="lower left"
)

save_publication_figure(
    fig,
    "fig7_slice_level_test_pr_curves",
)


# ---------------------------------------------------------------
# PATIENT OVERLAP HEATMAP
# ---------------------------------------------------------------

patient_split = (
    split_manifest
    .assign(
        present=1
    )
    .pivot_table(
        index="patient_id",
        columns="split",
        values="present",
        aggfunc="max",
        fill_value=0,
    )
)

ordered_columns = [
    c
    for c in [
        "train_70",
        "validation_15",
        "test_15",
    ]
    if c in patient_split.columns
]

patient_split = (
    patient_split[
        ordered_columns
    ]
    .sort_index()
)

fig, ax = plt.subplots(
    figsize=(
        7.0,
        max(
            7.0,
            0.18
            * len(
                patient_split
            )
        ),
    )
)

im = ax.imshow(
    patient_split.to_numpy(),
    aspect="auto",
    interpolation="nearest",
    cmap="Greys",
    vmin=0,
    vmax=1,
)

ax.set_xticks(
    np.arange(
        len(
            ordered_columns
        )
    )
)

ax.set_xticklabels(
    [
        "Train (70%)",
        "Validation (15%)",
        "Test (15%)",
    ]
)

ax.set_yticks(
    np.arange(
        len(
            patient_split
        )
    )
)

ax.set_yticklabels(
    patient_split.index
)

ax.set_xlabel(
    "Slice-level partition"
)

ax.set_ylabel(
    "Patient"
)

ax.set_title(
    "Patient Presence Across "
    "Slice-Level Partitions"
)

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.045,
    pad=0.03,
)

cbar.set_ticks(
    [0, 1]
)

cbar.set_ticklabels(
    [
        "Absent",
        "Present",
    ]
)

ax.grid(
    which="major",
    color="white",
    linewidth=0.7,
)

save_publication_figure(
    fig,
    "fig8_slice_level_patient_overlap_heatmap",
)


# ---------------------------------------------------------------
# PARAMETER COUNT
# ---------------------------------------------------------------

ep = (
    efficiency_summary
    .set_index("model")
    .reindex(
        plot_order
    )
)

values = (
    ep[
        "parameters_million"
    ]
    .to_numpy(
        dtype=float
    )
)

fig, ax = plt.subplots(
    figsize=(9.5, 6.2)
)

bars = ax.bar(
    x,
    values,
    width=0.62,
)

for bar, value in zip(
    bars,
    values
):

    ax.text(
        bar.get_x()
        + bar.get_width()
        / 2,
        value
        + max(values)
        * 0.025,
        f"{value:.2f}M",
        ha="center",
        va="bottom",
        fontweight="bold",
    )

ax.set_xticks(x)

ax.set_xticklabels(
    [
        display_names[
            m
        ]
        for m in plot_order
    ]
)

ax.set_xlabel(
    "Model"
)

ax.set_ylabel(
    "Parameters (millions)"
)

ax.set_title(
    "Model Parameter Count"
)

ax.grid(
    axis="y",
    alpha=0.25
)

save_publication_figure(
    fig,
    "fig9_model_parameter_count",
)


# ---------------------------------------------------------------
# CPU LATENCY
# ---------------------------------------------------------------

values = (
    ep[
        "cpu_latency_mean_ms"
    ]
    .to_numpy(
        dtype=float
    )
)

if np.isfinite(
    values
).any():

    fig, ax = plt.subplots(
        figsize=(9.5, 6.2)
    )

    bars = ax.bar(
        x,
        values,
        width=0.62,
    )

    finite_max = np.nanmax(
        values
    )

    for bar, value in zip(
        bars,
        values
    ):

        if np.isfinite(
            value
        ):

            ax.text(
                bar.get_x()
                + bar.get_width()
                / 2,
                value
                + finite_max
                * 0.025,
                f"{value:.2f}",
                ha="center",
                va="bottom",
                fontweight="bold",
            )

    ax.set_xticks(x)

    ax.set_xticklabels(
        [
            display_names[
                m
            ]
            for m in plot_order
        ]
    )

    ax.set_xlabel(
        "Model"
    )

    ax.set_ylabel(
        "CPU inference latency (ms/image)"
    )

    ax.set_title(
        "CPU Inference Latency"
    )

    ax.grid(
        axis="y",
        alpha=0.25
    )

    save_publication_figure(
        fig,
        "fig10_cpu_latency",
    )


# ================================================================
# 23. GRAD-CAM
# ================================================================
#
# This section is deliberately independent of the patient-level
# pipeline.
#
# REQUIRED DATA:
#   test_predictions_all
#
# REQUIRED CHECKPOINT:
#   *_seed42_slice_level_final.pth
#
# NO held_out_test_df is used anywhere.
#
# ================================================================

GRADCAM_TOP_FRACTION = 0.12

GRADCAM_COMPONENTS = 3

GRADCAM_MIN_COMPONENT_AREA = (
    0.0025
)

GRADCAM_ALPHA_MAX = 0.72

FAITHFULNESS_FRACTIONS = np.linspace(
    0.0,
    1.0,
    11,
)

FAITHFULNESS_N_PER_CLASS = 12


# ---------------------------------------------------------------
# Grad-CAM target layers
# ---------------------------------------------------------------

def get_gradcam_target_layer(
    model,
    model_name,
):

    if model_name == "resnet50":

        # Last convolution inside ResNet layer4.
        return model.layer4[
            -1
        ].conv3

    if (
        model_name
        == "mobilenet_v2"
    ):

        # Final convolutional layer.
        return model.features[
            -1
        ][0]

    if (
        model_name
        == "efficientnet_b0"
    ):

        # Final convolutional layer.
        return model.features[
            -1
        ][0]

    raise ValueError(
        f"Unsupported Grad-CAM model: "
        f"{model_name}"
    )


# ---------------------------------------------------------------
# Grad-CAM computation
# ---------------------------------------------------------------

def compute_gradcam(
    model,
    input_tensor,
    model_name,
):

    model.eval()

    target_layer = (
        get_gradcam_target_layer(
            model,
            model_name,
        )
    )

    activations = []
    gradients = []

    def forward_hook(
        module,
        inputs,
        output,
    ):

        activations.append(
            output
        )

    def backward_hook(
        module,
        grad_input,
        grad_output,
    ):

        if (
            grad_output
            and grad_output[0]
            is not None
        ):

            gradients.append(
                grad_output[0]
            )

    forward_handle = (
        target_layer.register_forward_hook(
            forward_hook
        )
    )

    try:

        try:

            backward_handle = (
                target_layer.register_full_backward_hook(
                    backward_hook
                )
            )

        except Exception:

            backward_handle = (
                target_layer.register_backward_hook(
                    backward_hook
                )
            )

        try:

            model.zero_grad(
                set_to_none=True
            )

            logits = model(
                input_tensor
            ).reshape(-1)

            if logits.numel() != 1:

                raise RuntimeError(
                    f"Expected one logit, "
                    f"got shape {tuple(logits.shape)}"
                )

            target_score = (
                logits[0]
            )

            target_score.backward()

            if not activations:

                raise RuntimeError(
                    f"No activation captured "
                    f"for {model_name}."
                )

            if not gradients:

                raise RuntimeError(
                    f"No gradient captured "
                    f"for {model_name}."
                )

            activation = (
                activations[0]
            )

            gradient = (
                gradients[0]
            )

            weights = gradient.mean(
                dim=(2, 3),
                keepdim=True,
            )

            cam = (
                weights
                * activation
            ).sum(
                dim=1,
                keepdim=False,
            )

            cam = F.relu(
                cam
            )

            cam = F.interpolate(
                cam.unsqueeze(1),
                size=(
                    IMG_SIZE,
                    IMG_SIZE,
                ),
                mode="bilinear",
                align_corners=False,
            ).squeeze()

            cam = (
                cam.detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            cam -= cam.min()

            maximum = cam.max()

            if maximum > 1e-12:

                cam /= maximum

            else:

                cam[:] = 0.0

            cam = np.clip(
                cam,
                0.0,
                1.0,
            )

            return cam

        finally:

            if backward_handle is not None:

                backward_handle.remove()

            model.zero_grad(
                set_to_none=True
            )

    finally:

        forward_handle.remove()


# ---------------------------------------------------------------
# Sparse CAM
# ---------------------------------------------------------------

def make_sparse_cam(
    cam,
    top_fraction=GRADCAM_TOP_FRACTION,
    max_components=GRADCAM_COMPONENTS,
    min_component_area=GRADCAM_MIN_COMPONENT_AREA,
):

    from scipy import ndimage

    cam = np.asarray(
        cam,
        dtype=np.float32,
    )

    positive_values = (
        cam[cam > 0]
    )

    if (
        positive_values.size
        == 0
    ):

        return (
            np.zeros_like(cam),
            1.0,
        )

    threshold = np.quantile(
        positive_values,
        1.0
        - top_fraction,
    )

    binary = (
        (cam >= threshold)
        & (cam > 0)
    )

    labeled, n_components = (
        ndimage.label(
            binary
        )
    )

    if n_components == 0:

        return (
            np.zeros_like(cam),
            threshold,
        )

    min_area = max(
        1,
        int(
            min_component_area
            * cam.size
        ),
    )

    component_scores = []

    for component_id in range(
        1,
        n_components + 1,
    ):

        component_mask = (
            labeled
            == component_id
        )

        area = int(
            component_mask.sum()
        )

        if area < min_area:

            continue

        score = float(
            cam[
                component_mask
            ].mean()
            * area
        )

        component_scores.append(
            (
                score,
                component_id,
            )
        )

    component_scores.sort(
        reverse=True
    )

    keep_ids = [
        component_id
        for _, component_id
        in component_scores[
            :max_components
        ]
    ]

    sparse_binary = np.isin(
        labeled,
        keep_ids,
    )

    sparse_cam = np.where(
        sparse_binary,
        cam,
        0.0,
    )

    return (
        sparse_cam,
        threshold,
    )


# ---------------------------------------------------------------
# Image loader
# ---------------------------------------------------------------

def load_gradcam_image(
    image_path
):

    image = Image.open(
        image_path
    ).convert("RGB")

    display_image = (
        np.asarray(
            image.resize(
                (
                    IMG_SIZE,
                    IMG_SIZE,
                )
            )
        )
        .astype(
            np.float32
        )
        / 255.0
    )

    tensor = (
        val_transform(
            image
        )
        .unsqueeze(0)
        .to(DEVICE)
    )

    return (
        display_image,
        tensor,
    )


# ---------------------------------------------------------------
# Overlay
# ---------------------------------------------------------------

def render_sparse_gradcam_overlay(
    display_image,
    cam,
):

    import matplotlib.cm as cm

    from scipy.ndimage import (
        gaussian_filter,
    )

    sparse_cam, threshold = (
        make_sparse_cam(
            cam
        )
    )

    if sparse_cam.max() > 0:

        smooth = gaussian_filter(
            sparse_cam,
            sigma=2.0,
        )

        support = (
            smooth
            > 0.15
            * smooth.max()
        )

        alpha = np.zeros_like(
            smooth,
            dtype=np.float32,
        )

        if smooth.max() > 0:

            alpha = (
                smooth
                / smooth.max()
            )

        alpha *= (
            support.astype(
                np.float32
            )
        )

        alpha = np.clip(
            alpha
            * GRADCAM_ALPHA_MAX,
            0,
            GRADCAM_ALPHA_MAX,
        )

    else:

        alpha = np.zeros_like(
            sparse_cam,
            dtype=np.float32,
        )

    heatmap = cm.turbo(
        np.clip(
            sparse_cam,
            0,
            1,
        )
    )

    heatmap_rgb = (
        heatmap[
            ...,
            :3
        ]
    )

    overlay = (
        display_image
        * (
            1.0
            - alpha[
                ...,
                None
            ]
        )
        + heatmap_rgb
        * alpha[
            ...,
            None
        ]
    )

    overlay = np.clip(
        overlay,
        0,
        1,
    )

    return (
        overlay,
        alpha,
        threshold,
    )


# ---------------------------------------------------------------
# Target confidence
# ---------------------------------------------------------------

@torch.no_grad()
def model_target_confidence(
    model,
    image_tensor,
    target_class,
):

    logits = model(
        image_tensor
    ).reshape(-1)

    probability_ich = (
        torch.sigmoid(
            logits
        )[0]
    )

    if target_class == 1:

        return float(
            probability_ich.item()
        )

    return float(
        1.0
        - probability_ich.item()
    )


# ---------------------------------------------------------------
# Faithfulness
# ---------------------------------------------------------------

def faithfulness_curves(
    model,
    image_tensor,
    cam,
    target_class,
):

    model.eval()

    x = image_tensor.to(
        DEVICE
    )

    baseline = torch.zeros_like(
        x
    )

    flat_cam = (
        cam.reshape(-1)
    )

    order = np.argsort(
        -flat_cam
    )

    height, width = (
        cam.shape
    )

    deletion_scores = []
    insertion_scores = []

    for fraction in (
        FAITHFULNESS_FRACTIONS
    ):

        k = int(
            round(
                fraction
                * len(order)
            )
        )

        mask_flat = np.zeros(
            len(order),
            dtype=np.float32,
        )

        if k > 0:

            mask_flat[
                order[:k]
            ] = 1.0

        mask = (
            torch.from_numpy(
                mask_flat.reshape(
                    height,
                    width,
                )
            )
            .to(
                DEVICE,
                dtype=x.dtype,
            )
            .unsqueeze(0)
            .unsqueeze(0)
            .expand(
                -1,
                x.shape[1],
                -1,
                -1,
            )
        )

        deleted = (
            x
            * (
                1.0
                - mask
            )
            + baseline
            * mask
        )

        inserted = (
            baseline
            * (
                1.0
                - mask
            )
            + x
            * mask
        )

        deletion_scores.append(
            model_target_confidence(
                model,
                deleted,
                target_class,
            )
        )

        insertion_scores.append(
            model_target_confidence(
                model,
                inserted,
                target_class,
            )
        )

    deletion_scores = np.asarray(
        deletion_scores,
        dtype=float,
    )

    insertion_scores = np.asarray(
        insertion_scores,
        dtype=float,
    )

    if hasattr(
        np,
        "trapezoid"
    ):

        deletion_auc = float(
            np.trapezoid(
                deletion_scores,
                FAITHFULNESS_FRACTIONS,
            )
        )

        insertion_auc = float(
            np.trapezoid(
                insertion_scores,
                FAITHFULNESS_FRACTIONS,
            )
        )

    else:

        deletion_auc = float(
            np.trapz(
                deletion_scores,
                FAITHFULNESS_FRACTIONS,
            )
        )

        insertion_auc = float(
            np.trapz(
                insertion_scores,
                FAITHFULNESS_FRACTIONS,
            )
        )

    return (
        FAITHFULNESS_FRACTIONS.copy(),
        deletion_scores,
        insertion_scores,
        deletion_auc,
        insertion_auc,
    )


# ---------------------------------------------------------------
# Select correctly classified cases
# ---------------------------------------------------------------

def select_gradcam_cases_for_model(
    model_name,
    predictions_df,
):

    g = (
        predictions_df[
            predictions_df.model
            == model_name
        ]
        .copy()
    )

    g[
        "predicted_label"
    ] = (
        g.probability
        >= PRIMARY_THRESHOLD
    ).astype(int)

    g[
        "correct"
    ] = (
        g.predicted_label
        == g.label
    )

    correct = g[
        g.correct
    ].copy()

    positives = (
        correct[
            correct.label == 1
        ]
        .sort_values(
            "probability"
        )
    )

    negatives = (
        correct[
            correct.label == 0
        ]
        .sort_values(
            "probability"
        )
    )

    selected = []

    if len(positives) >= 1:

        selected.append(
            positives.iloc[0]
        )

    if len(positives) >= 2:

        selected.append(
            positives.iloc[-1]
        )

    if len(negatives) >= 1:

        selected.append(
            negatives.iloc[-1]
        )

    if len(negatives) >= 2:

        selected.append(
            negatives.iloc[0]
        )

    if not selected:

        raise RuntimeError(
            f"No correctly classified "
            f"test cases available for "
            f"{model_name}."
        )

    return (
        pd.DataFrame(
            selected
        )
        .drop_duplicates(
            subset=[
                "image_path"
            ]
        )
        .reset_index(
            drop=True
        )
    )


# ---------------------------------------------------------------
# GRAD-CAM OUTPUTS
# ---------------------------------------------------------------

GRADCAM_DIR = (
    OUT_DIR
    / "gradcam"
)

GRADCAM_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

gradcam_case_rows = []

faithfulness_case_rows = []

faithfulness_curve_rows = []


# ---------------------------------------------------------------
# Generate Grad-CAM
# ---------------------------------------------------------------

for model_name in MODEL_NAMES:

    print()
    print("-" * 120)
    print(
        f"GRAD-CAM ANALYSIS: "
        f"{model_name}"
    )
    print("-" * 120)

    checkpoint_path = (
        OUT_DIR
        / f"{model_name}_seed{SEED}_slice_level_final.pth"
    )

    if not checkpoint_path.exists():

        raise FileNotFoundError(
            "Grad-CAM checkpoint missing:\n"
            f"{checkpoint_path}"
        )

    model = create_model(
        model_name
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
    )

    if (
        "model_state_dict"
        not in checkpoint
    ):

        raise RuntimeError(
            f"Checkpoint does not contain "
            f"'model_state_dict': "
            f"{checkpoint_path}"
        )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    model.eval()

    selected_cases = (
        select_gradcam_cases_for_model(
            model_name,
            test_predictions_all,
        )
    )

    model_dir = (
        GRADCAM_DIR
        / model_name
    )

    model_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # ------------------------------------------------------------
    # Qualitative cases
    # ------------------------------------------------------------

    n_cases = len(
        selected_cases
    )

    fig, axes = plt.subplots(
        n_cases,
        3,
        figsize=(
            12,
            max(
                4,
                3.4
                * n_cases
            ),
        ),
        squeeze=False,
    )

    for row_number, (_, row) in enumerate(
        selected_cases.iterrows()
    ):

        display_image, tensor = (
            load_gradcam_image(
                row.image_path
            )
        )

        cam = compute_gradcam(
            model,
            tensor,
            model_name,
        )

        overlay, alpha, threshold = (
            render_sparse_gradcam_overlay(
                display_image,
                cam,
            )
        )

        sparse_cam, _ = (
            make_sparse_cam(
                cam
            )
        )

        true_label_name = (
            "Hemorrhagic"
            if int(
                row.label
            ) == 1
            else "Normal"
        )

        predicted_label = int(
            row.predicted_label
        )

        gradcam_case_rows.append(
            {
                "model": model_name,
                "case_number": row_number + 1,
                "patient_id": row.patient_id,
                "label": int(
                    row.label
                ),
                "predicted_label": (
                    predicted_label
                ),
                "probability": float(
                    row.probability
                ),
                "image_path": row.image_path,
                "cam_threshold": float(
                    threshold
                ),
                "top_saliency_fraction": (
                    GRADCAM_TOP_FRACTION
                ),
            }
        )

        # Original
        ax = axes[
            row_number,
            0
        ]

        ax.imshow(
            display_image
        )

        ax.set_title(
            f"{true_label_name} | "
            f"P(ICH)={float(row.probability):.3f}"
        )

        ax.axis(
            "off"
        )

        # CAM
        ax = axes[
            row_number,
            1
        ]

        im = ax.imshow(
            sparse_cam,
            cmap="turbo",
            vmin=0,
            vmax=1,
        )

        ax.set_title(
            "Sparse Grad-CAM"
        )

        ax.axis(
            "off"
        )

        # Overlay
        ax = axes[
            row_number,
            2
        ]

        ax.imshow(
            overlay
        )

        if alpha.max() > 0:

            try:

                ax.contour(
                    alpha > 0.5,
                    levels=[0.5],
                    linewidths=0.9,
                )

            except Exception:
                pass

        ax.set_title(
            "Grad-CAM overlay"
        )

        ax.axis(
            "off"
        )

    fig.suptitle(
        f"Grad-CAM: "
        f"{display_names[model_name]}",
        fontsize=16,
    )

    plt.tight_layout(
        rect=[
            0,
            0,
            1,
            0.96,
        ]
    )

    save_publication_figure(
        fig,
        f"fig11_gradcam_{model_name}",
    )

    # ------------------------------------------------------------
    # Quantitative faithfulness
    # ------------------------------------------------------------

    prediction_subset = (
        test_predictions_all[
            test_predictions_all.model
            == model_name
        ]
        .copy()
    )

    prediction_subset[
        "predicted_label"
    ] = (
        prediction_subset.probability
        >= PRIMARY_THRESHOLD
    ).astype(int)

    prediction_subset[
        "correct"
    ] = (
        prediction_subset.predicted_label
        == prediction_subset.label
    )

    correct = prediction_subset[
        prediction_subset.correct
    ].copy()

    selected_faithfulness = []

    for target_label in [
        0,
        1,
    ]:

        candidates = correct[
            correct.label
            == target_label
        ].sort_values(
            "probability"
        )

        if candidates.empty:
            continue

        n_select = min(
            FAITHFULNESS_N_PER_CLASS,
            len(candidates),
        )

        if n_select == 1:

            positions = [
                0
            ]

        else:

            positions = np.linspace(
                0,
                len(candidates) - 1,
                n_select,
                dtype=int,
            )

        selected_faithfulness.append(
            candidates.iloc[
                positions
            ]
        )

    if selected_faithfulness:

        faith_cases = pd.concat(
            selected_faithfulness,
            ignore_index=True,
        )

    else:

        faith_cases = (
            pd.DataFrame()
        )

    for _, row in (
        faith_cases.iterrows()
    ):

        display_image, tensor = (
            load_gradcam_image(
                row.image_path
            )
        )

        cam = compute_gradcam(
            model,
            tensor,
            model_name,
        )

        (
            fractions,
            deletion_scores,
            insertion_scores,
            deletion_auc,
            insertion_auc,
        ) = faithfulness_curves(
            model,
            tensor,
            cam,
            target_class=int(
                row.label
            ),
        )

        faithfulness_case_rows.append(
            {
                "model": model_name,
                "patient_id": row.patient_id,
                "label": int(
                    row.label
                ),
                "probability": float(
                    row.probability
                ),
                "image_path": row.image_path,
                "deletion_auc": float(
                    deletion_auc
                ),
                "insertion_auc": float(
                    insertion_auc
                ),
            }
        )

        for fraction, deletion, insertion in zip(
            fractions,
            deletion_scores,
            insertion_scores,
        ):

            faithfulness_curve_rows.append(
                {
                    "model": model_name,
                    "patient_id": row.patient_id,
                    "label": int(
                        row.label
                    ),
                    "fraction": float(
                        fraction
                    ),
                    "deletion_confidence": float(
                        deletion
                    ),
                    "insertion_confidence": float(
                        insertion
                    ),
                }
            )

    del model

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ---------------------------------------------------------------
# Save Grad-CAM CSVs
# ---------------------------------------------------------------

gradcam_cases_df = pd.DataFrame(
    gradcam_case_rows
)

faithfulness_case_df = pd.DataFrame(
    faithfulness_case_rows
)

faithfulness_curve_df = pd.DataFrame(
    faithfulness_curve_rows
)

gradcam_cases_df.to_csv(
    OUT_DIR
    / "gradcam_qualitative_cases.csv",
    index=False,
)

faithfulness_case_df.to_csv(
    OUT_DIR
    / "gradcam_faithfulness_per_case.csv",
    index=False,
)

faithfulness_curve_df.to_csv(
    OUT_DIR
    / "gradcam_faithfulness_curves.csv",
    index=False,
)


# ================================================================
# 24. GRAD-CAM FAITHFULNESS SUMMARY
# ================================================================

if not faithfulness_case_df.empty:

    gradcam_faithfulness_summary = (
        faithfulness_case_df
        .groupby(
            "model"
        )
        .agg(
            n_cases=(
                "image_path",
                "count"
            ),
            mean_deletion_auc=(
                "deletion_auc",
                "mean"
            ),
            sd_deletion_auc=(
                "deletion_auc",
                "std"
            ),
            mean_insertion_auc=(
                "insertion_auc",
                "mean"
            ),
            sd_insertion_auc=(
                "insertion_auc",
                "std"
            ),
        )
        .reset_index()
    )

else:

    gradcam_faithfulness_summary = (
        pd.DataFrame(
            columns=[
                "model",
                "n_cases",
                "mean_deletion_auc",
                "sd_deletion_auc",
                "mean_insertion_auc",
                "sd_insertion_auc",
            ]
        )
    )

gradcam_faithfulness_summary.to_csv(
    OUT_DIR
    / "gradcam_faithfulness_summary.csv",
    index=False,
)


# ---------------------------------------------------------------
# Combined deletion/insertion curves
# ---------------------------------------------------------------

if not faithfulness_curve_df.empty:

    mean_curve = (
        faithfulness_curve_df
        .groupby(
            [
                "model",
                "fraction",
            ]
        )
        .agg(
            deletion_mean=(
                "deletion_confidence",
                "mean"
            ),
            insertion_mean=(
                "insertion_confidence",
                "mean"
            ),
        )
        .reset_index()
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(13, 5.8),
    )

    for model_name in MODEL_NAMES:

        g = mean_curve[
            mean_curve.model
            == model_name
        ]

        if g.empty:
            continue

        summary = (
            gradcam_faithfulness_summary[
                gradcam_faithfulness_summary.model
                == model_name
            ]
        )

        if not summary.empty:

            deletion_auc = float(
                summary[
                    "mean_deletion_auc"
                ].iloc[0]
            )

            insertion_auc = float(
                summary[
                    "mean_insertion_auc"
                ].iloc[0]
            )

        else:

            deletion_auc = np.nan
            insertion_auc = np.nan

        readable = (
            display_names[
                model_name
            ]
        )

        axes[0].plot(
            g.fraction,
            g.deletion_mean,
            marker="o",
            markersize=3,
            linewidth=2,
            label=(
                f"{readable} "
                f"(DAUC={deletion_auc:.3f})"
            ),
        )

        axes[1].plot(
            g.fraction,
            g.insertion_mean,
            marker="s",
            markersize=3,
            linewidth=2,
            label=(
                f"{readable} "
                f"(IAUC={insertion_auc:.3f})"
            ),
        )

    axes[0].set_xlabel(
        "Fraction of salient pixels removed"
    )

    axes[0].set_ylabel(
        "Target-class confidence"
    )

    axes[0].set_title(
        "Grad-CAM Deletion Faithfulness"
    )

    axes[0].set_xlim(
        0,
        1
    )

    axes[0].set_ylim(
        0,
        1.02
    )

    axes[0].grid(
        alpha=0.25
    )

    axes[0].legend()

    axes[1].set_xlabel(
        "Fraction of salient pixels inserted"
    )

    axes[1].set_ylabel(
        "Target-class confidence"
    )

    axes[1].set_title(
        "Grad-CAM Insertion Faithfulness"
    )

    axes[1].set_xlim(
        0,
        1
    )

    axes[1].set_ylim(
        0,
        1.02
    )

    axes[1].grid(
        alpha=0.25
    )

    axes[1].legend()

    fig.suptitle(
        "Quantitative Grad-CAM Faithfulness",
        fontsize=16,
    )

    plt.tight_layout(
        rect=[
            0,
            0,
            1,
            0.94,
        ]
    )

    save_publication_figure(
        fig,
        "fig12_gradcam_faithfulness",
    )


# ================================================================
# 25. PROTOCOL JSON
# ================================================================

protocol = {
    "protocol_name": (
        "ICH slice-level 70-15-15 evaluation"
    ),
    "seed": SEED,
    "dataset_images": int(
        len(df)
    ),
    "dataset_patients": int(
        df.patient_id.nunique()
    ),
    "split": (
        "slice-level stratified "
        "70% train / 15% validation / "
        "15% test"
    ),
    "split_random_state": SEED,
    "patient_overlap_allowed": True,
    "patient_overlap_audited": True,
    "slice_level_cv": (
        "4-fold StratifiedKFold "
        "inside 70% training partition"
    ),
    "models": MODEL_NAMES,
    "image_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "normalization_mean": NORMALIZE_MEAN,
    "normalization_std": NORMALIZE_STD,
    "optimizer": OPTIMIZER_NAME,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "dropout": DROPOUT_P,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": (
        EARLY_STOPPING_PATIENCE
    ),
    "scheduler": (
        "ReduceLROnPlateau on "
        "slice-level validation AUROC"
    ),
    "loss": (
        "patient-balanced weighted "
        "BCEWithLogitsLoss"
    ),
    "primary_endpoints": [
        "slice-level AUROC",
        "slice-level AUPRC",
    ],
    "primary_threshold": PRIMARY_THRESHOLD,
    "high_specificity_target": (
        HIGH_SPEC_TARGET
    ),
    "high_specificity_threshold_source": (
        "70% training OOF predictions only"
    ),
    "test_used_for_model_selection": False,
    "test_used_for_epoch_selection": False,
    "test_used_for_threshold_selection": False,
    "patient_aggregated_test_metrics": (
        "descriptive only because "
        "patient overlap is permitted"
    ),
    "bootstrap": (
        "patient-cluster bootstrap"
    ),
    "bootstrap_replicates": N_BOOTSTRAP,
    "selected_model_by_cv": (
        selected_model
    ),
    "external_validation": False,
    "publication_plot_dpi": PLOT_DPI,
    "publication_plot_formats": [
        "PNG",
        "PDF",
    ],
    "gradcam": True,
    "gradcam_method": "Grad-CAM",
    "gradcam_target": (
        "positive ICH logit"
    ),
    "gradcam_overlay": (
        "sparse high-saliency "
        "connected-region overlay"
    ),
    "gradcam_is_not_lesion_segmentation": True,
    "pixel_level_lesion_masks_available": False,
    "dice_iou_reported": False,
    "faithfulness": [
        "deletion AUC",
        "insertion AUC",
    ],
    "deletion_auc_interpretation": (
        "lower is better"
    ),
    "insertion_auc_interpretation": (
        "higher is better"
    ),
    "include_source_images_in_zip": (
        INCLUDE_SOURCE_IMAGES_IN_ZIP
    ),
}

with open(
    OUT_DIR
    / "protocol.json",
    "w",
) as f:

    json.dump(
        protocol,
        f,
        indent=2,
    )


# ================================================================
# 26. FINAL SUMMARY
# ================================================================

print()
print("=" * 120)
print("FINAL SLICE-LEVEL CV RESULTS")
print("=" * 120)

print(
    cv_model_summary.round(
        4
    ).to_string(
        index=False
    )
)

print()
print("=" * 120)
print("FINAL SLICE-LEVEL TEST RESULTS")
print("=" * 120)

print(
    slice_main_table.round(
        4
    ).to_string(
        index=False
    )
)

print()
print("=" * 120)
print("MODEL RANKING FROM DEVELOPMENT CV")
print("=" * 120)

print(
    model_selection.round(
        4
    ).to_string(
        index=False
    )
)

print()
print(
    f"Selected model by development CV AUROC: "
    f"{selected_model}"
)

print()
print("=" * 120)
print("PATIENT OVERLAP AUDIT")
print("=" * 120)

print(
    patient_overlap_audit.to_string(
        index=False
    )
)

print()
print("=" * 120)
print("GRAD-CAM FAITHFULNESS")
print("=" * 120)

if not (
    gradcam_faithfulness_summary.empty
):

    print(
        gradcam_faithfulness_summary.round(
            4
        ).to_string(
            index=False
        )
    )

else:

    print(
        "No Grad-CAM faithfulness results."
    )

print()
print("=" * 120)
print("EFFICIENCY")
print("=" * 120)

print(
    efficiency_summary.round(
        4
    ).to_string(
        index=False
    )
)


# ================================================================
# 27. CREATE COMPLETE ZIP
# ================================================================
#
# The ZIP contains:
#
#   A. All generated experiment outputs
#      - CSV
#      - JSON
#      - PNG
#      - PDF
#      - PTH model checkpoints
#
#   B. All Grad-CAM outputs
#
#   C. All publication figures
#
#   D. Dataset manifests
#
#   E. Original CT images, if
#      INCLUDE_SOURCE_IMAGES_IN_ZIP = True
#
# Original source images are added directly from /kaggle/input.
# They are NOT copied into another directory first.
#
# The ZIP itself is outside OUT_DIR, so it cannot recursively
# include itself.
# ================================================================

print()
print("=" * 120)
print("CREATING COMPLETE KAGGLE ZIP")
print("=" * 120)

if ZIP_PATH.exists():

    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zipf:

    # ------------------------------------------------------------
    # Generated output files
    # ------------------------------------------------------------

    generated_files = [
        p
        for p in OUT_DIR.rglob("*")
        if p.is_file()
    ]

    for file_path in generated_files:

        arcname = Path(
            "experiment_output"
        ) / file_path.relative_to(
            OUT_DIR
        )

        zipf.write(
            file_path,
            arcname=str(
                arcname
            ),
        )

    # ------------------------------------------------------------
    # Original source images
    # ------------------------------------------------------------

    source_file_count = 0

    if INCLUDE_SOURCE_IMAGES_IN_ZIP:

        source_roots = [
            (
                Path(
                    HEMORRHAGIC_DIR
                ),
                Path(
                    "source_data"
                )
                / "Hemorrhagic"
                / "KANAMA",
            ),
            (
                Path(
                    NORMAL_DIR
                ),
                Path(
                    "source_data"
                )
                / "NORMAL",
            ),
        ]

        for source_root, archive_root in (
            source_roots
        ):

            if not source_root.exists():

                raise FileNotFoundError(
                    "Source dataset directory "
                    f"not found: {source_root}"
                )

            for source_file in source_root.rglob("*"):

                if (
                    source_file.is_file()
                    and source_file.suffix.lower()
                    in VALID_EXTENSIONS
                ):

                    relative = (
                        source_file.relative_to(
                            source_root
                        )
                    )

                    arcname = (
                        archive_root
                        / relative
                    )

                    zipf.write(
                        source_file,
                        arcname=str(
                            arcname
                        ),
                    )

                    source_file_count += 1


# ================================================================
# 28. ZIP AUDIT
# ================================================================

if not ZIP_PATH.exists():

    raise RuntimeError(
        "Final ZIP was not created."
    )

zip_size_mb = (
    ZIP_PATH.stat().st_size
    / (
        1024 ** 2
    )
)

generated_file_count = len(
    generated_files
)

print()
print(
    "Generated experiment files:",
    generated_file_count,
)

print(
    "Original source images added:",
    source_file_count,
)

print(
    f"ZIP size: "
    f"{zip_size_mb:.2f} MB"
)

print()
print(
    "FINAL ZIP:"
)

print(
    ZIP_PATH
)

# ------------------------------------------------------------
# Verify ZIP integrity
# ------------------------------------------------------------

print()
print(
    "Checking ZIP integrity..."
)

with zipfile.ZipFile(
    ZIP_PATH,
    mode="r",
) as zipf:

    bad_file = zipf.testzip()

    if bad_file is not None:

        raise RuntimeError(
            "ZIP integrity check failed "
            f"at: {bad_file}"
        )

    members = zipf.namelist()

print(
    f"ZIP integrity check passed."
)

print(
    f"Files inside ZIP: "
    f"{len(members)}"
)

# ------------------------------------------------------------
# Important file checks
# ------------------------------------------------------------

expected_output_files = [
    "protocol.json",
    "patient_overlap_audit.csv",
    "slice_level_test_95CI.csv",
    "TABLE_slice_level_main_results.csv",
    "TABLE_slice_level_cv_results.csv",
    "gradcam_qualitative_cases.csv",
    "gradcam_faithfulness_per_case.csv",
    "gradcam_faithfulness_curves.csv",
    "gradcam_faithfulness_summary.csv",
]

missing_expected = []

for filename in expected_output_files:

    expected_path = (
        OUT_DIR
        / filename
    )

    if not expected_path.exists():

        missing_expected.append(
            filename
        )

if missing_expected:

    raise RuntimeError(
        "Some expected output files are missing:\n"
        + "\n".join(
            missing_expected
        )
    )

print()
print("=" * 120)
print("PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 120)

print()
print(
    f"Experiment output directory:\n"
    f"{OUT_DIR}"
)

print()
print(
    f"Complete ZIP:\n"
    f"{ZIP_PATH}"
)

print()
print(
    "The ZIP contains the generated results, "
    "publication figures, Grad-CAM outputs, "
    "CSV/JSON files, model checkpoints, "
    "and original CT images."
)

print("=" * 120)

ICH SLICE-LEVEL EVALUATION PIPELINE — CLEAN FINAL VERSION
Device: cuda
GPU: Tesla T4
Models: ['resnet50', 'mobilenet_v2', 'efficientnet_b0']
Seed: 42
Split: 70% train / 15% validation / 15% test
CV: 4-fold SLICE-level CV inside the 70% training partition
Primary endpoint: slice-level AUROC
Normalization: mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
Patient-balanced loss: ON

DATASET AUDIT
------------------------------------------------------------------------------------------------------------------------
Total images: 6795
Hemorrhagic images: 2690
Normal images: 4105
Total patients: 45
Hemorrhagic patients: 18
Normal patients: 27
Images per patient:
count     45.000000
mean     151.000000
std       61.746549
min       23.000000
25%      127.000000
50%      132.000000
75%      196.000000
max      360.000000
dtype: float64

SLICE-LEVEL 70/15/15 SPLIT
------------------------------------------------------------------------------------------------------------------------
Train 

100%|██████████| 97.8M/97.8M [00:00<00:00, 199MB/s]


Epoch 001 | TrainLoss=0.4957 | ValSliceAUC=0.9691 | ValAUPRC=0.9596 | LR=1.00e-04
Epoch 002 | TrainLoss=0.2123 | ValSliceAUC=0.9852 | ValAUPRC=0.9785 | LR=1.00e-04
Epoch 003 | TrainLoss=0.1153 | ValSliceAUC=0.9966 | ValAUPRC=0.9950 | LR=1.00e-04
Epoch 004 | TrainLoss=0.1046 | ValSliceAUC=0.9973 | ValAUPRC=0.9966 | LR=1.00e-04
Epoch 005 | TrainLoss=0.0562 | ValSliceAUC=0.9996 | ValAUPRC=0.9995 | LR=1.00e-04
Epoch 006 | TrainLoss=0.0397 | ValSliceAUC=1.0000 | ValAUPRC=1.0000 | LR=1.00e-04
Epoch 007 | TrainLoss=0.0398 | ValSliceAUC=0.9997 | ValAUPRC=0.9994 | LR=1.00e-04
Epoch 008 | TrainLoss=0.0394 | ValSliceAUC=1.0000 | ValAUPRC=1.0000 | LR=1.00e-04
Epoch 009 | TrainLoss=0.0232 | ValSliceAUC=0.9999 | ValAUPRC=0.9999 | LR=1.00e-04
Epoch 010 | TrainLoss=0.0243 | ValSliceAUC=0.9999 | ValAUPRC=0.9999 | LR=5.00e-05
Epoch 011 | TrainLoss=0.0112 | ValSliceAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 012 | TrainLoss=0.0063 | ValSliceAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 013 | Trai

100%|██████████| 13.6M/13.6M [00:00<00:00, 127MB/s]


Epoch 001 | TrainLoss=0.3948 | ValSliceAUC=0.9825 | ValAUPRC=0.9754 | LR=1.00e-04
Epoch 002 | TrainLoss=0.1732 | ValSliceAUC=0.9960 | ValAUPRC=0.9942 | LR=1.00e-04
Epoch 003 | TrainLoss=0.1106 | ValSliceAUC=0.9987 | ValAUPRC=0.9981 | LR=1.00e-04
Epoch 004 | TrainLoss=0.0899 | ValSliceAUC=0.9991 | ValAUPRC=0.9988 | LR=1.00e-04
Epoch 005 | TrainLoss=0.0633 | ValSliceAUC=0.9993 | ValAUPRC=0.9992 | LR=1.00e-04
Epoch 006 | TrainLoss=0.0505 | ValSliceAUC=0.9999 | ValAUPRC=0.9998 | LR=1.00e-04
Epoch 007 | TrainLoss=0.0281 | ValSliceAUC=0.9998 | ValAUPRC=0.9997 | LR=1.00e-04
Epoch 008 | TrainLoss=0.0355 | ValSliceAUC=0.9962 | ValAUPRC=0.9939 | LR=1.00e-04
Epoch 009 | TrainLoss=0.0380 | ValSliceAUC=0.9998 | ValAUPRC=0.9998 | LR=1.00e-04
Epoch 010 | TrainLoss=0.0277 | ValSliceAUC=0.9998 | ValAUPRC=0.9997 | LR=5.00e-05
Epoch 011 | TrainLoss=0.0212 | ValSliceAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 012 | TrainLoss=0.0230 | ValSliceAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 013 | Trai

100%|██████████| 20.5M/20.5M [00:00<00:00, 164MB/s]


Epoch 001 | TrainLoss=0.5353 | ValSliceAUC=0.9747 | ValAUPRC=0.9692 | LR=1.00e-04
Epoch 002 | TrainLoss=0.2322 | ValSliceAUC=0.9960 | ValAUPRC=0.9941 | LR=1.00e-04
Epoch 003 | TrainLoss=0.1403 | ValSliceAUC=0.9987 | ValAUPRC=0.9981 | LR=1.00e-04
Epoch 004 | TrainLoss=0.1017 | ValSliceAUC=0.9997 | ValAUPRC=0.9996 | LR=1.00e-04
Epoch 005 | TrainLoss=0.0686 | ValSliceAUC=0.9999 | ValAUPRC=0.9998 | LR=1.00e-04
Epoch 006 | TrainLoss=0.0712 | ValSliceAUC=0.9999 | ValAUPRC=0.9998 | LR=1.00e-04
Epoch 007 | TrainLoss=0.0554 | ValSliceAUC=0.9999 | ValAUPRC=0.9999 | LR=1.00e-04
Epoch 008 | TrainLoss=0.0621 | ValSliceAUC=1.0000 | ValAUPRC=1.0000 | LR=1.00e-04
Epoch 009 | TrainLoss=0.0424 | ValSliceAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 010 | TrainLoss=0.0283 | ValSliceAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 011 | TrainLoss=0.0254 | ValSliceAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 012 | TrainLoss=0.0177 | ValSliceAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 013 | Trai

NameError: name 'SLICE_THRESHOLD' is not defined

In [1]:
# ================================================================
# ICH SLICE-LEVEL EVALUATION PIPELINE — FINAL PUBLICATION VERSION
# ================================================================
#
# IMPORTANT:
#   * Slice-level 70/15/15 split is intentionally preserved.
#   * Patient overlap across partitions is allowed and audited.
#   * Patient-aggregated test metrics are descriptive only.
#   * Primary endpoints are slice-level AUROC and AUPRC.
#   * Four models: 2 lightweight + 2 larger/conventional CNNs.
#   * Grad-CAM explains the positive ICH logit; it is not lesion segmentation.
#   * No Dice/IoU is calculated because validated lesion masks are absent.
#
# ----------------------------------------------------------------
# WHAT WAS FIXED IN THIS VERSION (nothing else was changed)
#
#   1) THE CRASH: annotated_bar_chart() is called twice (fig4, fig5) with
#      asymmetric error bars built as np.vstack([yerr_low, yerr_high]),
#      shape (2, n_models). That shape is exactly right for matplotlib's
#      `ax.bar(..., yerr=...)`. But the label-placement loop then did
#      `zip(bars, values, errors)`, which walks the first axis of that
#      2-row array instead of one value per bar — so `err` for the first
#      bar became the *entire* yerr_low array, and
#      `np.isfinite(err)` on a multi-element array inside `if ... else`
#      raised exactly the "truth value of an array with more than one
#      element is ambiguous" error you hit, right after DenseNet-121's
#      CV/test/CI results (and the master CSV) had already been computed
#      and saved successfully. The fix extracts the upper-error row for
#      per-bar label placement when a 2-D error array is passed, while
#      the bar chart itself (ax.bar(..., yerr=errors)) is untouched.
#
#   2) A dormant device-mismatch bug in try_compute_flops(): the dummy
#      profiling tensor is created on DEVICE (GPU), but nothing guarantees
#      the model itself is still on GPU at that point — measure_latency()
#      in this script doesn't restore the model's device after the CPU
#      latency benchmark. This was invisible only because `thop` isn't
#      installed, so it fails before the mismatch could occur. Fixed by
#      moving the model to DEVICE inside try_compute_flops() itself.
#
#   3) torch.load(checkpoint_path, map_location=DEVICE) in the Grad-CAM
#      section reloads checkpoints that mix tensors with plain Python
#      metadata (epoch lists, thresholds, etc.). Recent PyTorch versions
#      default `torch.load` to weights_only=True, which can refuse to
#      load exactly this kind of mixed checkpoint. Fixed by passing
#      weights_only=False explicitly (safe here: these are your own
#      checkpoints written by this same script, not untrusted files).
#
#   4) Purely additive: a safe, no-op-if-offline attempt to `pip install
#      thop` at startup so FLOPs/MACs actually get computed instead of
#      silently staying NaN.
# ================================================================

import os
import gc
import re
import json
import time
import random
import warnings
import zipfile
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------
# Optional: enable real FLOPs/MACs reporting if `thop` is missing.
# This does not change any modeling, data, or evaluation logic — it only
# lets the existing try_compute_flops() succeed instead of returning NaN.
# Safe no-op if there is no internet access in the Kaggle environment.
# ----------------------------------------------------------------
try:
    import thop  # noqa: F401
except ImportError:
    try:
        import subprocess
        import sys as _sys
        subprocess.run(
            [_sys.executable, "-m", "pip", "install", "-q", "thop"],
            check=False,
        )
    except Exception:
        pass


# ================================================================
# 1. CONFIGURATION
# ================================================================

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
DROPOUT_P = 0.20

OPTIMIZER_NAME = "Adam"
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 3
SCHEDULER_MIN_LR = 1e-7
N_FOLDS = 4

# DO NOT change these: the user requested the original slice-level split.
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15

PRIMARY_THRESHOLD = 0.50
HIGH_SPEC_TARGET = 0.90
N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 4242

LATENCY_WARMUP = 50
LATENCY_REPEATS_GPU = 200
LATENCY_REPEATS_CPU = 100

NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]

# Two lightweight + two larger/conventional CNNs.
MODEL_NAMES = [
    "mobilenet_v2",
    "efficientnet_b0",
    "resnet50",
    "densenet121",
]
LIGHTWEIGHT_MODELS = ["mobilenet_v2", "efficientnet_b0"]
LARGER_CNN_MODELS = ["resnet50", "densenet121"]

HEMORRHAGIC_DIR = (
    "/kaggle/input/datasets/abdulkader90/"
    "brain-ct-hemorrhage-dataset/Data/Hemorrhagic/KANAMA"
)
NORMAL_DIR = (
    "/kaggle/input/datasets/abdulkader90/"
    "brain-ct-hemorrhage-dataset/Data/NORMAL"
)

OUT_DIR = Path("/kaggle/working/ich_slice_level_clean")
OUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = Path("/kaggle/working/ICH_slice_level_complete_results.zip")
INCLUDE_SOURCE_IMAGES_IN_ZIP = True

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not available. In Kaggle, enable GPU accelerator/T4 before running this pipeline."
    )

DEVICE = torch.device("cuda")
USE_AMP = True

print("=" * 120)
print("ICH SLICE-LEVEL EVALUATION PIPELINE — FINAL PUBLICATION VERSION")
print("=" * 120)
print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")
print(f"Models: {MODEL_NAMES}")
print(f"Lightweight: {LIGHTWEIGHT_MODELS}")
print(f"Larger/conventional CNNs: {LARGER_CNN_MODELS}")
print(f"Seed: {SEED}")
print("Split: 70% train / 15% validation / 15% test (slice-level, unchanged)")
print(f"CV: {N_FOLDS}-fold slice-level StratifiedKFold inside 70% training partition")
print("Grad-CAM: enabled")
print("Source images in final ZIP:", INCLUDE_SOURCE_IMAGES_IN_ZIP)
print()


# ================================================================
# 2. REPRODUCIBILITY
# ================================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)


# ================================================================
# 3. DATA MANIFEST
# ================================================================

def natural_key(text):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", str(text))]


def build_records(class_dir, label, class_name):
    if not os.path.isdir(class_dir):
        raise FileNotFoundError(f"Dataset directory not found:\n{class_dir}")
    records = []
    folders = sorted(
        [f for f in os.listdir(class_dir) if os.path.isdir(os.path.join(class_dir, f))],
        key=natural_key,
    )
    for folder in folders:
        folder_path = os.path.join(class_dir, folder)
        files = sorted(
            [
                f for f in os.listdir(folder_path)
                if os.path.splitext(f)[1].lower() in VALID_EXTENSIONS
            ],
            key=natural_key,
        )
        patient_id = f"{class_name}_{folder}"
        for fn in files:
            records.append(
                {
                    "patient_id": patient_id,
                    "folder_name": folder,
                    "class_name": class_name,
                    "label": int(label),
                    "image_path": os.path.join(folder_path, fn),
                }
            )
    return records


def build_manifest():
    records = build_records(HEMORRHAGIC_DIR, 1, "HEMORRHAGIC")
    records += build_records(NORMAL_DIR, 0, "NORMAL")
    df = pd.DataFrame(records)
    if df.empty:
        raise RuntimeError("Dataset manifest is empty.")
    labels_per_patient = df.groupby("patient_id")["label"].nunique()
    if not (labels_per_patient == 1).all():
        raise RuntimeError("At least one patient has mixed labels.")
    return df


df = build_manifest()

print("=" * 120)
print("DATASET AUDIT")
print("=" * 120)
print(f"Total slices: {len(df)}")
print(f"Hemorrhagic slices: {int((df.label == 1).sum())}")
print(f"Normal slices: {int((df.label == 0).sum())}")
print(f"Total patients: {df.patient_id.nunique()}")
print(f"Hemorrhagic patients: {df.loc[df.label == 1, 'patient_id'].nunique()}")
print(f"Normal patients: {df.loc[df.label == 0, 'patient_id'].nunique()}")
print("\nSlices per patient:")
print(df.groupby("patient_id").size().describe())
print()


# ================================================================
# 4. SLICE-LEVEL 70/15/15 SPLIT — UNCHANGED
# ================================================================

indices = np.arange(len(df))
labels = df["label"].to_numpy()

train_val_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_FRACTION,
    stratify=labels,
    random_state=SEED,
)

train_val_labels = labels[train_val_idx]
remaining_fraction = VAL_FRACTION / (TRAIN_FRACTION + VAL_FRACTION)

train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=remaining_fraction,
    stratify=train_val_labels,
    random_state=SEED,
)

train_idx = np.asarray(train_idx, dtype=int)
val_idx = np.asarray(val_idx, dtype=int)
test_idx = np.asarray(test_idx, dtype=int)

assert len(train_idx) + len(val_idx) + len(test_idx) == len(df)
assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)


def patient_set(frame):
    return set(frame.patient_id.unique())


train_patients = patient_set(train_df)
val_patients = patient_set(val_df)
test_patients = patient_set(test_df)

patient_overlap_audit = pd.DataFrame(
    [
        {
            "split_a": "train",
            "split_b": "validation",
            "overlap_patients": len(train_patients & val_patients),
        },
        {
            "split_a": "train",
            "split_b": "test",
            "overlap_patients": len(train_patients & test_patients),
        },
        {
            "split_a": "validation",
            "split_b": "test",
            "overlap_patients": len(val_patients & test_patients),
        },
    ]
)

print("=" * 120)
print("SLICE-LEVEL 70/15/15 SPLIT")
print("=" * 120)
for name, frame in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(
        f"{name:12s}: {len(frame):5d} slices | "
        f"positive={int(frame.label.sum()):5d} | "
        f"negative={int((frame.label == 0).sum()):5d} | "
        f"patients={frame.patient_id.nunique():3d}"
    )
print("\nPatient overlap audit:")
print(patient_overlap_audit.to_string(index=False))
print()

split_manifest = df.copy()
split_manifest["split"] = ""
split_manifest.loc[train_idx, "split"] = "train_70"
split_manifest.loc[val_idx, "split"] = "validation_15"
split_manifest.loc[test_idx, "split"] = "test_15"


# ================================================================
# 5. TRANSFORMS
# ================================================================

train_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05)),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
    ]
)

val_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
    ]
)


# ================================================================
# 6. PATIENT-BALANCED WEIGHTS
# ================================================================

def add_patient_balanced_weights(dataframe):
    out = dataframe.copy()
    patient_slice_counts = out.groupby("patient_id").size().to_dict()
    class_patient_counts = out.groupby("label")["patient_id"].nunique().to_dict()
    n_total = len(out)
    weights = []
    for _, row in out.iterrows():
        n_slices = patient_slice_counts[row.patient_id]
        n_class_patients = class_patient_counts[int(row.label)]
        weight = n_total * 0.5 / (n_class_patients * n_slices)
        weights.append(weight)
    out["sample_weight"] = np.asarray(weights, dtype=np.float32)
    sums = out.groupby("label")["sample_weight"].sum()
    if len(sums) == 2:
        assert np.isclose(sums.iloc[0], sums.iloc[1], rtol=1e-5, atol=1e-5)
    assert np.isclose(out.sample_weight.mean(), 1.0, rtol=1e-5, atol=1e-5)
    return out


# ================================================================
# 7. DATASET / DATALOADER
# ================================================================

class BrainCTDataset(Dataset):
    def __init__(self, dataframe, transform=None, use_weights=False):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform
        self.use_weights = use_weights

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row.image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(float(row.label), dtype=torch.float32)
        weight = torch.tensor(
            float(row.sample_weight if self.use_weights else 1.0),
            dtype=torch.float32,
        )
        return image, label, weight, row.patient_id, row.image_path


def create_loader(dataframe, transform, shuffle, use_weights=False):
    return DataLoader(
        BrainCTDataset(dataframe, transform=transform, use_weights=use_weights),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=False,
    )


# ================================================================
# 8. MODELS
# ================================================================

def create_model(model_name):
    try:
        if model_name == "resnet50":
            model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
            model.fc = nn.Sequential(
                nn.Dropout(DROPOUT_P),
                nn.Linear(model.fc.in_features, 1),
            )
        elif model_name == "mobilenet_v2":
            model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
            model.classifier = nn.Sequential(
                nn.Dropout(DROPOUT_P),
                nn.Linear(model.classifier[-1].in_features, 1),
            )
        elif model_name == "efficientnet_b0":
            model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
            model.classifier = nn.Sequential(
                nn.Dropout(DROPOUT_P),
                nn.Linear(model.classifier[-1].in_features, 1),
            )
        elif model_name == "densenet121":
            model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
            model.classifier = nn.Sequential(
                nn.Dropout(DROPOUT_P),
                nn.Linear(model.classifier.in_features, 1),
            )
        else:
            raise ValueError(f"Unknown model: {model_name}")
    except Exception as exc:
        raise RuntimeError(
            f"Could not create {model_name}. Check torchvision pretrained-weight availability."
        ) from exc
    return model.to(DEVICE)


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# ================================================================
# 9. METRICS / AGGREGATION
# ================================================================

def calculate_metrics(labels, probabilities, threshold=0.5):
    labels = np.asarray(labels).astype(int)
    probabilities = np.asarray(probabilities).astype(float)
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    auc = roc_auc_score(labels, probabilities) if len(np.unique(labels)) == 2 else np.nan
    auprc = average_precision_score(labels, probabilities) if len(np.unique(labels)) == 2 else np.nan
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "sensitivity": recall_score(labels, predictions, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(labels, predictions, zero_division=0),
        "auc": auc,
        "auprc": auprc,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def make_slice_prediction_df(labels, probabilities, patient_ids, paths, split, model, seed, fold=np.nan):
    probabilities = np.asarray(probabilities, dtype=float)
    labels = np.asarray(labels).astype(int)
    return pd.DataFrame(
        {
            "model": model,
            "seed": seed,
            "split": split,
            "fold": fold,
            "patient_id": list(patient_ids),
            "label": labels,
            "probability": probabilities,
            "prediction_0_50": (probabilities >= PRIMARY_THRESHOLD).astype(int),
            "image_path": list(paths),
        }
    )


def aggregate_patient_predictions(slice_prediction_df):
    rows = []
    for pid, g in slice_prediction_df.groupby("patient_id"):
        p = g.probability.to_numpy(dtype=float)
        k = max(1, int(np.ceil(len(p) * 0.10)))
        rows.append(
            {
                "patient_id": pid,
                "label": int(g.label.iloc[0]),
                "num_slices_in_split": len(g),
                "mean_probability": float(p.mean()),
                "top10_mean_probability": float(np.mean(np.sort(p)[-k:])),
                "max_probability": float(p.max()),
            }
        )
    return pd.DataFrame(rows)


def select_high_specificity_threshold(labels, probabilities, target_specificity=0.90):
    candidates = np.unique(np.concatenate([np.array([0.0, 1.0]), np.asarray(probabilities, dtype=float)]))
    rows = []
    for threshold in candidates:
        m = calculate_metrics(labels, probabilities, float(threshold))
        rows.append({
            "threshold": float(threshold),
            "sensitivity": m["sensitivity"],
            "specificity": m["specificity"],
            "accuracy": m["accuracy"],
            "precision": m["precision"],
            "f1": m["f1"],
        })
    table = pd.DataFrame(rows)
    eligible = table[table.specificity >= float(target_specificity)]
    if len(eligible):
        chosen = eligible.sort_values(["sensitivity", "specificity", "threshold"], ascending=[False, False, False]).iloc[0]
        fallback = False
    else:
        chosen = table.sort_values(["specificity", "sensitivity", "threshold"], ascending=[False, False, False]).iloc[0]
        fallback = True
    return float(chosen.threshold), fallback, table


def bootstrap_ci_by_patient(labels, probabilities, patient_ids, threshold=0.5, n_bootstrap=5000, seed=4242):
    labels = np.asarray(labels).astype(int)
    probabilities = np.asarray(probabilities).astype(float)
    patient_ids = np.asarray(patient_ids)
    unique_patients = np.unique(patient_ids)
    patient_to_indices = {pid: np.flatnonzero(patient_ids == pid) for pid in unique_patients}
    rng = np.random.default_rng(seed)
    metric_names = ["accuracy", "precision", "sensitivity", "specificity", "f1", "auc", "auprc"]
    samples = {name: [] for name in metric_names}
    for _ in range(n_bootstrap):
        sampled = rng.choice(unique_patients, size=len(unique_patients), replace=True)
        sampled_indices = np.concatenate([patient_to_indices[pid] for pid in sampled])
        y = labels[sampled_indices]
        p = probabilities[sampled_indices]
        if len(np.unique(y)) < 2:
            continue
        m = calculate_metrics(y, p, threshold)
        for key in metric_names:
            if np.isfinite(m[key]):
                samples[key].append(m[key])
    ci = {}
    for key in metric_names:
        arr = np.asarray(samples[key], dtype=float)
        ci[key] = (
            (float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5)))
            if len(arr) >= 100 else (np.nan, np.nan)
        )
    return ci


# ================================================================
# 10. TRAINING
# ================================================================

def amp_scaler():
    return torch.amp.GradScaler("cuda", enabled=True)


def autocast_context():
    return torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP)


def patient_balanced_bce_loss(logits, labels, sample_weights):
    per_sample = F.binary_cross_entropy_with_logits(logits, labels, reduction="none")
    return (per_sample * sample_weights).sum() / sample_weights.sum().clamp_min(1e-8)


def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0.0
    ys, ps = [], []
    for images, labels_batch, sample_weights, _, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels_batch = labels_batch.to(DEVICE, non_blocking=True)
        sample_weights = sample_weights.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast_context():
            logits = model(images).squeeze(1)
            loss = patient_balanced_bce_loss(logits, labels_batch, sample_weights)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += float(loss.item()) * images.size(0)
        ys.extend(labels_batch.detach().cpu().numpy().tolist())
        ps.extend(torch.sigmoid(logits).detach().cpu().numpy().tolist())
    metrics = calculate_metrics(ys, ps, PRIMARY_THRESHOLD)
    return total_loss / len(loader.dataset), metrics


@torch.no_grad()
def evaluate(model, loader, split="unknown", model_name="unknown", fold=np.nan):
    model.eval()
    ys, ps, pids, paths = [], [], [], []
    total_loss = 0.0
    for images, labels_batch, _, patient_ids_batch, image_paths_batch in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels_batch = labels_batch.to(DEVICE, non_blocking=True)
        with autocast_context():
            logits = model(images).squeeze(1)
            loss = F.binary_cross_entropy_with_logits(logits, labels_batch, reduction="mean")
        probs = torch.sigmoid(logits)
        total_loss += float(loss.item()) * images.size(0)
        ys.extend(labels_batch.detach().cpu().numpy().tolist())
        ps.extend(probs.detach().cpu().numpy().tolist())
        pids.extend(list(patient_ids_batch))
        paths.extend(list(image_paths_batch))
    return {
        "loss": total_loss / len(loader.dataset),
        "labels": np.asarray(ys),
        "probabilities": np.asarray(ps),
        "patient_ids": np.asarray(pids),
        "paths": np.asarray(paths),
        "slice_metrics": calculate_metrics(ys, ps, PRIMARY_THRESHOLD),
        "slice_df": make_slice_prediction_df(ys, ps, pids, paths, split, model_name, SEED, fold),
    }


def build_optimizer(model):
    if OPTIMIZER_NAME.lower() == "adam":
        return torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    if OPTIMIZER_NAME.lower() == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    raise ValueError("OPTIMIZER_NAME must be Adam or AdamW.")


# ================================================================
# 11. CROSS-VALIDATION
# ================================================================

def train_one_cv_fold(model_name, fold_number, fold_train_df, fold_val_df):
    print("\n" + "#" * 120)
    print(f"MODEL={model_name} | SLICE-CV FOLD={fold_number}/{N_FOLDS}")
    print("#" * 120)

    weighted_train_df = add_patient_balanced_weights(fold_train_df)
    train_loader = create_loader(weighted_train_df, train_transform, True, True)
    val_loader = create_loader(fold_val_df, val_transform, False, False)
    seed_everything(SEED)
    model = create_model(model_name)
    total_params, trainable_params = count_parameters(model)
    optimizer = build_optimizer(model)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE, min_lr=SCHEDULER_MIN_LR
    )
    scaler = amp_scaler()
    history = []
    best_auc = -np.inf
    best_epoch = 0
    best_state = None
    patience_counter = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.time()
        train_loss, train_metrics = train_one_epoch(model, train_loader, optimizer, scaler)
        val_eval = evaluate(model, val_loader, "cv_validation", model_name, fold_number)
        vm = val_eval["slice_metrics"]
        val_auc = vm["auc"]
        scheduler.step(val_auc if np.isfinite(val_auc) else -np.inf)
        lr = optimizer.param_groups[0]["lr"]
        history.append(
            {
                "model": model_name,
                "seed": SEED,
                "fold": fold_number,
                "epoch": epoch,
                "train_loss": train_loss,
                "train_accuracy": train_metrics["accuracy"],
                "train_f1": train_metrics["f1"],
                "train_auc": train_metrics["auc"],
                "val_loss": val_eval["loss"],
                "val_accuracy": vm["accuracy"],
                "val_precision": vm["precision"],
                "val_sensitivity": vm["sensitivity"],
                "val_specificity": vm["specificity"],
                "val_f1": vm["f1"],
                "val_auc": vm["auc"],
                "val_auprc": vm["auprc"],
                "learning_rate": lr,
                "epoch_seconds": time.time() - epoch_start,
            }
        )
        print(
            f"Epoch {epoch:03d} | TrainLoss={train_loss:.4f} | "
            f"ValAUC={vm['auc']:.4f} | ValAUPRC={vm['auprc']:.4f} | LR={lr:.2e}"
        )
        if np.isfinite(val_auc) and val_auc > best_auc:
            best_auc = val_auc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

    if best_state is None:
        raise RuntimeError(f"No valid best state for {model_name}, fold {fold_number}.")
    model.load_state_dict(best_state)
    final_val = evaluate(model, val_loader, "cv_validation", model_name, fold_number)
    result = {
        "model": model_name,
        "fold": fold_number,
        "best_epoch": best_epoch,
        "best_val_auc": best_auc,
        "history_df": pd.DataFrame(history),
        "val_eval": final_val,
        "val_slice_df": final_val["slice_df"].copy(),
        "total_params": total_params,
        "trainable_params": trainable_params,
    }
    del model, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()
    return result


# ================================================================
# 12. FINAL TRAINING
# ================================================================

def train_final_model(model_name, num_epochs):
    weighted_train_df = add_patient_balanced_weights(train_df)
    train_loader = create_loader(weighted_train_df, train_transform, True, True)
    seed_everything(SEED)
    model = create_model(model_name)
    optimizer = build_optimizer(model)
    scaler = amp_scaler()
    history = []
    start_time = time.time()
    for epoch in range(1, num_epochs + 1):
        epoch_start = time.time()
        loss, metrics = train_one_epoch(model, train_loader, optimizer, scaler)
        history.append(
            {
                "model": model_name,
                "seed": SEED,
                "stage": "final_training_70pct",
                "epoch": epoch,
                "train_loss": loss,
                "train_accuracy": metrics["accuracy"],
                "train_f1": metrics["f1"],
                "train_auc": metrics["auc"],
                "learning_rate": optimizer.param_groups[0]["lr"],
                "epoch_seconds": time.time() - epoch_start,
            }
        )
        print(
            f"FINAL {model_name} | Epoch {epoch:03d}/{num_epochs} | "
            f"Loss={loss:.4f} | TrainAUC={metrics['auc']:.4f}"
        )
    total_seconds = time.time() - start_time
    del optimizer, scaler, train_loader
    gc.collect()
    return model, pd.DataFrame(history), total_seconds


# ================================================================
# 13. EFFICIENCY
# ================================================================

def measure_latency(model, device, repeats, warmup):
    model = model.to(device)
    model.eval()
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    with torch.inference_mode():
        for _ in range(warmup):
            _ = model(dummy)
        torch.cuda.synchronize() if device.type == "cuda" else None
        timings = []
        for _ in range(repeats):
            if device.type == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()
            _ = model(dummy)
            if device.type == "cuda":
                torch.cuda.synchronize()
            timings.append((time.perf_counter() - start) * 1000.0)
    arr = np.asarray(timings, dtype=float)
    return {
        "mean_ms": float(arr.mean()),
        "median_ms": float(np.median(arr)),
        "sd_ms": float(arr.std(ddof=1)),
        "p95_ms": float(np.percentile(arr, 95)),
    }


def try_compute_flops(model):
    try:
        from thop import profile
        # FIX: measure_latency() may leave `model` sitting on whatever device
        # it was last benchmarked on (e.g. CPU, after the CPU-latency test).
        # Force it back onto DEVICE here so the dummy tensor below (which is
        # always created on DEVICE) never mismatches the model's device —
        # this mismatch was previously invisible only because `thop` was not
        # installed and this function never got this far.
        model = model.to(DEVICE)
        model.eval()
        dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
        macs, params = profile(model, inputs=(dummy,), verbose=False)
        return float(2 * macs), float(macs), float(params)
    except Exception as exc:
        print("THOP unavailable or failed:", exc)
        return np.nan, np.nan, np.nan


# ================================================================
# 14. STORAGE CONTAINERS
# ================================================================

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
cv_splits = list(skf.split(train_df.index.to_numpy(), train_df.label.to_numpy()))

all_cv_fold_rows = []
all_cv_predictions = []
all_histories = []
all_final_histories = []
all_validation_predictions = []
all_test_predictions = []
all_efficiency = []
all_ci_rows = []
model_selection_rows = []
all_threshold_rows = []
all_patient_aggregated_rows = []


# ================================================================
# 15. MAIN SLICE-LEVEL EXPERIMENT
# ================================================================

print("\n" + "=" * 120)
print("STARTING SLICE-LEVEL EXPERIMENT")
print("=" * 120)

for model_name in MODEL_NAMES:
    print("\n" + "=" * 120)
    print(f"START MODEL: {model_name}")
    print("=" * 120)

    cv_results = []
    for fold_number, (fold_train_positions, fold_val_positions) in enumerate(cv_splits, start=1):
        fold_train_df = train_df.iloc[fold_train_positions].reset_index(drop=True)
        fold_val_df = train_df.iloc[fold_val_positions].reset_index(drop=True)
        result = train_one_cv_fold(model_name, fold_number, fold_train_df, fold_val_df)
        cv_results.append(result)
        all_histories.append(result["history_df"])
        all_cv_predictions.append(result["val_slice_df"])
        metrics = result["val_eval"]["slice_metrics"]
        all_cv_fold_rows.append(
            {
                "model": model_name,
                "model_group": "lightweight" if model_name in LIGHTWEIGHT_MODELS else "larger_conventional_cnn",
                "seed": SEED,
                "fold": fold_number,
                "train_slices": len(fold_train_df),
                "validation_slices": len(fold_val_df),
                "train_patients": fold_train_df.patient_id.nunique(),
                "validation_patients": fold_val_df.patient_id.nunique(),
                "best_epoch": result["best_epoch"],
                "best_validation_slice_auc": result["best_val_auc"],
                "validation_accuracy": metrics["accuracy"],
                "validation_precision": metrics["precision"],
                "validation_sensitivity": metrics["sensitivity"],
                "validation_specificity": metrics["specificity"],
                "validation_f1": metrics["f1"],
                "validation_auc": metrics["auc"],
                "validation_auprc": metrics["auprc"],
            }
        )

    oof_slice = pd.concat([r["val_slice_df"] for r in cv_results], ignore_index=True)
    if len(oof_slice) != len(train_df):
        raise RuntimeError(f"OOF count mismatch for {model_name}: expected {len(train_df)}, got {len(oof_slice)}")
    if oof_slice.image_path.nunique() != len(train_df) or oof_slice.image_path.duplicated().any():
        raise RuntimeError(f"Duplicate/missing OOF slices detected for {model_name}.")

    oof_metrics = calculate_metrics(oof_slice.label, oof_slice.probability, PRIMARY_THRESHOLD)
    oof_patient = aggregate_patient_predictions(oof_slice)
    oof_patient_metrics = calculate_metrics(oof_patient.label, oof_patient.mean_probability, PRIMARY_THRESHOLD)

    high_spec_threshold, threshold_fallback, threshold_table = select_high_specificity_threshold(
        oof_slice.label.to_numpy(), oof_slice.probability.to_numpy(), HIGH_SPEC_TARGET
    )
    threshold_table["model"] = model_name
    threshold_table["model_group"] = "lightweight" if model_name in LIGHTWEIGHT_MODELS else "larger_conventional_cnn"
    threshold_table["seed"] = SEED
    threshold_table["source"] = "70pct_training_OOF"
    threshold_table["fallback_used"] = threshold_fallback
    all_threshold_rows.append(threshold_table)

    high_spec_oof_metrics = calculate_metrics(oof_slice.label, oof_slice.probability, high_spec_threshold)
    best_epochs = [r["best_epoch"] for r in cv_results]
    final_epochs = max(1, min(int(np.median(best_epochs)), MAX_EPOCHS))

    model_selection_rows.append(
        {
            "model": model_name,
            "model_group": "lightweight" if model_name in LIGHTWEIGHT_MODELS else "larger_conventional_cnn",
            "seed": SEED,
            "mean_cv_slice_auc": float(np.mean([r["best_val_auc"] for r in cv_results])),
            "sd_cv_slice_auc": float(np.std([r["best_val_auc"] for r in cv_results], ddof=1)),
            "mean_cv_slice_auprc": float(np.mean([r["val_eval"]["slice_metrics"]["auprc"] for r in cv_results])),
            "sd_cv_slice_auprc": float(np.std([r["val_eval"]["slice_metrics"]["auprc"] for r in cv_results], ddof=1)),
            "oof_slice_auc": float(oof_metrics["auc"]),
            "oof_slice_auprc": float(oof_metrics["auprc"]),
            "oof_patient_aggregated_auc_descriptive": float(oof_patient_metrics["auc"]),
            "oof_high_spec_sensitivity": float(high_spec_oof_metrics["sensitivity"]),
            "oof_high_spec_specificity": float(high_spec_oof_metrics["specificity"]),
            "high_spec_threshold": high_spec_threshold,
            "threshold_fallback": threshold_fallback,
            "mean_best_epoch": float(np.mean(best_epochs)),
            "median_best_epoch": int(final_epochs),
        }
    )

    print(f"{model_name} OOF AUROC: {oof_metrics['auc']:.4f}")
    print(f"{model_name} OOF AUPRC: {oof_metrics['auprc']:.4f}")
    print(f"OOF high-specificity threshold: {high_spec_threshold:.4f}")

    # Final model trained on the 70% training partition only; epoch count comes from CV.
    final_model, final_history, final_training_seconds = train_final_model(model_name, final_epochs)
    all_final_histories.append(final_history)

    model_path = OUT_DIR / f"{model_name}_seed{SEED}_slice_level_final.pth"
    torch.save(
        {
            "model_name": model_name,
            "seed": SEED,
            "protocol": "slice_level_70_15_15",
            "cv_protocol": "4_fold_slice_level_CV_within_70pct_train",
            "img_size": IMG_SIZE,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "optimizer": OPTIMIZER_NAME,
            "dropout": DROPOUT_P,
            "normalization_mean": NORMALIZE_MEAN,
            "normalization_std": NORMALIZE_STD,
            "primary_threshold": PRIMARY_THRESHOLD,
            "high_spec_target": HIGH_SPEC_TARGET,
            "high_spec_threshold_from_oof": high_spec_threshold,
            "cv_best_epochs": best_epochs,
            "final_training_epochs": final_epochs,
            "model_state_dict": final_model.state_dict(),
        },
        model_path,
    )

    total_params, trainable_params = count_parameters(final_model)
    gpu_latency = measure_latency(final_model, DEVICE, LATENCY_REPEATS_GPU, LATENCY_WARMUP)
    try:
        cpu_latency = measure_latency(final_model, torch.device("cpu"), LATENCY_REPEATS_CPU, 20)
    except Exception as exc:
        print("CPU latency unavailable:", exc)
        cpu_latency = {"mean_ms": np.nan, "median_ms": np.nan, "sd_ms": np.nan, "p95_ms": np.nan}
        final_model = final_model.to(DEVICE)
        gc.collect()

    flops, macs, thop_params = try_compute_flops(final_model)
    final_model = final_model.to(DEVICE)
    model_size_mb = model_path.stat().st_size / (1024 ** 2)
    all_efficiency.append(
        {
            "model": model_name,
            "model_group": "lightweight" if model_name in LIGHTWEIGHT_MODELS else "larger_conventional_cnn",
            "seed": SEED,
            "total_parameters": total_params,
            "trainable_parameters": trainable_params,
            "parameters_million": total_params / 1e6,
            "model_size_mb": model_size_mb,
            "gpu_latency_mean_ms_per_image": gpu_latency["mean_ms"],
            "gpu_latency_median_ms_per_image": gpu_latency["median_ms"],
            "gpu_latency_sd_ms_per_image": gpu_latency["sd_ms"],
            "gpu_latency_p95_ms_per_image": gpu_latency["p95_ms"],
            "cpu_latency_mean_ms_per_image": cpu_latency["mean_ms"],
            "cpu_latency_median_ms_per_image": cpu_latency["median_ms"],
            "cpu_latency_sd_ms_per_image": cpu_latency["sd_ms"],
            "cpu_latency_p95_ms_per_image": cpu_latency["p95_ms"],
            "FLOPs": flops,
            "MACs": macs,
            "thop_parameters": thop_params,
            "final_training_seconds": final_training_seconds,
        }
    )

    val_loader_final = create_loader(val_df, val_transform, False, False)
    val_eval_final = evaluate(final_model, val_loader_final, "validation_15", model_name)
    val_slice_pred = val_eval_final["slice_df"].copy()
    val_slice_pred["prediction_high_spec"] = (val_slice_pred.probability >= high_spec_threshold).astype(int)
    all_validation_predictions.append(val_slice_pred)

    test_loader_final = create_loader(test_df, val_transform, False, False)
    test_eval_final = evaluate(final_model, test_loader_final, "test_15", model_name)
    test_slice_pred = test_eval_final["slice_df"].copy()
    test_slice_pred["prediction_high_spec"] = (test_slice_pred.probability >= high_spec_threshold).astype(int)
    all_test_predictions.append(test_slice_pred)

    test_metrics = calculate_metrics(test_slice_pred.label, test_slice_pred.probability, PRIMARY_THRESHOLD)
    test_high_spec_metrics = calculate_metrics(test_slice_pred.label, test_slice_pred.probability, high_spec_threshold)
    test_patient_pred = aggregate_patient_predictions(test_slice_pred)
    test_patient_pred["model"] = model_name
    test_patient_pred["seed"] = SEED
    test_patient_pred["split"] = "test_15_descriptive_patient_aggregation"
    all_patient_aggregated_rows.append(test_patient_pred)
    test_patient_metrics = calculate_metrics(test_patient_pred.label, test_patient_pred.mean_probability, PRIMARY_THRESHOLD)

    ci_primary = bootstrap_ci_by_patient(
        test_slice_pred.label.to_numpy(),
        test_slice_pred.probability.to_numpy(),
        test_slice_pred.patient_id.to_numpy(),
        threshold=PRIMARY_THRESHOLD,
        n_bootstrap=N_BOOTSTRAP,
        seed=BOOTSTRAP_SEED,
    )
    ci_high = bootstrap_ci_by_patient(
        test_slice_pred.label.to_numpy(),
        test_slice_pred.probability.to_numpy(),
        test_slice_pred.patient_id.to_numpy(),
        threshold=high_spec_threshold,
        n_bootstrap=N_BOOTSTRAP,
        seed=BOOTSTRAP_SEED + 1,
    )

    all_ci_rows.append(
        {
            "model": model_name,
            "model_group": "lightweight" if model_name in LIGHTWEIGHT_MODELS else "larger_conventional_cnn",
            "seed": SEED,
            "test_slices": len(test_slice_pred),
            "test_patients": test_slice_pred.patient_id.nunique(),
            "primary_threshold": PRIMARY_THRESHOLD,
            "accuracy": test_metrics["accuracy"], "accuracy_ci_low": ci_primary["accuracy"][0], "accuracy_ci_high": ci_primary["accuracy"][1],
            "precision": test_metrics["precision"], "precision_ci_low": ci_primary["precision"][0], "precision_ci_high": ci_primary["precision"][1],
            "sensitivity": test_metrics["sensitivity"], "sensitivity_ci_low": ci_primary["sensitivity"][0], "sensitivity_ci_high": ci_primary["sensitivity"][1],
            "specificity": test_metrics["specificity"], "specificity_ci_low": ci_primary["specificity"][0], "specificity_ci_high": ci_primary["specificity"][1],
            "f1": test_metrics["f1"], "f1_ci_low": ci_primary["f1"][0], "f1_ci_high": ci_primary["f1"][1],
            "auc": test_metrics["auc"], "auc_ci_low": ci_primary["auc"][0], "auc_ci_high": ci_primary["auc"][1],
            "auprc": test_metrics["auprc"], "auprc_ci_low": ci_primary["auprc"][0], "auprc_ci_high": ci_primary["auprc"][1],
            "tn": test_metrics["tn"], "fp": test_metrics["fp"], "fn": test_metrics["fn"], "tp": test_metrics["tp"],
            "high_spec_threshold_from_oof": high_spec_threshold,
            "high_spec_sensitivity": test_high_spec_metrics["sensitivity"],
            "high_spec_sensitivity_ci_low": ci_high["sensitivity"][0],
            "high_spec_sensitivity_ci_high": ci_high["sensitivity"][1],
            "high_spec_specificity": test_high_spec_metrics["specificity"],
            "high_spec_specificity_ci_low": ci_high["specificity"][0],
            "high_spec_specificity_ci_high": ci_high["specificity"][1],
            "descriptive_patient_aggregated_auc": test_patient_metrics["auc"],
            "descriptive_patient_aggregated_auprc": test_patient_metrics["auprc"],
            "descriptive_patient_aggregated_accuracy": test_patient_metrics["accuracy"],
            "descriptive_patient_aggregated_sensitivity": test_patient_metrics["sensitivity"],
            "descriptive_patient_aggregated_specificity": test_patient_metrics["specificity"],
            "descriptive_patient_aggregated_f1": test_patient_metrics["f1"],
        }
    )

    print(
        f"{model_name} TEST: AUROC={test_metrics['auc']:.4f} | "
        f"AUPRC={test_metrics['auprc']:.4f} | "
        f"Sensitivity={test_metrics['sensitivity']:.4f} | "
        f"Specificity={test_metrics['specificity']:.4f} | F1={test_metrics['f1']:.4f}"
    )

    del final_model, val_loader_final, test_loader_final
    gc.collect()
    torch.cuda.empty_cache()


# ================================================================
# 16. CONSOLIDATED RESULTS
# ================================================================

cv_fold_results = pd.DataFrame(all_cv_fold_rows)
cv_model_summary = (
    cv_fold_results.groupby("model")
    .agg(
        mean_cv_auc=("validation_auc", "mean"),
        sd_cv_auc=("validation_auc", "std"),
        mean_cv_auprc=("validation_auprc", "mean"),
        sd_cv_auprc=("validation_auprc", "std"),
        mean_cv_f1=("validation_f1", "mean"),
        sd_cv_f1=("validation_f1", "std"),
        mean_best_epoch=("best_epoch", "mean"),
        median_best_epoch=("best_epoch", "median"),
    )
    .reset_index()
)

model_selection = pd.DataFrame(model_selection_rows).sort_values(
    ["mean_cv_slice_auc", "mean_cv_slice_auprc"], ascending=[False, False]
).reset_index(drop=True)
model_selection["cv_rank"] = np.arange(1, len(model_selection) + 1)
selected_model = str(model_selection.iloc[0]["model"])

efficiency_df = pd.DataFrame(all_efficiency)
efficiency_summary = (
    efficiency_df.groupby("model")
    .agg(
        parameters_million=("parameters_million", "mean"),
        model_size_mb=("model_size_mb", "mean"),
        FLOPs=("FLOPs", "mean"),
        MACs=("MACs", "mean"),
        gpu_latency_mean_ms=("gpu_latency_mean_ms_per_image", "mean"),
        gpu_latency_median_ms=("gpu_latency_median_ms_per_image", "mean"),
        cpu_latency_mean_ms=("cpu_latency_mean_ms_per_image", "mean"),
        cpu_latency_median_ms=("cpu_latency_median_ms_per_image", "mean"),
    )
    .reset_index()
)

if "resnet50" in efficiency_summary.model.values:
    resnet_ref = efficiency_summary[efficiency_summary.model == "resnet50"].iloc[0]
    efficiency_summary["parameter_reduction_vs_resnet50"] = 1.0 - efficiency_summary.parameters_million / resnet_ref.parameters_million
    efficiency_summary["model_size_reduction_vs_resnet50"] = 1.0 - efficiency_summary.model_size_mb / resnet_ref.model_size_mb

cv_predictions_all = pd.concat(all_cv_predictions, ignore_index=True)
validation_predictions_all = pd.concat(all_validation_predictions, ignore_index=True)
test_predictions_all = pd.concat(all_test_predictions, ignore_index=True)
ci_df = pd.DataFrame(all_ci_rows)
history_df = pd.concat(all_histories, ignore_index=True)
final_history_df = pd.concat(all_final_histories, ignore_index=True)
threshold_candidates_df = pd.concat(all_threshold_rows, ignore_index=True)
patient_aggregated_df = pd.concat(all_patient_aggregated_rows, ignore_index=True)


# ================================================================
# 17. PUBLICATION TABLES
# ================================================================

slice_main_table = ci_df[
    ["model", "model_group", "test_slices", "test_patients", "auc", "auc_ci_low", "auc_ci_high", "auprc", "auprc_ci_low", "auprc_ci_high"]
].copy().rename(
    columns={"auc": "test_auroc", "auc_ci_low": "test_auroc_ci_low", "auc_ci_high": "test_auroc_ci_high", "auprc": "test_auprc", "auprc_ci_low": "test_auprc_ci_low", "auprc_ci_high": "test_auprc_ci_high"}
)

slice_cv_table = cv_model_summary[
    ["model", "mean_cv_auc", "sd_cv_auc", "mean_cv_auprc", "sd_cv_auprc", "mean_best_epoch", "median_best_epoch"]
].copy().rename(
    columns={"mean_cv_auc": "cv_auroc_mean", "sd_cv_auc": "cv_auroc_sd", "mean_cv_auprc": "cv_auprc_mean", "sd_cv_auprc": "cv_auprc_sd"}
)


# ================================================================
# 18. SINGLE MASTER CSV — ALL STRUCTURED DATA
# ================================================================
# One CSV is the consolidated machine-readable archive. Each row has
# record_type and section-specific fields; heterogeneous sections are
# combined with a common union of columns.
# ================================================================

master_frames = []

def add_master(frame, record_type):
    if frame is None or len(frame) == 0:
        return
    x = frame.copy()
    x.insert(0, "record_type", record_type)
    master_frames.append(x)

add_master(split_manifest, "dataset_manifest")
add_master(patient_overlap_audit, "patient_overlap_audit")
add_master(cv_fold_results, "cv_fold_results")
add_master(cv_model_summary, "cv_model_summary")
add_master(model_selection, "model_selection")
add_master(efficiency_df, "efficiency_per_model")
add_master(efficiency_summary, "efficiency_summary")
add_master(cv_predictions_all, "oof_slice_predictions")
add_master(validation_predictions_all, "validation15_slice_predictions")
add_master(test_predictions_all, "test15_slice_predictions")
add_master(ci_df, "test_patient_cluster_bootstrap_ci")
add_master(history_df, "cv_training_history")
add_master(final_history_df, "final_training_history")
add_master(slice_main_table, "table_slice_level_main_results")
add_master(slice_cv_table, "table_slice_level_cv_results")
add_master(threshold_candidates_df, "oof_threshold_candidates")
add_master(patient_aggregated_df, "descriptive_patient_aggregated_test")

master_csv = pd.concat(master_frames, ignore_index=True, sort=False)
master_csv_path = OUT_DIR / "MASTER_all_experiment_data.csv"
master_csv.to_csv(master_csv_path, index=False)


# ================================================================
# 19. OPTIONAL HTML TABLES / READABILITY FILES
# ================================================================

slice_main_table.to_html(OUT_DIR / "TABLE_slice_level_main_results.html", index=False)
slice_cv_table.to_html(OUT_DIR / "TABLE_slice_level_cv_results.html", index=False)
model_selection.to_html(OUT_DIR / "TABLE_slice_level_model_selection.html", index=False)
efficiency_summary.to_html(OUT_DIR / "TABLE_slice_level_efficiency.html", index=False)


# ================================================================
# 20. PUBLICATION FIGURE HELPERS
# ================================================================

PLOT_DPI = 600
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 12,
        "axes.titlesize": 16,
        "axes.labelsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 10,
        "axes.linewidth": 1.0,
        "figure.dpi": 150,
        "savefig.dpi": PLOT_DPI,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)


def save_publication_figure(fig, filename_stem):
    fig.savefig(OUT_DIR / f"{filename_stem}.png", dpi=PLOT_DPI, bbox_inches="tight", facecolor="white")
    fig.savefig(OUT_DIR / f"{filename_stem}.pdf", bbox_inches="tight", facecolor="white")
    plt.close(fig)


display_names = {
    "mobilenet_v2": "MobileNetV2",
    "efficientnet_b0": "EfficientNet-B0",
    "resnet50": "ResNet-50",
    "densenet121": "DenseNet-121",
}
plot_order = MODEL_NAMES
x = np.arange(len(plot_order))


def annotated_bar_chart(values, errors, ylabel, title, filename_stem, decimals=3, y_max=None):
    values = np.asarray(values, dtype=float)
    errors = np.asarray(errors, dtype=float) if errors is not None else np.zeros_like(values)

    # FIX: `errors` may be either:
    #   - 1-D, shape (n_models,)   -> symmetric error bars (e.g. CV mean/sd)
    #   - 2-D, shape (2, n_models) -> asymmetric [lower, upper] error bars
    #     (e.g. bootstrap CI half-widths passed in via np.vstack([lower, upper]))
    # matplotlib's ax.bar(..., yerr=errors) already understands both shapes
    # correctly. The per-bar text-label placement below needs one scalar
    # offset per bar, though — zipping directly against a 2-row array (as
    # the original code did) walks the wrong axis: the first "bar" got
    # paired with the *entire* lower-error array instead of its own single
    # value, and `np.isfinite()` on that multi-element array is exactly
    # what raised "truth value of an array... is ambiguous". For the 2-D
    # case we use the upper-error row (how far above the bar the error cap
    # extends) to place each label.
    if errors.ndim == 2:
        label_offsets = errors[1]
    else:
        label_offsets = errors

    fig, ax = plt.subplots(figsize=(10.5, 6.8))
    bars = ax.bar(x, values, yerr=errors, capsize=5, width=0.62)
    finite_upper = np.nanmax(np.where(np.isfinite(values + label_offsets), values + label_offsets, np.nan)) if np.isfinite(values).any() else 1.0
    top = float(max(1.05 if finite_upper <= 1.0 else finite_upper * 1.18, 1.0 if "AUROC" in ylabel or "AUPRC" in ylabel else finite_upper * 1.15))
    if y_max is not None:
        top = y_max
    for bar, value, err in zip(bars, values, label_offsets):
        if not np.isfinite(value):
            continue
        label_y = value + (err if np.isfinite(err) else 0.0) + max(0.02 * top, 0.015)
        ax.text(bar.get_x() + bar.get_width() / 2, label_y, f"{value:.{decimals}f}", ha="center", va="bottom", fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([display_names[m] for m in plot_order])
    ax.set_xlabel("Model")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_ylim(0, top)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    save_publication_figure(fig, filename_stem)


# ================================================================
# 21. PUBLICATION FIGURES
# ================================================================

plot_df = cv_model_summary.set_index("model").reindex(plot_order)
annotated_bar_chart(
    plot_df["mean_cv_auc"].to_numpy(dtype=float),
    plot_df["sd_cv_auc"].fillna(0).to_numpy(dtype=float),
    "Mean 4-fold CV AUROC",
    "Slice-Level Cross-Validation AUROC",
    "fig2_slice_level_cv_auroc",
    decimals=3,
    y_max=1.15,
)
annotated_bar_chart(
    plot_df["mean_cv_auprc"].to_numpy(dtype=float),
    plot_df["sd_cv_auprc"].fillna(0).to_numpy(dtype=float),
    "Mean 4-fold CV AUPRC",
    "Slice-Level Cross-Validation AUPRC",
    "fig3_slice_level_cv_auprc",
    decimals=3,
    y_max=1.15,
)

tp = ci_df.set_index("model").reindex(plot_order)
y = tp["auc"].to_numpy(dtype=float)
yerr_low = np.maximum(0.0, y - tp["auc_ci_low"].to_numpy(dtype=float))
yerr_high = np.maximum(0.0, tp["auc_ci_high"].to_numpy(dtype=float) - y)
annotated_bar_chart(
    y,
    np.vstack([yerr_low, yerr_high]),
    "Held-out slice-level AUROC",
    "Slice-Level Test AUROC With Patient-Cluster Bootstrap 95% CI",
    "fig4_slice_level_test_auroc_ci",
    decimals=4,
    y_max=1.15,
)

y = tp["auprc"].to_numpy(dtype=float)
yerr_low = np.maximum(0.0, y - tp["auprc_ci_low"].to_numpy(dtype=float))
yerr_high = np.maximum(0.0, tp["auprc_ci_high"].to_numpy(dtype=float) - y)
annotated_bar_chart(
    y,
    np.vstack([yerr_low, yerr_high]),
    "Held-out slice-level AUPRC",
    "Slice-Level Test AUPRC With Patient-Cluster Bootstrap 95% CI",
    "fig5_slice_level_test_auprc_ci",
    decimals=4,
    y_max=1.15,
)

# Test ROC
fig, ax = plt.subplots(figsize=(8.0, 7.2))
for model_name in MODEL_NAMES:
    g = test_predictions_all[test_predictions_all.model == model_name]
    if len(g) == 0 or g.label.nunique() < 2:
        continue
    fpr, tpr, _ = roc_curve(g.label.to_numpy(), g.probability.to_numpy())
    auc_value = roc_auc_score(g.label.to_numpy(), g.probability.to_numpy())
    ax.plot(fpr, tpr, linewidth=2.4, label=f"{display_names[model_name]} (AUROC={auc_value:.4f})")
ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.4, label="Chance")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Slice-Level Held-Out Test ROC Curves")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.grid(alpha=0.25); ax.legend(loc="lower right")
fig.tight_layout(); save_publication_figure(fig, "fig6_slice_level_test_roc_curves")

# Test PR
fig, ax = plt.subplots(figsize=(8.0, 7.2))
positive_prevalence = test_df.label.mean()
for model_name in MODEL_NAMES:
    g = test_predictions_all[test_predictions_all.model == model_name]
    if len(g) == 0 or g.label.nunique() < 2:
        continue
    precision, recall, _ = precision_recall_curve(g.label.to_numpy(), g.probability.to_numpy())
    auprc_value = average_precision_score(g.label.to_numpy(), g.probability.to_numpy())
    ax.plot(recall, precision, linewidth=2.4, label=f"{display_names[model_name]} (AUPRC={auprc_value:.4f})")
ax.axhline(positive_prevalence, linestyle="--", linewidth=1.4, label=f"No-skill prevalence={positive_prevalence:.3f}")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title("Slice-Level Held-Out Test Precision-Recall Curves")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.grid(alpha=0.25); ax.legend(loc="lower left")
fig.tight_layout(); save_publication_figure(fig, "fig7_slice_level_test_pr_curves")

# Patient overlap heatmap
patient_split = split_manifest.assign(present=1).pivot_table(index="patient_id", columns="split", values="present", aggfunc="max", fill_value=0)
ordered_columns = [c for c in ["train_70", "validation_15", "test_15"] if c in patient_split.columns]
patient_split = patient_split[ordered_columns].sort_index()
fig, ax = plt.subplots(figsize=(7.2, max(7.0, 0.18 * len(patient_split))))
im = ax.imshow(patient_split.to_numpy(), aspect="auto", interpolation="nearest", cmap="Greys", vmin=0, vmax=1)
ax.set_xticks(np.arange(len(ordered_columns))); ax.set_xticklabels(["Train (70%)", "Validation (15%)", "Test (15%)"])
ax.set_yticks(np.arange(len(patient_split))); ax.set_yticklabels(patient_split.index)
ax.set_xlabel("Slice-level partition"); ax.set_ylabel("Patient"); ax.set_title("Patient Presence Across Slice-Level Partitions")
cbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03); cbar.set_ticks([0, 1]); cbar.set_ticklabels(["Absent", "Present"])
ax.grid(which="major", color="white", linewidth=0.7)
fig.tight_layout(); save_publication_figure(fig, "fig8_slice_level_patient_overlap_heatmap")

# Parameter count
param_df = efficiency_summary.set_index("model").reindex(plot_order)
values = param_df.parameters_million.to_numpy(dtype=float)
fig, ax = plt.subplots(figsize=(10.0, 6.5))
bars = ax.bar(x, values, width=0.62)
upper = np.nanmax(values) * 1.18
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, value + np.nanmax(values)*0.025, f"{value:.2f}M", ha="center", va="bottom", fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels([display_names[m] for m in plot_order])
ax.set_xlabel("Model"); ax.set_ylabel("Parameters (millions)"); ax.set_title("Model Parameter Count"); ax.set_ylim(0, upper); ax.grid(axis="y", alpha=0.25)
fig.tight_layout(); save_publication_figure(fig, "fig9_model_parameter_count")

# CPU latency
values = param_df.cpu_latency_mean_ms.to_numpy(dtype=float)
if np.isfinite(values).any():
    fig, ax = plt.subplots(figsize=(10.0, 6.5))
    bars = ax.bar(x, values, width=0.62)
    finite_max = np.nanmax(values)
    for bar, value in zip(bars, values):
        if np.isfinite(value):
            ax.text(bar.get_x() + bar.get_width()/2, value + finite_max*0.025, f"{value:.2f}", ha="center", va="bottom", fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels([display_names[m] for m in plot_order])
    ax.set_xlabel("Model"); ax.set_ylabel("CPU inference latency (ms/image)"); ax.set_title("CPU Inference Latency"); ax.set_ylim(0, max(finite_max*1.18, 1.0)); ax.grid(axis="y", alpha=0.25)
    fig.tight_layout(); save_publication_figure(fig, "fig10_cpu_latency")

# GPU latency
values = param_df.gpu_latency_mean_ms.to_numpy(dtype=float)
fig, ax = plt.subplots(figsize=(10.0, 6.5))
bars = ax.bar(x, values, width=0.62)
finite_max = np.nanmax(values)
for bar, value in zip(bars, values):
    if np.isfinite(value):
        ax.text(bar.get_x() + bar.get_width()/2, value + finite_max*0.025, f"{value:.2f}", ha="center", va="bottom", fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels([display_names[m] for m in plot_order])
ax.set_xlabel("Model"); ax.set_ylabel("GPU inference latency (ms/image)"); ax.set_title("GPU Inference Latency on Kaggle CUDA Device"); ax.set_ylim(0, max(finite_max*1.18, 1.0)); ax.grid(axis="y", alpha=0.25)
fig.tight_layout(); save_publication_figure(fig, "fig10b_gpu_latency")


# ================================================================
# 22. ROBUST GRAD-CAM
# ================================================================
# The previous implementation used module backward hooks. This version
# avoids register_full_backward_hook/register_backward_hook entirely.
# A forward hook captures the activation, and a tensor-level hook captures
# the gradient of that activation during backpropagation.
# ================================================================

GRADCAM_TOP_FRACTION = 0.07       # smaller/sparser positive CT overlay
GRADCAM_COMPONENTS = 2
GRADCAM_MIN_COMPONENT_AREA = 0.0025
GRADCAM_ALPHA_MAX = 0.58
FAITHFULNESS_FRACTIONS = np.linspace(0.0, 1.0, 11)
FAITHFULNESS_N_PER_CLASS = 12
GRADCAM_OUTPUT_DIR = OUT_DIR / "gradcam"
GRADCAM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def get_gradcam_target_layer(model, model_name):
    if model_name == "resnet50":
        return model.layer4[-1].conv3
    if model_name == "mobilenet_v2":
        return model.features[-1][0]
    if model_name == "efficientnet_b0":
        return model.features[-1][0]
    if model_name == "densenet121":
        return model.features.denseblock4.denselayer16.conv2
    raise ValueError(f"Unsupported Grad-CAM model: {model_name}")


def compute_gradcam(model, input_tensor, model_name):
    model.eval()
    x = input_tensor.detach().clone().to(DEVICE, dtype=torch.float32)
    x.requires_grad_(True)
    target_layer = get_gradcam_target_layer(model, model_name)
    activations = []
    gradients = []

    def forward_hook(_module, _inputs, output):
        if not torch.is_tensor(output) or output.ndim != 4:
            raise RuntimeError(f"Grad-CAM target output for {model_name} must be 4-D, got {type(output)} / {getattr(output, 'shape', None)}")
        activations.append(output)
        output.register_hook(lambda grad: gradients.append(grad))

    handle = target_layer.register_forward_hook(forward_hook)
    try:
        model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            logits = model(x).reshape(-1)
            if logits.numel() != 1:
                raise RuntimeError(f"Expected one logit for Grad-CAM, got shape {tuple(logits.shape)}")
            logits[0].backward()
        if not activations:
            raise RuntimeError(f"No Grad-CAM activation captured for {model_name}.")
        if not gradients:
            raise RuntimeError(f"No Grad-CAM activation gradient captured for {model_name}.")
        activation = activations[-1]
        gradient = gradients[-1]
        weights = gradient.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((weights * activation).sum(dim=1))
        cam = F.interpolate(cam.unsqueeze(1), size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False).squeeze(0).squeeze(0)
        cam = cam.detach().cpu().numpy().astype(np.float32)
        cam -= float(cam.min())
        max_value = float(cam.max())
        if max_value > 1e-12:
            cam /= max_value
        else:
            cam.fill(0.0)
        return np.clip(cam, 0.0, 1.0)
    finally:
        handle.remove()
        model.zero_grad(set_to_none=True)


def make_sparse_cam(cam, top_fraction=GRADCAM_TOP_FRACTION, max_components=GRADCAM_COMPONENTS, min_component_area=GRADCAM_MIN_COMPONENT_AREA):
    from scipy import ndimage
    cam = np.asarray(cam, dtype=np.float32)
    positive = cam[cam > 0]
    if positive.size == 0:
        return np.zeros_like(cam), 1.0
    threshold = float(np.quantile(positive, 1.0 - top_fraction))
    binary = (cam >= threshold) & (cam > 0)
    labeled, n_components = ndimage.label(binary)
    if n_components == 0:
        return np.zeros_like(cam), threshold
    min_area = max(1, int(min_component_area * cam.size))
    scores = []
    for cid in range(1, n_components + 1):
        mask = labeled == cid
        area = int(mask.sum())
        if area < min_area:
            continue
        score = float(cam[mask].mean() * area)
        scores.append((score, cid))
    if not scores:
        # deterministic fallback: keep the strongest connected region if all regions are tiny
        scores = sorted(
            [(float(cam[labeled == cid].mean() * (labeled == cid).sum()), cid) for cid in range(1, n_components + 1)],
            reverse=True,
        )[:1]
    else:
        scores.sort(reverse=True)
    keep_ids = [cid for _, cid in scores[:max_components]]
    sparse_binary = np.isin(labeled, keep_ids)
    return np.where(sparse_binary, cam, 0.0).astype(np.float32), threshold


def load_gradcam_image(image_path):
    image = Image.open(image_path).convert("RGB")
    display_image = np.asarray(image.resize((IMG_SIZE, IMG_SIZE))).astype(np.float32) / 255.0
    tensor = val_transform(image).unsqueeze(0).to(DEVICE, dtype=torch.float32)
    return display_image, tensor


def render_sparse_gradcam_overlay(display_image, cam):
    import matplotlib.cm as cm
    from scipy.ndimage import gaussian_filter
    sparse_cam, threshold = make_sparse_cam(cam)
    if sparse_cam.max() > 0:
        smooth = gaussian_filter(sparse_cam, sigma=1.5)
        if smooth.max() > 0:
            alpha = smooth / smooth.max()
            support = smooth > 0.25 * smooth.max()
            alpha *= support.astype(np.float32)
        else:
            alpha = np.zeros_like(smooth)
        alpha = np.clip(alpha * GRADCAM_ALPHA_MAX, 0, GRADCAM_ALPHA_MAX)
    else:
        alpha = np.zeros_like(sparse_cam, dtype=np.float32)
    heatmap_rgb = cm.turbo(np.clip(sparse_cam, 0, 1))[..., :3]
    overlay = display_image * (1 - alpha[..., None]) + heatmap_rgb * alpha[..., None]
    return np.clip(overlay, 0, 1), alpha, threshold


@torch.no_grad()
def model_target_confidence(model, image_tensor, target_class):
    with autocast_context():
        logits = model(image_tensor.to(DEVICE)).reshape(-1)
    p_ich = torch.sigmoid(logits[0])
    return float(p_ich.item() if target_class == 1 else 1.0 - p_ich.item())


def faithfulness_curves(model, image_tensor, cam, target_class):
    model.eval()
    x = image_tensor.to(DEVICE, dtype=torch.float32)
    # Blurred baseline is more stable than raw all-zero normalized pixels.
    baseline = F.avg_pool2d(x, kernel_size=15, stride=1, padding=7)
    flat_cam = np.asarray(cam).reshape(-1)
    order = np.argsort(-flat_cam)
    h, w = cam.shape
    deletion_scores, insertion_scores = [], []
    for fraction in FAITHFULNESS_FRACTIONS:
        k = int(round(fraction * len(order)))
        mask_flat = np.zeros(len(order), dtype=np.float32)
        if k > 0:
            mask_flat[order[:k]] = 1.0
        mask = torch.from_numpy(mask_flat.reshape(h, w)).to(DEVICE, dtype=x.dtype)[None, None].expand(-1, x.shape[1], -1, -1)
        deleted = x * (1.0 - mask) + baseline * mask
        inserted = baseline * (1.0 - mask) + x * mask
        deletion_scores.append(model_target_confidence(model, deleted, target_class))
        insertion_scores.append(model_target_confidence(model, inserted, target_class))
    deletion_scores = np.asarray(deletion_scores, dtype=float)
    insertion_scores = np.asarray(insertion_scores, dtype=float)
    integration = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    return (
        FAITHFULNESS_FRACTIONS.copy(),
        deletion_scores,
        insertion_scores,
        float(integration(deletion_scores, FAITHFULNESS_FRACTIONS)),
        float(integration(insertion_scores, FAITHFULNESS_FRACTIONS)),
    )


def select_gradcam_cases_for_model(model_name, predictions_df):
    g = predictions_df[predictions_df.model == model_name].copy()
    g["predicted_label"] = (g.probability >= PRIMARY_THRESHOLD).astype(int)
    g["correct"] = g.predicted_label == g.label
    correct = g[g.correct].copy()
    positives = correct[correct.label == 1].sort_values(["probability", "image_path"])
    negatives = correct[correct.label == 0].sort_values(["probability", "image_path"])
    selected = []
    # Fixed deterministic cases: low/high-confidence correct examples per class.
    if len(positives) >= 1: selected.append(positives.iloc[0])
    if len(positives) >= 2: selected.append(positives.iloc[-1])
    if len(negatives) >= 1: selected.append(negatives.iloc[-1])
    if len(negatives) >= 2: selected.append(negatives.iloc[0])
    if not selected:
        raise RuntimeError(f"No correctly classified held-out test slices available for Grad-CAM: {model_name}")
    return pd.DataFrame(selected).drop_duplicates(subset=["image_path"]).reset_index(drop=True)


gradcam_case_rows = []
faithfulness_case_rows = []
faithfulness_curve_rows = []

for model_name in MODEL_NAMES:
    print("\n" + "-" * 120)
    print(f"GRAD-CAM ANALYSIS: {model_name}")
    print("-" * 120)
    checkpoint_path = OUT_DIR / f"{model_name}_seed{SEED}_slice_level_final.pth"
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Grad-CAM checkpoint missing: {checkpoint_path}")

    model = create_model(model_name)
    # FIX: this checkpoint mixes tensors with plain Python metadata (epoch
    # lists, thresholds, etc.). Newer PyTorch versions default `torch.load`
    # to weights_only=True, which can refuse to load exactly this kind of
    # mixed checkpoint. weights_only=False is safe here because this file
    # was written by this same script a few lines above, not downloaded
    # from an untrusted source.
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    if "model_state_dict" not in checkpoint:
        raise RuntimeError(f"Checkpoint does not contain model_state_dict: {checkpoint_path}")
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    selected_cases = select_gradcam_cases_for_model(model_name, test_predictions_all)
    model_dir = GRADCAM_OUTPUT_DIR / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    n_cases = len(selected_cases)
    fig, axes = plt.subplots(n_cases, 3, figsize=(12, max(4, 3.2 * n_cases)), squeeze=False)
    for row_number, (_, row) in enumerate(selected_cases.iterrows()):
        display_image, tensor = load_gradcam_image(row.image_path)
        cam = compute_gradcam(model, tensor, model_name)
        overlay, alpha, threshold = render_sparse_gradcam_overlay(display_image, cam)
        sparse_cam, _ = make_sparse_cam(cam)
        true_name = "Hemorrhagic" if int(row.label) == 1 else "Normal"
        gradcam_case_rows.append(
            {
                "model": model_name,
                "model_group": "lightweight" if model_name in LIGHTWEIGHT_MODELS else "larger_conventional_cnn",
                "case_number": row_number + 1,
                "patient_id": row.patient_id,
                "label": int(row.label),
                "predicted_label": int(row.predicted_label),
                "probability": float(row.probability),
                "image_path": row.image_path,
                "target": "positive ICH logit",
                "cam_threshold": float(threshold),
                "top_saliency_fraction": GRADCAM_TOP_FRACTION,
            }
        )
        axes[row_number, 0].imshow(display_image); axes[row_number, 0].set_title(f"{true_name} | P(ICH)={float(row.probability):.3f}"); axes[row_number, 0].axis("off")
        axes[row_number, 1].imshow(sparse_cam, cmap="turbo", vmin=0, vmax=1); axes[row_number, 1].set_title("Sparse Grad-CAM"); axes[row_number, 1].axis("off")
        axes[row_number, 2].imshow(overlay); axes[row_number, 2].set_title("Grad-CAM overlay")
        if alpha.max() > 0:
            try: axes[row_number, 2].contour(alpha > 0.5, levels=[0.5], linewidths=0.8)
            except Exception: pass
        axes[row_number, 2].axis("off")
    fig.suptitle(f"Grad-CAM: {display_names[model_name]}", fontsize=16)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    save_publication_figure(fig, f"fig11_gradcam_{model_name}")

    prediction_subset = test_predictions_all[test_predictions_all.model == model_name].copy()
    prediction_subset["predicted_label"] = (prediction_subset.probability >= PRIMARY_THRESHOLD).astype(int)
    prediction_subset["correct"] = prediction_subset.predicted_label == prediction_subset.label
    correct = prediction_subset[prediction_subset.correct].copy()
    selected_faithfulness = []
    for target_label in [0, 1]:
        candidates = correct[correct.label == target_label].sort_values(["probability", "image_path"])
        if candidates.empty:
            continue
        n_select = min(FAITHFULNESS_N_PER_CLASS, len(candidates))
        pos = [0] if n_select == 1 else np.linspace(0, len(candidates)-1, n_select, dtype=int)
        selected_faithfulness.append(candidates.iloc[pos])
    faith_cases = pd.concat(selected_faithfulness, ignore_index=True) if selected_faithfulness else pd.DataFrame()

    for _, row in faith_cases.iterrows():
        _, tensor = load_gradcam_image(row.image_path)
        cam = compute_gradcam(model, tensor, model_name)
        fractions, deletion_scores, insertion_scores, deletion_auc, insertion_auc = faithfulness_curves(model, tensor, cam, int(row.label))
        faithfulness_case_rows.append(
            {
                "model": model_name,
                "model_group": "lightweight" if model_name in LIGHTWEIGHT_MODELS else "larger_conventional_cnn",
                "patient_id": row.patient_id,
                "label": int(row.label),
                "probability": float(row.probability),
                "image_path": row.image_path,
                "deletion_auc": deletion_auc,
                "insertion_auc": insertion_auc,
            }
        )
        for fraction, deletion, insertion in zip(fractions, deletion_scores, insertion_scores):
            faithfulness_curve_rows.append(
                {
                    "model": model_name,
                    "model_group": "lightweight" if model_name in LIGHTWEIGHT_MODELS else "larger_conventional_cnn",
                    "patient_id": row.patient_id,
                    "label": int(row.label),
                    "fraction": float(fraction),
                    "deletion_confidence": float(deletion),
                    "insertion_confidence": float(insertion),
                }
            )
    del model
    gc.collect(); torch.cuda.empty_cache()


gradcam_cases_df = pd.DataFrame(gradcam_case_rows)
faithfulness_case_df = pd.DataFrame(faithfulness_case_rows)
faithfulness_curve_df = pd.DataFrame(faithfulness_curve_rows)


# ================================================================
# 23. GRAD-CAM FAITHFULNESS FIGURE / MASTER CSV APPEND
# ================================================================

if not faithfulness_case_df.empty:
    gradcam_faithfulness_summary = (
        faithfulness_case_df.groupby("model")
        .agg(
            n_cases=("image_path", "count"),
            mean_deletion_auc=("deletion_auc", "mean"),
            sd_deletion_auc=("deletion_auc", "std"),
            mean_insertion_auc=("insertion_auc", "mean"),
            sd_insertion_auc=("insertion_auc", "std"),
        )
        .reset_index()
    )
else:
    gradcam_faithfulness_summary = pd.DataFrame(columns=["model", "n_cases", "mean_deletion_auc", "sd_deletion_auc", "mean_insertion_auc", "sd_insertion_auc"])

if not faithfulness_curve_df.empty:
    mean_curve = faithfulness_curve_df.groupby(["model", "fraction"]).agg(
        deletion_mean=("deletion_confidence", "mean"), insertion_mean=("insertion_confidence", "mean")
    ).reset_index()
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.8))
    for model_name in MODEL_NAMES:
        g = mean_curve[mean_curve.model == model_name]
        if g.empty: continue
        s = gradcam_faithfulness_summary[gradcam_faithfulness_summary.model == model_name]
        da = float(s.mean_deletion_auc.iloc[0]) if not s.empty else np.nan
        ia = float(s.mean_insertion_auc.iloc[0]) if not s.empty else np.nan
        readable = display_names[model_name]
        axes[0].plot(g.fraction, g.deletion_mean, marker="o", markersize=3, linewidth=2, label=f"{readable} (DAUC={da:.3f})")
        axes[1].plot(g.fraction, g.insertion_mean, marker="s", markersize=3, linewidth=2, label=f"{readable} (IAUC={ia:.3f})")
    axes[0].set_xlabel("Fraction of salient pixels removed"); axes[0].set_ylabel("Target-class confidence"); axes[0].set_title("Grad-CAM Deletion Faithfulness"); axes[0].set_xlim(0,1); axes[0].set_ylim(0,1.02); axes[0].grid(alpha=0.25); axes[0].legend()
    axes[1].set_xlabel("Fraction of salient pixels inserted"); axes[1].set_ylabel("Target-class confidence"); axes[1].set_title("Grad-CAM Insertion Faithfulness"); axes[1].set_xlim(0,1); axes[1].set_ylim(0,1.02); axes[1].grid(alpha=0.25); axes[1].legend()
    fig.suptitle("Quantitative Grad-CAM Faithfulness", fontsize=16); fig.tight_layout(rect=[0,0,1,0.94])
    save_publication_figure(fig, "fig12_gradcam_faithfulness")

# Rebuild master CSV with Grad-CAM records included.
master_frames = master_frames[:]
def add_master_now(frame, record_type):
    if frame is None or len(frame) == 0: return
    x = frame.copy(); x.insert(0, "record_type", record_type); master_frames.append(x)
add_master_now(gradcam_cases_df, "gradcam_qualitative_cases")
add_master_now(faithfulness_case_df, "gradcam_faithfulness_per_case")
add_master_now(faithfulness_curve_df, "gradcam_faithfulness_curves")
add_master_now(gradcam_faithfulness_summary, "gradcam_faithfulness_summary")
master_csv = pd.concat(master_frames, ignore_index=True, sort=False)
master_csv.to_csv(master_csv_path, index=False)


# ================================================================
# 24. PROTOCOL JSON
# ================================================================

protocol = {
    "protocol_name": "ICH slice-level 70-15-15 evaluation",
    "seed": SEED,
    "dataset_images": int(len(df)),
    "dataset_patients": int(df.patient_id.nunique()),
    "split": "slice-level stratified 70% train / 15% validation / 15% test",
    "split_random_state": SEED,
    "patient_overlap_allowed": True,
    "patient_overlap_audited": True,
    "slice_level_cv": "4-fold StratifiedKFold inside 70% training partition",
    "models": MODEL_NAMES,
    "model_groups": {"lightweight": LIGHTWEIGHT_MODELS, "larger_conventional_cnn": LARGER_CNN_MODELS},
    "image_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "normalization_mean": NORMALIZE_MEAN,
    "normalization_std": NORMALIZE_STD,
    "optimizer": OPTIMIZER_NAME,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "dropout": DROPOUT_P,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "scheduler": "ReduceLROnPlateau on slice-level validation AUROC",
    "loss": "patient-balanced weighted BCEWithLogitsLoss",
    "primary_endpoints": ["slice-level AUROC", "slice-level AUPRC"],
    "primary_threshold": PRIMARY_THRESHOLD,
    "high_specificity_target": HIGH_SPEC_TARGET,
    "high_specificity_threshold_source": "70% training OOF predictions only",
    "test_used_for_model_selection": False,
    "test_used_for_epoch_selection": False,
    "test_used_for_threshold_selection": False,
    "patient_aggregated_test_metrics": "descriptive only because patient overlap is permitted",
    "bootstrap": "patient-cluster bootstrap",
    "bootstrap_replicates": N_BOOTSTRAP,
    "selected_model_by_cv": selected_model,
    "external_validation": False,
    "publication_plot_dpi": PLOT_DPI,
    "publication_plot_formats": ["PNG", "PDF"],
    "gradcam": True,
    "gradcam_method": "Grad-CAM with activation-tensor gradient hook",
    "gradcam_target": "positive ICH logit",
    "gradcam_target_layers": {
        "mobilenet_v2": "features[-1][0]",
        "efficientnet_b0": "features[-1][0]",
        "resnet50": "layer4[-1].conv3",
        "densenet121": "features.denseblock4.denselayer16.conv2",
    },
    "gradcam_overlay": "sparse high-saliency connected-region overlay",
    "gradcam_top_saliency_fraction": GRADCAM_TOP_FRACTION,
    "gradcam_is_not_lesion_segmentation": True,
    "pixel_level_lesion_masks_available": False,
    "dice_iou_reported": False,
    "faithfulness": ["deletion AUC", "insertion AUC"],
    "deletion_auc_interpretation": "lower is better",
    "insertion_auc_interpretation": "higher is better",
    "gpu_required": True,
    "master_csv": "MASTER_all_experiment_data.csv",
    "include_source_images_in_zip": INCLUDE_SOURCE_IMAGES_IN_ZIP,
}
with open(OUT_DIR / "protocol.json", "w", encoding="utf-8") as f:
    json.dump(protocol, f, indent=2)


# ================================================================
# 25. SAVE LIGHTWEIGHT AUDIT JSON / TEXT
# ================================================================

with open(OUT_DIR / "run_summary.txt", "w", encoding="utf-8") as f:
    f.write(f"Device: {DEVICE}\nGPU: {torch.cuda.get_device_name(0)}\nCUDA: {torch.version.cuda}\nPyTorch: {torch.__version__}\n")
    f.write(f"Selected model by CV: {selected_model}\n")
    f.write(f"Models: {', '.join(MODEL_NAMES)}\n")
    f.write("Slice-level split preserved: 70/15/15\n")
    f.write("Patient overlap allowed and audited\n")
    f.write("Master CSV: MASTER_all_experiment_data.csv\n")


# ================================================================
# 26. FINAL CONSOLE SUMMARY
# ================================================================

print("\n" + "=" * 120)
print("FINAL SLICE-LEVEL CV RESULTS")
print("=" * 120)
print(cv_model_summary.round(4).to_string(index=False))

print("\n" + "=" * 120)
print("FINAL SLICE-LEVEL TEST RESULTS")
print("=" * 120)
print(slice_main_table.round(4).to_string(index=False))

print("\n" + "=" * 120)
print("MODEL RANKING FROM DEVELOPMENT CV")
print("=" * 120)
print(model_selection.round(4).to_string(index=False))
print(f"\nSelected model by development CV AUROC: {selected_model}")

print("\n" + "=" * 120)
print("PATIENT OVERLAP AUDIT")
print("=" * 120)
print(patient_overlap_audit.to_string(index=False))

print("\n" + "=" * 120)
print("EFFICIENCY")
print("=" * 120)
print(efficiency_summary.round(4).to_string(index=False))

print("\n" + "=" * 120)
print("GRAD-CAM FAITHFULNESS")
print("=" * 120)
print(gradcam_faithfulness_summary.round(4).to_string(index=False) if not gradcam_faithfulness_summary.empty else "No Grad-CAM faithfulness results.")


# ================================================================
# 27. COMPLETE ZIP EXPORT
# ================================================================

print("\n" + "=" * 120)
print("CREATING COMPLETE KAGGLE ZIP")
print("=" * 120)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

generated_files = [p for p in OUT_DIR.rglob("*") if p.is_file()]
source_file_count = 0

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zipf:
    # All generated results, including PNG/PDF/CSV/JSON/PTH/HTML/text.
    for file_path in generated_files:
        arcname = Path("experiment_output") / file_path.relative_to(OUT_DIR)
        zipf.write(file_path, arcname=str(arcname))

    # Original CT images directly from Kaggle input.
    if INCLUDE_SOURCE_IMAGES_IN_ZIP:
        source_roots = [
            (Path(HEMORRHAGIC_DIR), Path("source_data") / "Hemorrhagic" / "KANAMA"),
            (Path(NORMAL_DIR), Path("source_data") / "NORMAL"),
        ]
        for source_root, archive_root in source_roots:
            if not source_root.exists():
                raise FileNotFoundError(f"Source dataset directory not found: {source_root}")
            for source_file in source_root.rglob("*"):
                if source_file.is_file() and source_file.suffix.lower() in VALID_EXTENSIONS:
                    zipf.write(source_file, arcname=str(archive_root / source_file.relative_to(source_root)))
                    source_file_count += 1

if not ZIP_PATH.exists():
    raise RuntimeError("Final ZIP was not created.")

with zipfile.ZipFile(ZIP_PATH, "r") as zipf:
    bad_file = zipf.testzip()
    if bad_file is not None:
        raise RuntimeError(f"ZIP integrity check failed at: {bad_file}")
    zip_members = zipf.namelist()

expected_outputs = [
    "MASTER_all_experiment_data.csv",
    "protocol.json",
    "run_summary.txt",
    "gradcam",
    "fig2_slice_level_cv_auroc.png",
    "fig4_slice_level_test_auroc_ci.png",
]
missing = []
for item in expected_outputs:
    p = OUT_DIR / item
    if item == "gradcam":
        if not p.exists() or not any(p.rglob("*.png")):
            missing.append(item)
    elif not p.exists():
        missing.append(item)
if missing:
    raise RuntimeError("Expected outputs are missing: " + ", ".join(missing))

zip_size_mb = ZIP_PATH.stat().st_size / (1024 ** 2)
print(f"Generated experiment files: {len(generated_files)}")
print(f"Original source images added: {source_file_count}")
print(f"ZIP size: {zip_size_mb:.2f} MB")
print(f"ZIP integrity check passed. Files inside ZIP: {len(zip_members)}")
print(f"Master CSV: {master_csv_path}")
print(f"Output directory: {OUT_DIR}")
print(f"Complete ZIP: {ZIP_PATH}")
print("\nPIPELINE COMPLETED SUCCESSFULLY")
print("=" * 120)

ICH SLICE-LEVEL EVALUATION PIPELINE — FINAL PUBLICATION VERSION
Device: cuda
GPU: Tesla T4
CUDA: 12.8
PyTorch: 2.10.0+cu128
Models: ['mobilenet_v2', 'efficientnet_b0', 'resnet50', 'densenet121']
Lightweight: ['mobilenet_v2', 'efficientnet_b0']
Larger/conventional CNNs: ['resnet50', 'densenet121']
Seed: 42
Split: 70% train / 15% validation / 15% test (slice-level, unchanged)
CV: 4-fold slice-level StratifiedKFold inside 70% training partition
Grad-CAM: enabled
Source images in final ZIP: True

DATASET AUDIT
Total slices: 6795
Hemorrhagic slices: 2690
Normal slices: 4105
Total patients: 45
Hemorrhagic patients: 18
Normal patients: 27

Slices per patient:
count     45.000000
mean     151.000000
std       61.746549
min       23.000000
25%      127.000000
50%      132.000000
75%      196.000000
max      360.000000
dtype: float64

SLICE-LEVEL 70/15/15 SPLIT
Train       :  4755 slices | positive= 1882 | negative= 2873 | patients= 45
Validation  :  1020 slices | positive=  404 | negative=  616

100%|██████████| 13.6M/13.6M [00:00<00:00, 169MB/s]


Epoch 001 | TrainLoss=0.3948 | ValAUC=0.9825 | ValAUPRC=0.9754 | LR=1.00e-04
Epoch 002 | TrainLoss=0.1732 | ValAUC=0.9960 | ValAUPRC=0.9942 | LR=1.00e-04
Epoch 003 | TrainLoss=0.1106 | ValAUC=0.9987 | ValAUPRC=0.9981 | LR=1.00e-04
Epoch 004 | TrainLoss=0.0899 | ValAUC=0.9991 | ValAUPRC=0.9988 | LR=1.00e-04
Epoch 005 | TrainLoss=0.0633 | ValAUC=0.9993 | ValAUPRC=0.9992 | LR=1.00e-04
Epoch 006 | TrainLoss=0.0505 | ValAUC=0.9999 | ValAUPRC=0.9998 | LR=1.00e-04
Epoch 007 | TrainLoss=0.0281 | ValAUC=0.9998 | ValAUPRC=0.9997 | LR=1.00e-04
Epoch 008 | TrainLoss=0.0355 | ValAUC=0.9962 | ValAUPRC=0.9939 | LR=1.00e-04
Epoch 009 | TrainLoss=0.0380 | ValAUC=0.9998 | ValAUPRC=0.9998 | LR=1.00e-04
Epoch 010 | TrainLoss=0.0277 | ValAUC=0.9998 | ValAUPRC=0.9997 | LR=5.00e-05
Epoch 011 | TrainLoss=0.0212 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 012 | TrainLoss=0.0230 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 013 | TrainLoss=0.0141 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05

100%|██████████| 20.5M/20.5M [00:00<00:00, 166MB/s]


Epoch 001 | TrainLoss=0.5353 | ValAUC=0.9747 | ValAUPRC=0.9692 | LR=1.00e-04
Epoch 002 | TrainLoss=0.2322 | ValAUC=0.9960 | ValAUPRC=0.9941 | LR=1.00e-04
Epoch 003 | TrainLoss=0.1403 | ValAUC=0.9987 | ValAUPRC=0.9981 | LR=1.00e-04
Epoch 004 | TrainLoss=0.1017 | ValAUC=0.9997 | ValAUPRC=0.9996 | LR=1.00e-04
Epoch 005 | TrainLoss=0.0686 | ValAUC=0.9999 | ValAUPRC=0.9998 | LR=1.00e-04
Epoch 006 | TrainLoss=0.0712 | ValAUC=0.9999 | ValAUPRC=0.9998 | LR=1.00e-04
Epoch 007 | TrainLoss=0.0554 | ValAUC=0.9999 | ValAUPRC=0.9999 | LR=1.00e-04
Epoch 008 | TrainLoss=0.0621 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=1.00e-04
Epoch 009 | TrainLoss=0.0424 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 010 | TrainLoss=0.0283 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 011 | TrainLoss=0.0254 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 012 | TrainLoss=0.0177 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 013 | TrainLoss=0.0166 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05

100%|██████████| 97.8M/97.8M [00:00<00:00, 209MB/s]


Epoch 001 | TrainLoss=0.4957 | ValAUC=0.9691 | ValAUPRC=0.9596 | LR=1.00e-04
Epoch 002 | TrainLoss=0.2123 | ValAUC=0.9852 | ValAUPRC=0.9785 | LR=1.00e-04
Epoch 003 | TrainLoss=0.1153 | ValAUC=0.9966 | ValAUPRC=0.9950 | LR=1.00e-04
Epoch 004 | TrainLoss=0.1046 | ValAUC=0.9973 | ValAUPRC=0.9966 | LR=1.00e-04
Epoch 005 | TrainLoss=0.0562 | ValAUC=0.9996 | ValAUPRC=0.9995 | LR=1.00e-04
Epoch 006 | TrainLoss=0.0397 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=1.00e-04
Epoch 007 | TrainLoss=0.0398 | ValAUC=0.9997 | ValAUPRC=0.9994 | LR=1.00e-04
Epoch 008 | TrainLoss=0.0394 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=1.00e-04
Epoch 009 | TrainLoss=0.0232 | ValAUC=0.9999 | ValAUPRC=0.9999 | LR=1.00e-04
Epoch 010 | TrainLoss=0.0243 | ValAUC=0.9999 | ValAUPRC=0.9999 | LR=5.00e-05
Epoch 011 | TrainLoss=0.0112 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 012 | TrainLoss=0.0063 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 013 | TrainLoss=0.0175 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05

100%|██████████| 30.8M/30.8M [00:00<00:00, 168MB/s]


Epoch 001 | TrainLoss=0.3779 | ValAUC=0.9883 | ValAUPRC=0.9844 | LR=1.00e-04
Epoch 002 | TrainLoss=0.1500 | ValAUC=0.9991 | ValAUPRC=0.9986 | LR=1.00e-04
Epoch 003 | TrainLoss=0.0838 | ValAUC=0.9999 | ValAUPRC=0.9998 | LR=1.00e-04
Epoch 004 | TrainLoss=0.0537 | ValAUC=0.9990 | ValAUPRC=0.9985 | LR=1.00e-04
Epoch 005 | TrainLoss=0.0352 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=1.00e-04
Epoch 006 | TrainLoss=0.0428 | ValAUC=0.9999 | ValAUPRC=0.9998 | LR=1.00e-04
Epoch 007 | TrainLoss=0.0472 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=1.00e-04
Epoch 008 | TrainLoss=0.0271 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=1.00e-04
Epoch 009 | TrainLoss=0.0254 | ValAUC=0.9998 | ValAUPRC=0.9996 | LR=5.00e-05
Epoch 010 | TrainLoss=0.0100 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 011 | TrainLoss=0.0091 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 012 | TrainLoss=0.0078 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=5.00e-05
Epoch 013 | TrainLoss=0.0048 | ValAUC=1.0000 | ValAUPRC=1.0000 | LR=2.50e-05

RuntimeError: Expected outputs are missing: gradcam

In [2]:
from pathlib import Path
p = Path("/kaggle/working/ICH_slice_level_complete_results.zip")
print(p.exists(), p.stat().st_size / (1024**2), "MB")

True 267.7912406921387 MB


In [4]:
import zipfile

zip_path = "/kaggle/working/ICH_slice_level_complete_results.zip"

with zipfile.ZipFile(zip_path) as z:
    names = z.namelist()
    matches = [n for n in names if "fig11" in n or "fig12" in n or "gradcam" in n.lower()]

if matches:
    print(f"Found {len(matches)} matching entries:\n")
    for name in sorted(matches):
        print(name)
else:
    print("No matches found. Total entries in zip:", len(names))

Found 10 matching entries:

experiment_output/fig11_gradcam_densenet121.pdf
experiment_output/fig11_gradcam_densenet121.png
experiment_output/fig11_gradcam_efficientnet_b0.pdf
experiment_output/fig11_gradcam_efficientnet_b0.png
experiment_output/fig11_gradcam_mobilenet_v2.pdf
experiment_output/fig11_gradcam_mobilenet_v2.png
experiment_output/fig11_gradcam_resnet50.pdf
experiment_output/fig11_gradcam_resnet50.png
experiment_output/fig12_gradcam_faithfulness.pdf
experiment_output/fig12_gradcam_faithfulness.png


In [5]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models

import zipfile

# ================================================================
# CONFIG — matches the values used in the main pipeline
# ================================================================
OUT_DIR = Path("/kaggle/working/ich_slice_level_clean")
MASTER_CSV_PATH = OUT_DIR / "MASTER_all_experiment_data.csv"
GRADCAM_ROOT = OUT_DIR / "gradcam"          # same folders the main script already created
NEW_ZIP_PATH = Path("/kaggle/working/ICH_gradcam_per_case_by_architecture.zip")

IMG_SIZE = 224
DROPOUT_P = 0.20
PRIMARY_THRESHOLD = 0.50
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]

MODEL_NAMES = ["mobilenet_v2", "efficientnet_b0", "resnet50", "densenet121"]
display_names = {
    "mobilenet_v2": "MobileNetV2",
    "efficientnet_b0": "EfficientNet-B0",
    "resnet50": "ResNet-50",
    "densenet121": "DenseNet-121",
}

GRADCAM_TOP_FRACTION = 0.07
GRADCAM_COMPONENTS = 2
GRADCAM_MIN_COMPONENT_AREA = 0.0025
GRADCAM_ALPHA_MAX = 0.58
FIG_DPI = 600

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {DEVICE}")

val_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
    ]
)

# ================================================================
# LOAD TEST PREDICTIONS FROM THE MASTER CSV (no need to retrain)
# ================================================================
master_df = pd.read_csv(MASTER_CSV_PATH, low_memory=False)
test_predictions_all = master_df[master_df.record_type == "test15_slice_predictions"].copy()
test_predictions_all["label"] = test_predictions_all["label"].astype(int)
test_predictions_all["probability"] = test_predictions_all["probability"].astype(float)
print(f"Loaded {len(test_predictions_all)} held-out test slice predictions from master CSV.")


# ================================================================
# MODEL SKELETON — architecture only, weights come from the checkpoint,
# so there is no need to re-download ImageNet pretrained weights here.
# ================================================================
def create_model_skeleton(model_name):
    if model_name == "resnet50":
        model = models.resnet50(weights=None)
        model.fc = nn.Sequential(nn.Dropout(DROPOUT_P), nn.Linear(model.fc.in_features, 1))
    elif model_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=None)
        model.classifier = nn.Sequential(nn.Dropout(DROPOUT_P), nn.Linear(model.classifier[-1].in_features, 1))
    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        model.classifier = nn.Sequential(nn.Dropout(DROPOUT_P), nn.Linear(model.classifier[-1].in_features, 1))
    elif model_name == "densenet121":
        model = models.densenet121(weights=None)
        model.classifier = nn.Sequential(nn.Dropout(DROPOUT_P), nn.Linear(model.classifier.in_features, 1))
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return model.to(DEVICE)


def get_gradcam_target_layer(model, model_name):
    if model_name == "resnet50":
        return model.layer4[-1].conv3
    if model_name == "mobilenet_v2":
        return model.features[-1][0]
    if model_name == "efficientnet_b0":
        return model.features[-1][0]
    if model_name == "densenet121":
        return model.features.denseblock4.denselayer16.conv2
    raise ValueError(f"Unsupported Grad-CAM model: {model_name}")


def compute_gradcam(model, input_tensor, model_name):
    model.eval()
    x = input_tensor.detach().clone().to(DEVICE, dtype=torch.float32)
    x.requires_grad_(True)
    target_layer = get_gradcam_target_layer(model, model_name)
    activations, gradients = [], []

    def forward_hook(_module, _inputs, output):
        if not torch.is_tensor(output) or output.ndim != 4:
            raise RuntimeError(f"Grad-CAM target output for {model_name} must be 4-D.")
        activations.append(output)
        output.register_hook(lambda grad: gradients.append(grad))

    handle = target_layer.register_forward_hook(forward_hook)
    try:
        model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            logits = model(x).reshape(-1)
            if logits.numel() != 1:
                raise RuntimeError(f"Expected one logit, got shape {tuple(logits.shape)}")
            logits[0].backward()
        activation = activations[-1]
        gradient = gradients[-1]
        weights = gradient.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((weights * activation).sum(dim=1))
        cam = F.interpolate(cam.unsqueeze(1), size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False).squeeze(0).squeeze(0)
        cam = cam.detach().cpu().numpy().astype(np.float32)
        cam -= float(cam.min())
        max_value = float(cam.max())
        cam = cam / max_value if max_value > 1e-12 else np.zeros_like(cam)
        return np.clip(cam, 0.0, 1.0)
    finally:
        handle.remove()
        model.zero_grad(set_to_none=True)


def make_sparse_cam(cam, top_fraction=GRADCAM_TOP_FRACTION, max_components=GRADCAM_COMPONENTS, min_component_area=GRADCAM_MIN_COMPONENT_AREA):
    from scipy import ndimage
    cam = np.asarray(cam, dtype=np.float32)
    positive = cam[cam > 0]
    if positive.size == 0:
        return np.zeros_like(cam), 1.0
    threshold = float(np.quantile(positive, 1.0 - top_fraction))
    binary = (cam >= threshold) & (cam > 0)
    labeled, n_components = ndimage.label(binary)
    if n_components == 0:
        return np.zeros_like(cam), threshold
    min_area = max(1, int(min_component_area * cam.size))
    scores = []
    for cid in range(1, n_components + 1):
        mask = labeled == cid
        area = int(mask.sum())
        if area < min_area:
            continue
        scores.append((float(cam[mask].mean() * area), cid))
    if not scores:
        scores = sorted(
            [(float(cam[labeled == cid].mean() * (labeled == cid).sum()), cid) for cid in range(1, n_components + 1)],
            reverse=True,
        )[:1]
    else:
        scores.sort(reverse=True)
    keep_ids = [cid for _, cid in scores[:max_components]]
    sparse_binary = np.isin(labeled, keep_ids)
    return np.where(sparse_binary, cam, 0.0).astype(np.float32), threshold


def render_sparse_gradcam_overlay(display_image, cam):
    import matplotlib.cm as cm
    from scipy.ndimage import gaussian_filter
    sparse_cam, threshold = make_sparse_cam(cam)
    if sparse_cam.max() > 0:
        smooth = gaussian_filter(sparse_cam, sigma=1.5)
        if smooth.max() > 0:
            alpha = smooth / smooth.max()
            support = smooth > 0.25 * smooth.max()
            alpha *= support.astype(np.float32)
        else:
            alpha = np.zeros_like(smooth)
        alpha = np.clip(alpha * GRADCAM_ALPHA_MAX, 0, GRADCAM_ALPHA_MAX)
    else:
        alpha = np.zeros_like(sparse_cam, dtype=np.float32)
    heatmap_rgb = cm.turbo(np.clip(sparse_cam, 0, 1))[..., :3]
    overlay = display_image * (1 - alpha[..., None]) + heatmap_rgb * alpha[..., None]
    return np.clip(overlay, 0, 1), alpha, threshold


def load_gradcam_image(image_path):
    image = Image.open(image_path).convert("RGB")
    display_image = np.asarray(image.resize((IMG_SIZE, IMG_SIZE))).astype(np.float32) / 255.0
    tensor = val_transform(image).unsqueeze(0).to(DEVICE, dtype=torch.float32)
    return display_image, tensor


def select_gradcam_cases_for_model(model_name, predictions_df):
    g = predictions_df[predictions_df.model == model_name].copy()
    g["predicted_label"] = (g.probability >= PRIMARY_THRESHOLD).astype(int)
    g["correct"] = g.predicted_label == g.label
    correct = g[g.correct].copy()
    positives = correct[correct.label == 1].sort_values(["probability", "image_path"])
    negatives = correct[correct.label == 0].sort_values(["probability", "image_path"])
    selected = []
    if len(positives) >= 1: selected.append(positives.iloc[0])
    if len(positives) >= 2: selected.append(positives.iloc[-1])
    if len(negatives) >= 1: selected.append(negatives.iloc[-1])
    if len(negatives) >= 2: selected.append(negatives.iloc[0])
    if not selected:
        raise RuntimeError(f"No correctly classified held-out test slices available for Grad-CAM: {model_name}")
    return pd.DataFrame(selected).drop_duplicates(subset=["image_path"]).reset_index(drop=True)


# ================================================================
# MAIN: one separate image per case, per-architecture folders
# ================================================================
manifest_rows = []

for model_name in MODEL_NAMES:
    print("\n" + "-" * 100)
    print(f"GRAD-CAM (per-case): {model_name}")
    print("-" * 100)

    checkpoint_path = OUT_DIR / f"{model_name}_seed42_slice_level_final.pth"
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint missing: {checkpoint_path}")

    model = create_model_skeleton(model_name)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    model_dir = GRADCAM_ROOT / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    selected_cases = select_gradcam_cases_for_model(model_name, test_predictions_all)

    for case_number, (_, row) in enumerate(selected_cases.iterrows(), start=1):
        display_image, tensor = load_gradcam_image(row.image_path)
        cam = compute_gradcam(model, tensor, model_name)
        overlay, alpha, threshold = render_sparse_gradcam_overlay(display_image, cam)
        sparse_cam, _ = make_sparse_cam(cam)
        true_name = "Hemorrhagic" if int(row.label) == 1 else "Normal"
        probability = float(row.probability)

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(display_image); axes[0].set_title(f"{true_name} | P(ICH)={probability:.3f}"); axes[0].axis("off")
        axes[1].imshow(sparse_cam, cmap="turbo", vmin=0, vmax=1); axes[1].set_title("Sparse Grad-CAM"); axes[1].axis("off")
        axes[2].imshow(overlay); axes[2].set_title("Grad-CAM overlay")
        if alpha.max() > 0:
            try:
                axes[2].contour(alpha > 0.5, levels=[0.5], linewidths=0.8)
            except Exception:
                pass
        axes[2].axis("off")
        fig.suptitle(f"{display_names[model_name]} — Case {case_number} ({true_name})", fontsize=14)
        fig.tight_layout(rect=[0, 0, 1, 0.92])

        stem = f"case_{case_number:02d}_{true_name.lower()}_p{probability:.3f}"
        fig.savefig(model_dir / f"{stem}.png", dpi=FIG_DPI, bbox_inches="tight", facecolor="white")
        fig.savefig(model_dir / f"{stem}.pdf", bbox_inches="tight", facecolor="white")
        plt.close(fig)

        manifest_rows.append(
            {
                "model": model_name,
                "case_number": case_number,
                "patient_id": row.patient_id,
                "label": int(row.label),
                "label_name": true_name,
                "probability": probability,
                "source_image_path": row.image_path,
                "output_png": str(model_dir / f"{stem}.png"),
            }
        )
        print(f"  saved case {case_number}: {true_name}, P(ICH)={probability:.3f} -> {stem}.png")

    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = GRADCAM_ROOT / "gradcam_cases_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)
print(f"\nManifest saved: {manifest_path}")

# ================================================================
# NEW ZIP — just the per-case Grad-CAM folders (separate from the big one)
# ================================================================
if NEW_ZIP_PATH.exists():
    NEW_ZIP_PATH.unlink()

with zipfile.ZipFile(NEW_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for file_path in GRADCAM_ROOT.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=str(file_path.relative_to(OUT_DIR)))

with zipfile.ZipFile(NEW_ZIP_PATH, "r") as zf:
    bad_file = zf.testzip()
    if bad_file is not None:
        raise RuntimeError(f"New Grad-CAM ZIP integrity check failed at: {bad_file}")
    members = zf.namelist()

print("\n" + "=" * 100)
print("NEW GRAD-CAM ZIP CREATED")
print("=" * 100)
print(f"ZIP path: {NEW_ZIP_PATH}")
print(f"ZIP size: {NEW_ZIP_PATH.stat().st_size / (1024 ** 2):.2f} MB")
print(f"Files inside: {len(members)}")
for name in sorted(members):
    print(" ", name)

Using device: cuda
Loaded 4080 held-out test slice predictions from master CSV.

----------------------------------------------------------------------------------------------------
GRAD-CAM (per-case): mobilenet_v2
----------------------------------------------------------------------------------------------------
  saved case 1: Hemorrhagic, P(ICH)=0.553 -> case_01_hemorrhagic_p0.553.png
  saved case 2: Hemorrhagic, P(ICH)=1.000 -> case_02_hemorrhagic_p1.000.png
  saved case 3: Normal, P(ICH)=0.412 -> case_03_normal_p0.412.png
  saved case 4: Normal, P(ICH)=0.000 -> case_04_normal_p0.000.png

----------------------------------------------------------------------------------------------------
GRAD-CAM (per-case): efficientnet_b0
----------------------------------------------------------------------------------------------------
  saved case 1: Hemorrhagic, P(ICH)=0.940 -> case_01_hemorrhagic_p0.940.png
  saved case 2: Hemorrhagic, P(ICH)=1.000 -> case_02_hemorrhagic_p1.000.png
  saved